# 5. HPA and pan-atlas analysis (Fig. 4, Fig. S5)

This notebook merges the former `6_pan_atlas.ipynb`, `5_HPA_heatmaps.ipynb` and `5_1_hpa_barplots.ipynb`, in the order their data dependencies require.

| Part | Content | Needs | Writes |
|---|---|---|---|
| A | Pan-atlas resources and analysis | `comparison_outputs/best_model_per_protein_all_metrics.csv` (notebook 1) | `pan_atlas_downloads/`, `6_pan_atlas_outputs/` |
| B | HPA class heatmaps | HPA file from Part A; cascade tables from notebook 4 | `4.2_HPA_heatmaps_outputs/` |
| C | HPA enrichment (Fig. 4) and Pearson–MSE overlap (Fig. S5) | same as B, plus model score tables | `4.0_hpa_v4_outputs/` |

Run the cells top to bottom. Helper functions that were identical in the original notebooks are defined once, and duplicate imports are removed.
Functions that differ between parts keep their part-specific definitions, which are redefined at the start of each part.
Parts B and C use the paper's primary proxy mode (`procedural_unified_tables/train_linear/...`) for the cascade partitions.

---
# Part A — Pan-atlas analysis (was `6_pan_atlas.ipynb`)

Downloads HPA, UniProt, GO and Reactome into `pan_atlas_downloads/` (needed by Parts B and C) and writes `6_pan_atlas_outputs/`.

## 6. pan_atlas — curated multi-atlas analysis

This notebook merges the useful parts of **4. pan_atlas** and **4.1 atlas_group** into one non-redundant workflow. It uses the organizational and grid conventions of the curated HPA notebook, but it does **not** reproduce the HPA cascade analysis.

The analysis is restricted to **Pearson** and **MSE**. It:

1. downloads or reuses HPA, HPA subcellular, UniProtKB, GOA, Reactome and WikiPathways resources;
2. normalizes all memberships into one atlas table;
3. constructs metric-specific evaluated universes and threshold sets;
4. reports each atlas group's size in the loaded resource, the overall tested universe and the current metric universe;
5. computes capture, expected overlap, fold enrichment, Fisher exact tests and FDR for every retained group, including zero-overlap groups;
6. creates compact grids that separate universe coverage, enrichment and capture without repeating the same top-group view in multiple formats.

All percentages state their denominator explicitly. The loaded atlas denominators are derived from the downloaded files and are not treated as immutable reference counts.


## 1. Analysis conventions

**Metric universes.** Pearson and MSE may be available for different genes. Each threshold is therefore tested against the genes with a valid value for that metric. Direct Pearson–MSE comparisons use only the joint evaluated universe.

**Group denominators.** Each atlas group has three useful sizes:

- `n_genes_atlas_loaded`: unique genes in the loaded atlas resource;
- `n_genes_tested_any_metric`: genes with at least one valid Pearson or MSE value;
- `n_genes_in_metric_universe`: genes with a valid value for the threshold's metric.

**Statistical denominator.** Fisher enrichment and expected overlap use `n_genes_in_metric_universe`, because this is the population from which a metric-specific threshold set can be selected.

**Capture denominators.** The notebook reports capture against the metric universe, the tested-any universe and the full loaded atlas group. These answer different questions and should not be substituted for one another.


In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "5_hpa_pan_atlas"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
from pathlib import Path
from collections import OrderedDict
import datetime as dt
import gzip
import math
import re
import shutil
import textwrap
import urllib.parse
import urllib.request
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

try:
    from scipy.stats import fisher_exact, hypergeom
except Exception as exc:
    fisher_exact = None
    hypergeom = None
    warnings.warn(f'SciPy is unavailable; inferential fields will be NaN. {exc}')

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 220)
pd.set_option('display.width', 260)
pd.set_option('display.max_colwidth', 140)

BASE_DIR = Path.cwd()
DOWNLOAD_DIR = BASE_DIR / 'pan_atlas_downloads'
OUTPUT_DIR = BASE_DIR / '6_pan_atlas_outputs'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
RESOURCE_ZIP = BASE_DIR / 'pan_atlas_resources.zip'
for folder in (DOWNLOAD_DIR, OUTPUT_DIR, TABLE_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Resource behaviour
RUN_DOWNLOADS = True
OVERWRITE_DOWNLOADS = False

# Performance input. Add explicit paths here when automatic discovery is not desired.
PERFORMANCE_TABLE_PATHS = []
AUTO_DISCOVER_PERFORMANCE_TABLE = True
MODEL_FAMILY_FILTER = None
COLUMN_OVERRIDES = {'gene': None, 'protein': None, 'method': None, 'pearson': None, 'mse': None}

# Pearson and MSE only. Add or remove threshold values here.
SELECTED_THRESHOLDS_BY_METRIC = OrderedDict([
    ('Pearson', [0.50, 0.70]),
    ('MSE', [0.20, 0.30]),
])
METRIC_ORDER = {'Pearson': 0, 'MSE': 1}

# Keep a permissive full export, but use more stable groups in main figures.
MIN_GROUP_SIZE_EXPORT = 5
MIN_GROUP_SIZE_PLOTS = 10
MIN_OVERLAP_FOR_FLAG = 3
FDR_CUTOFF = 0.05
FOLD_ENRICHMENT_CUTOFF = 1.50
TOP_N_RANKED = 14
TOP_N_CAPTURE_PER_SOURCE = 14

# Figure behaviour
COMPACT_GRID_MAX_COLS = 2
SAVE_PNG = True
SAVE_SVG = False
SAVE_PDF = False
DISPLAY_MAIN_FIGURES = True
MAKE_QC_FIGURES = True
DPI = 180

mpl.rcParams.update({
    'figure.dpi': 125,
    'savefig.dpi': DPI,
    'font.size': 9.0,
    'axes.titlesize': 10.5,
    'axes.titleweight': 'bold',
    'axes.labelsize': 9.2,
    'xtick.labelsize': 7.8,
    'ytick.labelsize': 7.8,
    'legend.fontsize': 8.0,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

print('Base directory:', BASE_DIR.resolve())
print('Output directory:', OUTPUT_DIR.resolve())
print('Thresholds:', dict(SELECTED_THRESHOLDS_BY_METRIC))

In [ ]:
def safe_filename(text, max_len=180):
    text = re.sub(r'[^A-Za-z0-9._ -]+', '_', str(text))
    text = re.sub(r'\s+', '_', text).strip('._-')
    return (text[:max_len] or 'figure')

def wrap_label(text, width=42):
    return '\n'.join(textwrap.wrap(str(text), width=width, break_long_words=False))

def normalize_text(value):
    value = str(value).strip().replace('–', '-').replace('—', '-').replace('−', '-')
    return re.sub(r'\s+', ' ', value).casefold()

def clean_gene_symbol(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if not value or value.casefold() in {'nan', 'none', 'na', 'null'}:
        return np.nan
    value = re.split(r'[;|,]+', value)[0].strip()
    value = re.sub(r'\.\d+$', '', value)
    value = value.split()[0]
    return value.upper() if value else np.nan

def normalize_group_name(value):
    if pd.isna(value):
        return np.nan
    value = re.sub(r'\s+', ' ', str(value).strip())
    return value if value and value.casefold() not in {'nan', 'none', 'na', 'null', 'unknown'} else np.nan

def split_multi_value(value, seps=(';', ',')):
    if pd.isna(value):
        return []
    value = str(value).strip()
    if not value:
        return []
    pattern = '|'.join(re.escape(x) for x in seps)
    out, seen = [], set()
    for part in re.split(pattern, value):
        term = normalize_group_name(part)
        key = normalize_text(term) if pd.notna(term) else None
        if key and key not in seen:
            out.append(term)
            seen.add(key)
    return out

def pick_column(df, candidates, required=True):
    direct = {str(c).casefold(): c for c in df.columns}
    normalized = {re.sub(r'[^a-z0-9]+', '', str(c).casefold()): c for c in df.columns}
    for candidate in candidates:
        if str(candidate).casefold() in direct:
            return direct[str(candidate).casefold()]
    for candidate in candidates:
        key = re.sub(r'[^a-z0-9]+', '', str(candidate).casefold())
        if key in normalized:
            return normalized[key]
    for candidate in candidates:
        key = re.sub(r'[^a-z0-9]+', '', str(candidate).casefold())
        for col in df.columns:
            if key and key in re.sub(r'[^a-z0-9]+', '', str(col).casefold()):
                return col
    if required:
        raise KeyError(f'Could not find any of {candidates}. Columns include: {list(df.columns)[:80]}')
    return None

def bh_fdr(pvalues):
    p = np.asarray(pvalues, dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return q
    pv = p[valid]
    order = np.argsort(pv)
    ranks = np.arange(1, len(pv) + 1)
    q_sorted = pv[order] * len(pv) / ranks
    q_sorted = np.minimum.accumulate(q_sorted[::-1])[::-1]
    q_valid = np.empty_like(pv)
    q_valid[order] = np.minimum(q_sorted, 1.0)
    q[valid] = q_valid
    return q

def safe_percent(numerator, denominator):
    n = np.asarray(numerator, dtype=float)
    d = np.asarray(denominator, dtype=float)
    out = np.zeros(np.broadcast(n, d).shape, dtype=float)
    np.divide(100.0 * n, d, out=out, where=d > 0)
    return out

def compact_grid(n_panels, panel_size=(6.0, 4.4), max_cols=None, **subplot_kwargs):
    n_panels = int(n_panels)
    if n_panels < 1:
        raise ValueError('n_panels must be at least 1')
    max_cols = COMPACT_GRID_MAX_COLS if max_cols is None else max_cols
    ncols = min(max_cols, n_panels)
    nrows = int(math.ceil(n_panels / ncols))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(panel_size[0] * ncols, panel_size[1] * nrows),
        squeeze=False,
        **subplot_kwargs,
    )
    flat = axes.ravel()
    for ax in flat[n_panels:]:
        ax.set_visible(False)
    return fig, flat[:n_panels], nrows, ncols

def apply_grid_axis_label_policy(fig):
    """In multi-column grids, retain y titles on left and x titles on right."""
    for ax in fig.axes:
        if not ax.get_visible():
            continue
        try:
            spec = ax.get_subplotspec()
            grid = spec.get_gridspec()
        except Exception:
            continue
        if grid.ncols <= 1:
            continue
        col = spec.colspan.start
        if col != 0:
            ax.set_ylabel('')
        if col != grid.ncols - 1:
            ax.set_xlabel('')

def save_figure(fig, stem):
    stem = safe_filename(stem)
    saved = []
    if SAVE_PNG:
        path = FIGURE_DIR / f'{stem}.png'
        fig.savefig(path, bbox_inches='tight', dpi=DPI, facecolor='white')
        saved.append(path)
    if SAVE_SVG:
        path = FIGURE_DIR / f'{stem}.svg'
        fig.savefig(path, bbox_inches='tight', facecolor='white')
        saved.append(path)
    if SAVE_PDF:
        path = FIGURE_DIR / f'{stem}.pdf'
        fig.savefig(path, bbox_inches='tight', facecolor='white')
        saved.append(path)
    return saved

def finish_figure(fig, stem, show=DISPLAY_MAIN_FIGURES):
    apply_grid_axis_label_policy(fig)
    save_figure(fig, stem)
    if show:
        plt.show()
    else:
        plt.close(fig)

def read_table_any(path, nrows=None):
    path = Path(path)
    suffixes = [x.casefold() for x in path.suffixes]
    if path.suffix.casefold() == '.parquet':
        df = pd.read_parquet(path)
        return df.head(nrows) if nrows is not None else df
    if suffixes and suffixes[-1] == '.zip':
        with zipfile.ZipFile(path) as zf:
            names = [n for n in zf.namelist() if not n.endswith('/') and n.casefold().endswith(('.tsv', '.txt', '.csv'))]
            if not names:
                raise ValueError(f'No delimited table found inside {path}')
            inner = names[0]
            sep = ',' if inner.casefold().endswith('.csv') else '\t'
            with zf.open(inner) as handle:
                return pd.read_csv(handle, sep=sep, nrows=nrows, low_memory=False)
    if suffixes and suffixes[-1] == '.gz':
        sep = ',' if len(suffixes) >= 2 and suffixes[-2] == '.csv' else '\t'
        return pd.read_csv(path, sep=sep, compression='gzip', nrows=nrows, low_memory=False)
    if path.suffix.casefold() == '.csv':
        return pd.read_csv(path, nrows=nrows, low_memory=False)
    return pd.read_csv(path, sep='\t', nrows=nrows, low_memory=False)

def threshold_operator(metric):
    return '<=' if metric == 'MSE' else '>='

def passes_threshold(values, metric, threshold):
    values = pd.to_numeric(values, errors='coerce')
    return values.le(threshold) if metric == 'MSE' else values.ge(threshold)

def condition_label(metric, threshold):
    return f'{metric} {threshold_operator(metric)} {float(threshold):.2f}'

## 2. Atlas resources

The normalized membership table retains the atlas source and collection. HPA protein classes and HPA subcellular locations remain separate; UniProt keywords, GO-derived labels and subcellular annotations are also separated rather than collapsed into one uninterpretable collection.


In [ ]:
RESOURCE_CATALOGUE = [
    {
        'resource': 'Human Protein Atlas', 'source': 'HPA', 'collection': 'protein_class',
        'filename': 'hpa_proteinatlas.tsv.zip',
        'url': 'https://www.proteinatlas.org/download/proteinatlas.tsv.zip',
        'format': 'zipped TSV',
    },
    {
        'resource': 'Human Protein Atlas subcellular location', 'source': 'HPA', 'collection': 'subcellular_location',
        'filename': 'hpa_subcellular_location.tsv.zip',
        'url': 'https://www.proteinatlas.org/download/subcellular_location.tsv.zip',
        'format': 'zipped TSV',
    },
    {
        'resource': 'UniProtKB Human', 'source': 'UniProtKB', 'collection': 'keywords_go_subcellular',
        'filename': 'uniprot_human.tsv.gz',
        'url': 'https://rest.uniprot.org/uniprotkb/stream?' + urllib.parse.urlencode({
            'compressed': 'true', 'format': 'tsv', 'query': '(organism_id:9606)',
            'fields': 'accession,gene_primary,gene_names,protein_name,keyword,go_id,go_p,go_f,go_c,cc_subcellular_location',
        }),
        'format': 'gzipped TSV',
    },
    {
        'resource': 'GOA Human annotations', 'source': 'GOA', 'collection': 'go_annotations',
        'filename': 'goa_human.gaf.gz',
        'url': 'https://current.geneontology.org/annotations/goa_human.gaf.gz',
        'format': 'gzipped GAF',
    },
    {
        'resource': 'Gene Ontology names', 'source': 'GO', 'collection': 'ontology_names',
        'filename': 'go-basic.obo',
        'url': 'https://current.geneontology.org/ontology/go-basic.obo',
        'format': 'OBO',
    },
    {
        'resource': 'Reactome pathways', 'source': 'Reactome', 'collection': 'pathways',
        'filename': 'reactome_pathways.gmt.zip',
        'url': 'https://reactome.org/download/current/ReactomePathways.gmt.zip',
        'format': 'zipped GMT',
    },
    {
        'resource': 'WikiPathways Homo sapiens pathways', 'source': 'WikiPathways', 'collection': 'pathways',
        'filename': 'wikipathways_human.gmt',
        'url': 'auto-discovered',
        'format': 'GMT',
    },
]
resource_catalogue = pd.DataFrame(RESOURCE_CATALOGUE)
resource_catalogue.to_csv(TABLE_DIR / 'resource_catalogue.csv', index=False)
display(resource_catalogue)

def extract_resource_zip_if_needed():
    if RESOURCE_ZIP.exists() and not any(DOWNLOAD_DIR.glob('*')):
        print('Extracting local resource bundle:', RESOURCE_ZIP)
        with zipfile.ZipFile(RESOURCE_ZIP) as zf:
            zf.extractall(DOWNLOAD_DIR)

def download_file(url, output_path, overwrite=False, timeout=240):
    output_path = Path(output_path)
    if output_path.exists() and output_path.stat().st_size > 0 and not overwrite:
        return True
    try:
        print('Downloading', output_path.name)
        with urllib.request.urlopen(url, timeout=timeout) as response, open(output_path, 'wb') as handle:
            shutil.copyfileobj(response, handle)
        return output_path.exists() and output_path.stat().st_size > 0
    except Exception as exc:
        print(f'Skipped {output_path.name}: {exc}')
        return False

def find_wikipathways_human_gmt():
    roots = [
        'https://data.wikipathways.org/current/gmt/',
        'https://wikipathways-data.wmcloud.org/current/gmt/',
    ]
    patterns = [
        r'href=["\']([^"\']*Homo_sapiens[^"\']*\.gmt)["\']',
        r'(wikipathways-[^\s"\']*gmt-Homo_sapiens\.gmt)',
    ]
    for root in roots:
        try:
            with urllib.request.urlopen(root, timeout=30) as response:
                html = response.read().decode('utf-8', errors='ignore')
            matches = []
            for pattern in patterns:
                matches.extend(re.findall(pattern, html))
            if matches:
                return urllib.parse.urljoin(root, sorted(set(matches))[-1])
        except Exception:
            continue
    return None

extract_resource_zip_if_needed()
download_rows = []
if RUN_DOWNLOADS:
    for row in RESOURCE_CATALOGUE:
        filename = row['filename']
        if filename == 'wikipathways_human.gmt':
            url = find_wikipathways_human_gmt()
            available = download_file(url, DOWNLOAD_DIR / filename, OVERWRITE_DOWNLOADS) if url else (DOWNLOAD_DIR / filename).exists()
        else:
            available = download_file(row['url'], DOWNLOAD_DIR / filename, OVERWRITE_DOWNLOADS)
        download_rows.append({'file': filename, 'available': bool(available)})
else:
    for row in RESOURCE_CATALOGUE:
        path = DOWNLOAD_DIR / row['filename']
        download_rows.append({'file': path.name, 'available': path.exists() and path.stat().st_size > 0})

download_status = pd.DataFrame(download_rows)
download_status.to_csv(TABLE_DIR / 'download_status.csv', index=False)
display(download_status)

with zipfile.ZipFile(RESOURCE_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(DOWNLOAD_DIR.glob('*')):
        if path.is_file() and path.stat().st_size > 0:
            zf.write(path, arcname=path.name)
print('Resource bundle:', RESOURCE_ZIP.resolve())

In [ ]:
def parse_go_obo(path):
    path = Path(path)
    if not path.exists():
        print('GO ontology file unavailable; GOA labels will use GO IDs.')
        return {}
    mapping, current = {}, {}
    with open(path, 'rt', encoding='utf-8', errors='ignore') as handle:
        for line in handle:
            line = line.rstrip('\n')
            if line == '[Term]':
                if current.get('id') and not current.get('is_obsolete'):
                    mapping[current['id']] = {'name': current.get('name', current['id']), 'namespace': current.get('namespace', 'GO')}
                current = {}
            elif line.startswith('id: '):
                current['id'] = line[4:]
            elif line.startswith('name: '):
                current['name'] = line[6:]
            elif line.startswith('namespace: '):
                current['namespace'] = line[11:]
            elif line == 'is_obsolete: true':
                current['is_obsolete'] = True
    if current.get('id') and not current.get('is_obsolete'):
        mapping[current['id']] = {'name': current.get('name', current['id']), 'namespace': current.get('namespace', 'GO')}
    return mapping

GO_TERM_MAP = parse_go_obo(DOWNLOAD_DIR / 'go-basic.obo')
print(f'GO term names loaded: {len(GO_TERM_MAP):,}')

atlas_frames = []
raw_resource_rows = []

def add_membership(df, source, collection, evidence_col=None, raw_value_col=None):
    if df is None or df.empty:
        return
    out = df.copy()
    out['source'] = source
    out['collection'] = collection
    for col in ['gene_symbol', 'uniprot_id', 'ensembl_id', 'group_name']:
        if col not in out.columns:
            out[col] = np.nan
    out['evidence'] = out[evidence_col] if evidence_col and evidence_col in out.columns else np.nan
    out['raw_value'] = out[raw_value_col] if raw_value_col and raw_value_col in out.columns else out['group_name']
    out['gene_symbol'] = out['gene_symbol'].map(clean_gene_symbol)
    out['group_name'] = out['group_name'].map(normalize_group_name)
    out = out[['source', 'collection', 'group_name', 'gene_symbol', 'uniprot_id', 'ensembl_id', 'evidence', 'raw_value']]
    out = out.dropna(subset=['gene_symbol', 'group_name']).drop_duplicates()
    if not out.empty:
        atlas_frames.append(out)
        print(f'Added {source} / {collection}: {len(out):,} memberships, {out.group_name.nunique():,} groups, {out.gene_symbol.nunique():,} genes')

def parse_hpa_proteinatlas(path):
    path = Path(path)
    if not path.exists():
        return
    hpa = read_table_any(path)
    raw_resource_rows.append({'source': 'HPA', 'collection': 'protein_class', 'file': path.name, 'rows': len(hpa), 'columns': len(hpa.columns)})
    gene_col = pick_column(hpa, ['Gene', 'Gene name', 'gene_symbol'])
    ens_col = pick_column(hpa, ['Ensembl', 'ensembl_id'], required=False)
    uni_col = pick_column(hpa, ['Uniprot', 'UniProt', 'uniprot_id'], required=False)
    class_col = pick_column(hpa, ['Protein class', 'Protein classes', 'protein_class'])
    rows = []
    # iterrows is used here because HPA column names contain spaces; itertuples
    # sanitizes those names and can make direct column lookup unreliable.
    for _, record in hpa.iterrows():
        for group in split_multi_value(record.get(class_col), seps=(';', ',')):
            rows.append({
                'gene_symbol': record.get(gene_col),
                'ensembl_id': record.get(ens_col) if ens_col else np.nan,
                'uniprot_id': record.get(uni_col) if uni_col else np.nan,
                'group_name': group,
                'raw_value': record.get(class_col),
            })
    add_membership(pd.DataFrame(rows), 'HPA', 'protein_class', raw_value_col='raw_value')

def parse_hpa_subcellular(path):
    path = Path(path)
    if not path.exists():
        return
    sub = read_table_any(path)
    raw_resource_rows.append({'source': 'HPA', 'collection': 'subcellular_location', 'file': path.name, 'rows': len(sub), 'columns': len(sub.columns)})
    gene_col = pick_column(sub, ['Gene', 'Gene name', 'gene_symbol'])
    ens_col = pick_column(sub, ['Ensembl', 'ensembl_id'], required=False)
    loc_cols = [c for c in sub.columns if 'location' in str(c).casefold() and 'reliability' not in str(c).casefold()]
    rows = []
    for _, record in sub.iterrows():
        for col in loc_cols:
            for group in split_multi_value(record.get(col), seps=(';', ',')):
                rows.append({
                    'gene_symbol': record.get(gene_col),
                    'ensembl_id': record.get(ens_col) if ens_col else np.nan,
                    'group_name': group,
                    'raw_value': record.get(col),
                })
    add_membership(pd.DataFrame(rows), 'HPA', 'subcellular_location', raw_value_col='raw_value')

def parse_uniprot(path):
    path = Path(path)
    if not path.exists():
        return
    uni = read_table_any(path)
    raw_resource_rows.append({'source': 'UniProtKB', 'collection': 'human_stream', 'file': path.name, 'rows': len(uni), 'columns': len(uni.columns)})
    gene_col = pick_column(uni, ['Gene Names (primary)', 'Gene Names', 'gene_primary'], required=False)
    acc_col = pick_column(uni, ['Entry', 'accession', 'Accession'], required=False)
    column_specs = []
    for col in uni.columns:
        key = str(col).casefold()
        if 'keyword' in key:
            column_specs.append((col, 'keywords', (';', ',')))
        elif 'biological process' in key or key in {'go_p', 'gene ontology (biological process)'}:
            column_specs.append((col, 'go_biological_process', (';',)))
        elif 'molecular function' in key or key in {'go_f', 'gene ontology (molecular function)'}:
            column_specs.append((col, 'go_molecular_function', (';',)))
        elif 'cellular component' in key or key in {'go_c', 'gene ontology (cellular component)'}:
            column_specs.append((col, 'go_cellular_component', (';',)))
        elif 'subcellular' in key and 'id' not in key:
            column_specs.append((col, 'subcellular_location', (';',)))
    for col, collection, separators in column_specs:
        rows = []
        for _, record in uni[[c for c in [gene_col, acc_col, col] if c]].iterrows():
            gene_raw = record.get(gene_col) if gene_col else np.nan
            gene = str(gene_raw).split()[0] if pd.notna(gene_raw) else np.nan
            for value in split_multi_value(record.get(col), seps=separators):
                value = re.sub(r'\s*\[GO:\d+\]', '', str(value)).strip()
                value = re.sub(r'^SUBCELLULAR LOCATION:\s*', '', value, flags=re.I)
                value = re.sub(r'\{[^}]+\}', '', value).strip()
                if 1 <= len(value) <= 180:
                    rows.append({'gene_symbol': gene, 'uniprot_id': record.get(acc_col) if acc_col else np.nan, 'group_name': value, 'raw_value': record.get(col)})
        add_membership(pd.DataFrame(rows), 'UniProtKB', collection, raw_value_col='raw_value')

def parse_goa(path):
    path = Path(path)
    if not path.exists():
        return
    columns = [
        'DB', 'DB_Object_ID', 'DB_Object_Symbol', 'Qualifier', 'GO_ID', 'DB_Reference', 'Evidence_Code',
        'With_From', 'Aspect', 'DB_Object_Name', 'DB_Object_Synonym', 'DB_Object_Type', 'Taxon', 'Date',
        'Assigned_By', 'Annotation_Extension', 'Gene_Product_Form_ID',
    ]
    usecols = ['DB_Object_ID', 'DB_Object_Symbol', 'Qualifier', 'GO_ID', 'Evidence_Code', 'Aspect']
    with gzip.open(path, 'rt', encoding='utf-8', errors='ignore') as handle:
        goa = pd.read_csv(handle, sep='\t', comment='!', names=columns, usecols=usecols, low_memory=False)
    raw_resource_rows.append({'source': 'GOA', 'collection': 'go_annotations', 'file': path.name, 'rows': len(goa), 'columns': len(usecols)})
    qualifier = goa['Qualifier'].fillna('').astype(str)
    goa = goa.loc[~qualifier.str.split('|', regex=False).map(lambda xs: 'NOT' in xs)].copy()
    goa = goa.dropna(subset=['DB_Object_Symbol', 'GO_ID'])
    aspect_map = {'P': 'biological_process', 'F': 'molecular_function', 'C': 'cellular_component'}
    goa['collection'] = goa['Aspect'].map(aspect_map).fillna('go_annotations')
    goa['group_name'] = goa['GO_ID'].map(lambda go_id: f"{GO_TERM_MAP.get(go_id, {}).get('name', go_id)} [{go_id}]")
    goa = goa.rename(columns={
        'DB_Object_Symbol': 'gene_symbol', 'DB_Object_ID': 'uniprot_id',
        'Evidence_Code': 'evidence', 'GO_ID': 'raw_value',
    })
    for collection, sub in goa.groupby('collection', sort=False):
        add_membership(sub, 'GOA', collection, evidence_col='evidence', raw_value_col='raw_value')

def read_gmt(content, source, collection):
    rows = []
    for line in content.splitlines():
        parts = line.rstrip('\n').split('\t')
        if len(parts) < 3:
            continue
        group = normalize_group_name(parts[0])
        description = parts[1]
        for gene in parts[2:]:
            gene = clean_gene_symbol(gene)
            if pd.notna(gene) and not str(gene).startswith('HTTP'):
                rows.append({'gene_symbol': gene, 'group_name': group, 'raw_value': description})
    add_membership(pd.DataFrame(rows), source, collection, raw_value_col='raw_value')

def parse_reactome(path):
    path = Path(path)
    if not path.exists():
        return
    with zipfile.ZipFile(path) as zf:
        names = [n for n in zf.namelist() if n.casefold().endswith('.gmt')]
        if not names:
            return
        preferred = [n for n in names if 'human' in n.casefold() or 'homo' in n.casefold()]
        name = sorted(preferred or names, key=lambda x: (len(x), x))[0]
        content = zf.read(name).decode('utf-8', errors='ignore')
    raw_resource_rows.append({'source': 'Reactome', 'collection': 'pathways', 'file': f'{path.name}:{name}', 'rows': len(content.splitlines()), 'columns': np.nan})
    read_gmt(content, 'Reactome', 'pathways')

def parse_wikipathways(path):
    path = Path(path)
    if not path.exists():
        return
    content = path.read_text(encoding='utf-8', errors='ignore')
    raw_resource_rows.append({'source': 'WikiPathways', 'collection': 'pathways', 'file': path.name, 'rows': len(content.splitlines()), 'columns': np.nan})
    read_gmt(content, 'WikiPathways', 'pathways')

parse_hpa_proteinatlas(DOWNLOAD_DIR / 'hpa_proteinatlas.tsv.zip')
parse_hpa_subcellular(DOWNLOAD_DIR / 'hpa_subcellular_location.tsv.zip')
parse_uniprot(DOWNLOAD_DIR / 'uniprot_human.tsv.gz')
parse_goa(DOWNLOAD_DIR / 'goa_human.gaf.gz')
parse_reactome(DOWNLOAD_DIR / 'reactome_pathways.gmt.zip')
parse_wikipathways(DOWNLOAD_DIR / 'wikipathways_human.gmt')

if not atlas_frames:
    raise RuntimeError('No atlas resources were parsed. Check pan_atlas_downloads/.')

atlas_long = pd.concat(atlas_frames, ignore_index=True).drop_duplicates()
atlas_long['group_id'] = atlas_long['source'] + ' | ' + atlas_long['collection'] + ' | ' + atlas_long['group_name']
atlas_long = atlas_long.sort_values(['source', 'collection', 'group_name', 'gene_symbol']).reset_index(drop=True)
atlas_long.to_csv(TABLE_DIR / 'atlas_membership_long.csv', index=False)

raw_resource_summary = pd.DataFrame(raw_resource_rows)
raw_resource_summary.to_csv(TABLE_DIR / 'raw_resource_summary.csv', index=False)

atlas_resource_summary = (
    atlas_long.groupby(['source', 'collection'], as_index=False)
    .agg(
        n_memberships=('group_id', 'size'),
        n_groups=('group_id', 'nunique'),
        n_genes_atlas_loaded=('gene_symbol', 'nunique'),
        median_group_size=('group_id', lambda x: np.nan),
    )
)
# Replace the placeholder with group-size medians calculated without double-counting genes.
_loaded_sizes = atlas_long.drop_duplicates(['group_id', 'gene_symbol']).groupby(['source', 'collection', 'group_id'])['gene_symbol'].nunique()
_medians = _loaded_sizes.groupby(['source', 'collection']).median().rename('median_group_size').reset_index()
atlas_resource_summary = atlas_resource_summary.drop(columns='median_group_size').merge(_medians, on=['source', 'collection'], how='left')
atlas_resource_summary.to_csv(TABLE_DIR / 'atlas_resource_summary.csv', index=False)
display(atlas_resource_summary)

# Resource-composition audits retained as tables rather than repetitive figures.
gene_annotation_burden = (
    atlas_long.drop_duplicates(['source', 'group_id', 'gene_symbol'])
    .groupby(['source', 'gene_symbol'], as_index=False)
    .agg(n_groups_annotating_gene=('group_id', 'nunique'))
)
gene_annotation_burden.to_csv(TABLE_DIR / 'gene_annotation_burden_by_source.csv', index=False)

goa_evidence_summary = (
    atlas_long.loc[atlas_long.source.eq('GOA') & atlas_long.evidence.notna()]
    .groupby(['collection', 'evidence'], as_index=False)
    .agg(n_memberships=('group_id', 'size'), n_genes=('gene_symbol', 'nunique'), n_groups=('group_id', 'nunique'))
    .sort_values(['collection', 'n_memberships'], ascending=[True, False])
)
goa_evidence_summary.to_csv(TABLE_DIR / 'goa_evidence_summary.csv', index=False)
if not goa_evidence_summary.empty:
    display(Markdown('### GOA evidence-code composition'))
    display(goa_evidence_summary)

## 3. Load and standardize model performance

The notebook selects the best available Pearson value and the best available MSE value per gene independently. Consequently, the selected Pearson and MSE values may originate from different model rows. The associated method columns preserve this provenance.

Identifier harmonization remains a critical assumption: the atlas resources are matched by cleaned gene symbol. The exported unmatched-gene tables should be inspected when a source has unexpectedly low coverage.


In [ ]:
PERFORMANCE_FILE_PRIORITY = [
    'atlas_performance_standardized.csv',
    'best_model_per_protein_all_metrics.csv',
    'all_model_results_summary.csv',
    'best_metrics_per_protein.csv',
]

GENE_ALIASES = ['gene_symbol', 'Gene', 'gene', 'symbol', 'target_gene', 'protein', 'protein_key', 'target']
PROTEIN_ALIASES = ['protein_key', 'protein_raw', 'protein', 'Protein', 'uniprot', 'UniProt', 'target']
METHOD_ALIASES = ['method', 'model', 'family', 'model_family', 'source', 'approach', 'predictor']
METRIC_ALIASES = {
    'pearson': ['pearson', 'test_pearson', 'pearson_corr', 'test_corr', 'correlation', 'corr', 'r', 'test_r'],
    'mse': ['mse', 'test_mse', 'mean_squared_error', 'test_mean_squared_error'],
}

def discover_performance_paths():
    explicit = [Path(p) for p in PERFORMANCE_TABLE_PATHS]
    found = [p for p in explicit if p.exists()]
    if found or not AUTO_DISCOVER_PERFORMANCE_TABLE:
        return found
    for filename in PERFORMANCE_FILE_PRIORITY:
        candidates = []
        direct = BASE_DIR / filename
        if direct.exists():
            candidates.append(direct)
        candidates.extend(
            p for p in BASE_DIR.rglob(filename)
            if '6_pan_atlas_outputs' not in p.parts and 'pan_atlas_downloads' not in p.parts
        )
        if candidates:
            candidates = sorted(set(candidates), key=lambda p: (len(p.parts), len(str(p))))
            print('Auto-selected performance table:', candidates[0])
            return [candidates[0]]
    return []

def standardize_performance_table(raw, source_file):
    gene_col = COLUMN_OVERRIDES.get('gene') or pick_column(raw, GENE_ALIASES)
    protein_col = COLUMN_OVERRIDES.get('protein') or pick_column(raw, PROTEIN_ALIASES, required=False)
    method_col = COLUMN_OVERRIDES.get('method') or pick_column(raw, METHOD_ALIASES, required=False)
    pearson_col = COLUMN_OVERRIDES.get('pearson') or pick_column(raw, METRIC_ALIASES['pearson'], required=False)
    mse_col = COLUMN_OVERRIDES.get('mse') or pick_column(raw, METRIC_ALIASES['mse'], required=False)
    if pearson_col is None and mse_col is None:
        raise ValueError(f'No Pearson or MSE column found in {source_file}.')
    out = pd.DataFrame({
        'gene_symbol': raw[gene_col].map(clean_gene_symbol),
        'protein_key': raw[protein_col].astype('string').str.strip() if protein_col else raw[gene_col].astype('string').str.strip(),
        'method': raw[method_col].astype('string') if method_col else 'unknown',
        'pearson': pd.to_numeric(raw[pearson_col], errors='coerce') if pearson_col else np.nan,
        'mse': pd.to_numeric(raw[mse_col], errors='coerce') if mse_col else np.nan,
        'source_file': str(source_file),
        'source_row': np.arange(len(raw)),
    })
    out = out.dropna(subset=['gene_symbol']).drop_duplicates()
    return out, {
        'source_file': str(source_file), 'gene_col': gene_col, 'protein_col': protein_col,
        'method_col': method_col, 'pearson_col': pearson_col, 'mse_col': mse_col,
    }

performance_paths = discover_performance_paths()
if not performance_paths:
    raise FileNotFoundError(
        'No performance table was found. Set PERFORMANCE_TABLE_PATHS in the setup cell. '
        f'Expected names include: {PERFORMANCE_FILE_PRIORITY}'
    )

perf_frames, detected_rows = [], []
for path in performance_paths:
    raw = read_table_any(path)
    standardized, detected = standardize_performance_table(raw, path)
    perf_frames.append(standardized)
    detected_rows.append(detected)

performance_long = pd.concat(perf_frames, ignore_index=True).drop_duplicates()
if MODEL_FAMILY_FILTER is not None:
    mask = performance_long['method'].astype(str).str.contains(str(MODEL_FAMILY_FILTER), case=False, na=False)
    if mask.any():
        performance_long = performance_long.loc[mask].copy()
    else:
        warnings.warn(f'MODEL_FAMILY_FILTER={MODEL_FAMILY_FILTER!r} matched no rows; all rows were retained.')

performance_long.to_csv(TABLE_DIR / 'performance_standardized_long.csv', index=False)
pd.DataFrame(detected_rows).to_csv(TABLE_DIR / 'performance_detected_columns.csv', index=False)

def best_metric_rows(df, metric, direction):
    valid = df.dropna(subset=['gene_symbol', metric]).copy()
    if valid.empty:
        return pd.DataFrame(columns=['gene_symbol', metric, f'{metric}_method', f'{metric}_protein_key', f'{metric}_source_file'])
    idx = valid.groupby('gene_symbol')[metric].idxmax() if direction == 'max' else valid.groupby('gene_symbol')[metric].idxmin()
    out = valid.loc[idx, ['gene_symbol', metric, 'method', 'protein_key', 'source_file']].copy()
    return out.rename(columns={
        'method': f'{metric}_method', 'protein_key': f'{metric}_protein_key', 'source_file': f'{metric}_source_file',
    })

best_pearson = best_metric_rows(performance_long, 'pearson', 'max')
best_mse = best_metric_rows(performance_long, 'mse', 'min')
gene_metrics = best_pearson.merge(best_mse, on='gene_symbol', how='outer')

burden = (
    performance_long.groupby('gene_symbol', as_index=False)
    .agg(n_source_rows=('source_row', 'size'), n_protein_keys=('protein_key', 'nunique'), n_methods=('method', 'nunique'))
)
gene_metrics = gene_metrics.merge(burden, on='gene_symbol', how='left').sort_values('gene_symbol').reset_index(drop=True)
gene_metrics.to_csv(TABLE_DIR / 'best_pearson_mse_per_gene.csv', index=False)

performance_summary = pd.DataFrame([
    {'metric': 'Pearson', 'n_evaluated_genes': gene_metrics['pearson'].notna().sum()},
    {'metric': 'MSE', 'n_evaluated_genes': gene_metrics['mse'].notna().sum()},
    {'metric': 'Either metric', 'n_evaluated_genes': gene_metrics[['pearson', 'mse']].notna().any(axis=1).sum()},
    {'metric': 'Both metrics', 'n_evaluated_genes': gene_metrics[['pearson', 'mse']].notna().all(axis=1).sum()},
])
display(performance_summary)

## 4. Threshold sets and metric-specific universes

Threshold-set size is not the same as evaluated-universe size. The first is the numerator selected by the cutoff; the second is the population eligible to pass that cutoff.


In [ ]:
condition_rows = []
for metric, thresholds in SELECTED_THRESHOLDS_BY_METRIC.items():
    for threshold in thresholds:
        condition_rows.append({
            'condition_id': safe_filename(f'{metric}_{threshold:.3f}'),
            'metric': metric,
            'metric_col': metric.casefold(),
            'threshold': float(threshold),
            'operator': threshold_operator(metric),
            'condition': condition_label(metric, threshold),
            'metric_order': METRIC_ORDER[metric],
        })
condition_table = pd.DataFrame(condition_rows).sort_values(['metric_order', 'threshold']).reset_index(drop=True)
condition_order = condition_table['condition'].tolist()
condition_table.to_csv(TABLE_DIR / 'threshold_definitions.csv', index=False)

metric_universe_sets = {
    'Pearson': set(gene_metrics.loc[gene_metrics['pearson'].notna(), 'gene_symbol']),
    'MSE': set(gene_metrics.loc[gene_metrics['mse'].notna(), 'gene_symbol']),
}
tested_any_genes = set(gene_metrics.loc[gene_metrics[['pearson', 'mse']].notna().any(axis=1), 'gene_symbol'])
joint_metric_genes = metric_universe_sets['Pearson'] & metric_universe_sets['MSE']

membership_frames = []
for case in condition_table.itertuples(index=False):
    mask = gene_metrics[case.metric_col].notna() & passes_threshold(gene_metrics[case.metric_col], case.metric, case.threshold)
    temp = gene_metrics.loc[mask, ['gene_symbol', case.metric_col]].copy()
    temp = temp.rename(columns={case.metric_col: 'metric_value'})
    temp['condition_id'] = case.condition_id
    temp['condition'] = case.condition
    temp['metric'] = case.metric
    temp['threshold'] = case.threshold
    temp['operator'] = case.operator
    membership_frames.append(temp)
threshold_membership = pd.concat(membership_frames, ignore_index=True) if membership_frames else pd.DataFrame()
threshold_membership.to_csv(TABLE_DIR / 'threshold_membership_long.csv', index=False)

threshold_summary_rows = []
for case in condition_table.itertuples(index=False):
    selected = set(threshold_membership.loc[threshold_membership['condition_id'].eq(case.condition_id), 'gene_symbol'])
    universe = metric_universe_sets[case.metric]
    threshold_summary_rows.append({
        'condition_id': case.condition_id,
        'condition': case.condition,
        'metric': case.metric,
        'threshold': case.threshold,
        'operator': case.operator,
        'n_genes_in_metric_universe': len(universe),
        'n_genes_passing_threshold': len(selected),
        'pct_metric_universe_passing': 100 * len(selected) / len(universe) if universe else np.nan,
        'n_genes_in_joint_metric_universe': len(joint_metric_genes),
    })
threshold_summary = pd.DataFrame(threshold_summary_rows)
threshold_summary.to_csv(TABLE_DIR / 'threshold_summary.csv', index=False)
display(threshold_summary)

# Nesting audit: stricter thresholds must be subsets of looser thresholds for the same metric.
nesting_rows = []
for metric, sub in condition_table.groupby('metric', sort=False):
    ordered = sub.sort_values('threshold', ascending=(metric == 'MSE'))
    cases = list(ordered.itertuples(index=False))
    for i in range(len(cases) - 1):
        strict, loose = cases[i], cases[i + 1]
        strict_set = set(threshold_membership.loc[threshold_membership.condition_id.eq(strict.condition_id), 'gene_symbol'])
        loose_set = set(threshold_membership.loc[threshold_membership.condition_id.eq(loose.condition_id), 'gene_symbol'])
        nesting_rows.append({
            'metric': metric, 'strict_condition': strict.condition, 'loose_condition': loose.condition,
            'n_strict': len(strict_set), 'n_loose': len(loose_set),
            'n_strict_not_in_loose': len(strict_set - loose_set),
            'nested_correctly': strict_set.issubset(loose_set),
        })
nesting_audit = pd.DataFrame(nesting_rows)
nesting_audit.to_csv(TABLE_DIR / 'threshold_nesting_audit.csv', index=False)
display(nesting_audit)
if not nesting_audit.empty and not nesting_audit['nested_correctly'].all():
    raise ValueError('Threshold nesting failed. Check metric directions, duplicate handling or score preprocessing.')

In [ ]:
# Pearson–MSE overlap must be measured in the common evaluated universe.
overlap_rows = []
pearson_cases = condition_table.loc[condition_table.metric.eq('Pearson')]
mse_cases = condition_table.loc[condition_table.metric.eq('MSE')]
joint_scores = gene_metrics.loc[gene_metrics.gene_symbol.isin(joint_metric_genes), ['gene_symbol', 'pearson', 'mse']].copy()
N_joint = len(joint_scores)

for p_case in pearson_cases.itertuples(index=False):
    p_set = set(joint_scores.loc[passes_threshold(joint_scores.pearson, 'Pearson', p_case.threshold), 'gene_symbol'])
    for m_case in mse_cases.itertuples(index=False):
        m_set = set(joint_scores.loc[passes_threshold(joint_scores.mse, 'MSE', m_case.threshold), 'gene_symbol'])
        shared = p_set & m_set
        expected = len(p_set) * len(m_set) / N_joint if N_joint else np.nan
        p_zero = hypergeom.pmf(0, N_joint, len(p_set), len(m_set)) if hypergeom is not None and N_joint else np.nan
        overlap_rows.append({
            'pearson_condition': p_case.condition,
            'mse_condition': m_case.condition,
            'joint_universe_size': N_joint,
            'n_pearson_pass': len(p_set),
            'n_mse_pass': len(m_set),
            'n_shared': len(shared),
            'n_union': len(p_set | m_set),
            'jaccard': len(shared) / len(p_set | m_set) if (p_set | m_set) else np.nan,
            'expected_shared_if_independent': expected,
            'probability_of_zero_overlap_fixed_set_sizes': p_zero,
        })
pearson_mse_overlap = pd.DataFrame(overlap_rows)
pearson_mse_overlap.to_csv(TABLE_DIR / 'pearson_mse_overlap_joint_universe.csv', index=False)
display(pearson_mse_overlap)

### Interpreting Pearson–MSE overlap

A zero shared set is not automatically an error. It means that no gene in the **jointly evaluated universe** passes both the Pearson and MSE thresholds. The expected overlap under random placement of fixed-size sets is approximately \(A B / N\), where \(A\) and \(B\) are the two passing-set sizes and \(N\) is the joint universe. When this expectation is below one, zero can be unsurprising. The exact fixed-set probability is exported in `pearson_mse_overlap_joint_universe.csv`.

A zero is more suspicious when the expected overlap is large or the exact zero-overlap probability is very small. In that case, verify identifier harmonization, score scales, threshold directions and whether both sets were restricted to the same evaluated genes.


## 5. Atlas universe audit

The complete group-level universe table is exported rather than printed in full because GOA and pathway resources may contain thousands of groups. The notebook displays source summaries and the largest groups for each threshold; the CSV retains every group.


In [ ]:
atlas_unique = atlas_long.drop_duplicates(['source', 'collection', 'group_id', 'group_name', 'gene_symbol']).copy()
loaded_group_sizes = (
    atlas_unique.groupby(['source', 'collection', 'group_id', 'group_name'], as_index=False)
    .agg(n_genes_atlas_loaded=('gene_symbol', 'nunique'))
)

tested_group_sizes = (
    atlas_unique.loc[atlas_unique.gene_symbol.isin(tested_any_genes)]
    .groupby(['source', 'collection', 'group_id', 'group_name'], as_index=False)
    .agg(n_genes_tested_any_metric=('gene_symbol', 'nunique'))
)

group_reference = loaded_group_sizes.merge(
    tested_group_sizes, on=['source', 'collection', 'group_id', 'group_name'], how='left'
)
group_reference['n_genes_tested_any_metric'] = group_reference['n_genes_tested_any_metric'].fillna(0).astype(int)
group_reference['pct_loaded_group_tested_any_metric'] = safe_percent(
    group_reference['n_genes_tested_any_metric'], group_reference['n_genes_atlas_loaded']
)

metric_group_frames = []
for metric, genes in metric_universe_sets.items():
    temp = (
        atlas_unique.loc[atlas_unique.gene_symbol.isin(genes)]
        .groupby(['source', 'collection', 'group_id', 'group_name'], as_index=False)
        .agg(n_genes_in_metric_universe=('gene_symbol', 'nunique'))
    )
    temp['metric'] = metric
    metric_group_frames.append(temp)
metric_group_sizes = pd.concat(metric_group_frames, ignore_index=True)

# Full condition-by-group grid: groups absent from a metric universe are retained with zero.
condition_group_universe = (
    condition_table[['condition_id', 'condition', 'metric', 'threshold', 'operator']].assign(_key=1)
    .merge(group_reference.assign(_key=1), on='_key', how='inner')
    .drop(columns='_key')
    .merge(
        metric_group_sizes,
        on=['metric', 'source', 'collection', 'group_id', 'group_name'],
        how='left',
    )
)
condition_group_universe['n_genes_in_metric_universe'] = condition_group_universe['n_genes_in_metric_universe'].fillna(0).astype(int)
condition_group_universe['pct_loaded_group_in_metric_universe'] = safe_percent(
    condition_group_universe['n_genes_in_metric_universe'], condition_group_universe['n_genes_atlas_loaded']
)
condition_group_universe.to_csv(TABLE_DIR / 'atlas_group_universe_by_threshold.csv', index=False)

source_universe_rows = []
for source, source_df in atlas_unique.groupby('source', sort=True):
    source_genes = set(source_df.gene_symbol)
    source_universe_rows.append({
        'source': source,
        'n_genes_atlas_loaded': len(source_genes),
        'n_genes_tested_any_metric': len(source_genes & tested_any_genes),
        'n_genes_pearson_universe': len(source_genes & metric_universe_sets['Pearson']),
        'n_genes_mse_universe': len(source_genes & metric_universe_sets['MSE']),
        'n_genes_joint_universe': len(source_genes & joint_metric_genes),
    })
source_universe_summary = pd.DataFrame(source_universe_rows)
for col in ['n_genes_tested_any_metric', 'n_genes_pearson_universe', 'n_genes_mse_universe', 'n_genes_joint_universe']:
    source_universe_summary[f'pct_{col.removeprefix("n_genes_")}_of_loaded'] = safe_percent(
        source_universe_summary[col], source_universe_summary['n_genes_atlas_loaded']
    )
source_universe_summary.to_csv(TABLE_DIR / 'atlas_source_universe_summary.csv', index=False)
display(source_universe_summary)

# Genes evaluated by the model but absent from every loaded atlas source.
atlas_genes_any_source = set(atlas_unique.gene_symbol)
unmatched_model_genes = gene_metrics.loc[
    gene_metrics.gene_symbol.isin(tested_any_genes - atlas_genes_any_source),
    ['gene_symbol', 'pearson', 'mse', 'n_source_rows', 'n_protein_keys', 'n_methods'],
].sort_values('gene_symbol')
unmatched_model_genes.to_csv(TABLE_DIR / 'model_genes_unmatched_to_any_atlas.csv', index=False)
print(f'Model-tested genes unmatched to every loaded atlas: {len(unmatched_model_genes):,}')

for case in condition_table.itertuples(index=False):
    sub = condition_group_universe.loc[condition_group_universe.condition_id.eq(case.condition_id)].copy()
    source_summary = (
        sub.groupby(['source', 'collection'], as_index=False)
        .agg(
            n_groups=('group_id', 'nunique'),
            n_groups_with_metric_members=('n_genes_in_metric_universe', lambda s: int((s > 0).sum())),
            total_unique_group_memberships_in_metric_universe=('n_genes_in_metric_universe', 'sum'),
        )
    )
    display(Markdown(f'### Universe summary — {case.condition}'))
    display(source_summary)
    display(Markdown('Largest groups in this metric universe (full table exported):'))
    display(
        sub.sort_values('n_genes_in_metric_universe', ascending=False)
        [['source', 'collection', 'group_name', 'n_genes_in_metric_universe']]
        .head(15)
        .reset_index(drop=True)
    )

## 6. Unified group analysis

One master table now replaces the separate enrichment and capture computations from the source notebooks. It includes groups with zero overlap, so absence of capture is distinguishable from absence of evaluation.


In [ ]:
analysis_frames = []
for case in condition_table.itertuples(index=False):
    universe_genes = metric_universe_sets[case.metric]
    selected_genes = set(threshold_membership.loc[threshold_membership.condition_id.eq(case.condition_id), 'gene_symbol'])
    N = len(universe_genes)
    K = len(selected_genes)

    groups = condition_group_universe.loc[
        condition_group_universe.condition_id.eq(case.condition_id)
        & condition_group_universe.n_genes_in_metric_universe.ge(MIN_GROUP_SIZE_EXPORT)
    ].copy()

    selected_counts = (
        atlas_unique.loc[atlas_unique.gene_symbol.isin(selected_genes)]
        .groupby('group_id')['gene_symbol'].nunique()
        .rename('n_genes_passing_in_group')
    )
    groups = groups.merge(selected_counts, on='group_id', how='left')
    groups['n_genes_passing_in_group'] = groups['n_genes_passing_in_group'].fillna(0).astype(int)
    groups['metric_universe_size'] = N
    groups['threshold_set_size'] = K

    groups['pct_threshold_set_in_group'] = safe_percent(groups['n_genes_passing_in_group'], K)
    groups['pct_metric_universe_group_captured'] = safe_percent(groups['n_genes_passing_in_group'], groups['n_genes_in_metric_universe'])
    groups['pct_tested_any_group_captured'] = safe_percent(groups['n_genes_passing_in_group'], groups['n_genes_tested_any_metric'])
    groups['pct_loaded_atlas_group_captured'] = safe_percent(groups['n_genes_passing_in_group'], groups['n_genes_atlas_loaded'])
    groups['expected_overlap'] = K * groups['n_genes_in_metric_universe'] / N if N else np.nan
    groups['fold_enrichment'] = np.where(groups.expected_overlap > 0, groups.n_genes_passing_in_group / groups.expected_overlap, np.nan)

    odds, pvals = [], []
    for row in groups.itertuples(index=False):
        a = int(row.n_genes_passing_in_group)
        b = K - a
        c = int(row.n_genes_in_metric_universe) - a
        d = N - a - b - c
        if min(a, b, c, d) < 0 or fisher_exact is None:
            odds.append(np.nan)
            pvals.append(np.nan)
        else:
            result = fisher_exact([[a, b], [c, d]], alternative='greater')
            odds.append(float(result.statistic if hasattr(result, 'statistic') else result[0]))
            pvals.append(float(result.pvalue if hasattr(result, 'pvalue') else result[1]))
    groups['odds_ratio'] = odds
    groups['pvalue'] = pvals
    groups['fdr_within_condition'] = bh_fdr(groups['pvalue'])
    groups['fdr_within_source_condition'] = np.nan
    for source, idx in groups.groupby('source').groups.items():
        groups.loc[idx, 'fdr_within_source_condition'] = bh_fdr(groups.loc[idx, 'pvalue'])
    groups['well_predicted_group'] = (
        groups['fdr_within_source_condition'].le(FDR_CUTOFF)
        & groups['fold_enrichment'].ge(FOLD_ENRICHMENT_CUTOFF)
        & groups['n_genes_passing_in_group'].ge(MIN_OVERLAP_FOR_FLAG)
    )
    analysis_frames.append(groups)

atlas_group_analysis = pd.concat(analysis_frames, ignore_index=True) if analysis_frames else pd.DataFrame()
atlas_group_analysis = atlas_group_analysis.sort_values(
    ['condition', 'source', 'fdr_within_source_condition', 'fold_enrichment'],
    ascending=[True, True, True, False],
).reset_index(drop=True)
atlas_group_analysis.to_csv(TABLE_DIR / 'pan_atlas_group_analysis_all.csv', index=False)

source_condition_summary = (
    atlas_group_analysis.groupby(['condition', 'metric', 'source', 'collection'], as_index=False)
    .agg(
        n_groups_evaluated=('group_id', 'nunique'),
        n_well_predicted_groups=('well_predicted_group', 'sum'),
        median_metric_group_capture_pct=('pct_metric_universe_group_captured', 'median'),
        max_metric_group_capture_pct=('pct_metric_universe_group_captured', 'max'),
        median_fold_enrichment=('fold_enrichment', 'median'),
        max_fold_enrichment=('fold_enrichment', 'max'),
    )
)
source_condition_summary.to_csv(TABLE_DIR / 'source_condition_summary.csv', index=False)
display(source_condition_summary)

capture_cutoff_rows = []
for keys, sub in atlas_group_analysis.groupby(['condition', 'source', 'collection'], sort=False):
    condition, source, collection = keys
    row = {'condition': condition, 'source': source, 'collection': collection, 'n_groups': sub.group_id.nunique()}
    for cutoff in (25, 50, 75, 90, 100):
        row[f'n_groups_capture_ge_{cutoff}pct'] = sub.loc[sub.pct_metric_universe_group_captured.ge(cutoff), 'group_id'].nunique()
    capture_cutoff_rows.append(row)
capture_cutoff_summary = pd.DataFrame(capture_cutoff_rows)
capture_cutoff_summary.to_csv(TABLE_DIR / 'capture_cutoff_summary.csv', index=False)

well_predicted = atlas_group_analysis.loc[atlas_group_analysis.well_predicted_group].copy()
well_predicted.to_csv(TABLE_DIR / 'well_predicted_groups.csv', index=False)
print(f'Group-condition rows: {len(atlas_group_analysis):,}')
print(f'Well-predicted group-condition rows: {len(well_predicted):,}')

## 7. Curated main figures

The main figures answer distinct questions:

1. **source universe decomposition** — how much of each loaded atlas is evaluated and selected;
2. **significant-group counts** — where statistically supported signals occur;
3. **ranked enrichment** — which groups are most enriched at each threshold;
4. **source capture heatmaps** — how group member coverage changes across thresholds;
5. **capture–enrichment landscape** — why high capture and high enrichment are not equivalent.

The notebook intentionally omits repetitive lollipops, top-group tiles, duplicate bubble matrices and multiple arbitrary capture-cutoff heatmaps.


In [ ]:
SOURCE_COLORS = OrderedDict([
    ('HPA', '#4E79A7'),
    ('UniProtKB', '#F28E2B'),
    ('GOA', '#59A14F'),
    ('Reactome', '#E15759'),
    ('WikiPathways', '#B07AA1'),
])
UNIVERSE_COMPONENT_COLORS = OrderedDict([
    ('Pass threshold', '#4E79A7'),
    ('Evaluated, does not pass', '#B07AA1'),
    ('Not evaluated for metric', '#E6E6E6'),
])

def source_condition_universe_decomposition():
    rows = []
    for case in condition_table.itertuples(index=False):
        metric_genes = metric_universe_sets[case.metric]
        selected_genes = set(threshold_membership.loc[threshold_membership.condition_id.eq(case.condition_id), 'gene_symbol'])
        for source, sub in atlas_unique.groupby('source', sort=True):
            loaded = set(sub.gene_symbol)
            n_loaded = len(loaded)
            n_eval = len(loaded & metric_genes)
            n_pass = len(loaded & selected_genes)
            rows.append({
                'condition_id': case.condition_id, 'condition': case.condition, 'metric': case.metric,
                'source': source, 'n_loaded': n_loaded, 'n_pass': n_pass,
                'n_evaluated_not_pass': n_eval - n_pass,
                'n_not_evaluated': n_loaded - n_eval,
                'pct_pass': 100 * n_pass / n_loaded if n_loaded else np.nan,
                'pct_evaluated_not_pass': 100 * (n_eval - n_pass) / n_loaded if n_loaded else np.nan,
                'pct_not_evaluated': 100 * (n_loaded - n_eval) / n_loaded if n_loaded else np.nan,
            })
    return pd.DataFrame(rows)

source_decomposition = source_condition_universe_decomposition()
source_decomposition.to_csv(TABLE_DIR / 'source_universe_decomposition_by_threshold.csv', index=False)

def plot_source_universe_decomposition_grid():
    fig, axes, _, _ = compact_grid(len(condition_table), panel_size=(6.1, 4.2), max_cols=2)
    for ax, case in zip(axes, condition_table.itertuples(index=False)):
        sub = source_decomposition.loc[source_decomposition.condition_id.eq(case.condition_id)].sort_values('source')
        x = np.arange(len(sub))
        bottom = np.zeros(len(sub))
        for label, col in [
            ('Pass threshold', 'pct_pass'),
            ('Evaluated, does not pass', 'pct_evaluated_not_pass'),
            ('Not evaluated for metric', 'pct_not_evaluated'),
        ]:
            values = sub[col].to_numpy(dtype=float)
            ax.bar(x, values, bottom=bottom, color=UNIVERSE_COMPONENT_COLORS[label], edgecolor='white', linewidth=0.35, label=label)
            bottom += values
        ax.set_xticks(x)
        ax.set_xticklabels(sub.source, rotation=28, ha='right')
        ax.set_ylim(0, 100)
        ax.set_ylabel('Share of loaded source genes (%)')
        ax.set_xlabel('Atlas source')
        ax.set_title(case.condition)
        ax.grid(axis='y', alpha=0.16, linewidth=0.6)
    handles = [mpl.patches.Patch(facecolor=color, label=label) for label, color in UNIVERSE_COMPONENT_COLORS.items()]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.005), ncol=3, frameon=False)
    fig.suptitle('Loaded atlas genes: threshold pass, evaluated non-pass, and not evaluated', y=0.995, fontsize=12, fontweight='bold')
    fig.tight_layout(rect=(0, 0.10, 1, 0.95), h_pad=1.2, w_pad=0.9)
    finish_figure(fig, 'main_01_source_universe_decomposition_grid')

plot_source_universe_decomposition_grid()

In [ ]:
def plot_well_predicted_counts_grid():
    fig, axes, _, _ = compact_grid(len(condition_table), panel_size=(6.1, 4.0), max_cols=2)
    sources = sorted(atlas_group_analysis.source.unique())
    for ax, case in zip(axes, condition_table.itertuples(index=False)):
        sub = atlas_group_analysis.loc[atlas_group_analysis.condition_id.eq(case.condition_id)]
        counts = sub.loc[sub.well_predicted_group].groupby('source').group_id.nunique().reindex(sources, fill_value=0)
        x = np.arange(len(sources))
        ax.bar(x, counts.values, color=[SOURCE_COLORS.get(source, '#7F7F7F') for source in sources], width=0.76)
        ax.set_xticks(x)
        ax.set_xticklabels(sources, rotation=28, ha='right')
        ax.set_ylabel('Well-predicted groups (n)')
        ax.set_xlabel('Atlas source')
        ax.set_title(case.condition)
        ax.grid(axis='y', alpha=0.16, linewidth=0.6)
    fig.suptitle('Statistically supported enriched groups by atlas source', y=0.995, fontsize=12, fontweight='bold')
    fig.tight_layout(rect=(0, 0.04, 1, 0.95), h_pad=1.2, w_pad=0.9)
    finish_figure(fig, 'main_02_well_predicted_group_counts_grid')

plot_well_predicted_counts_grid()

In [ ]:
def ranked_groups_for_case(case_id, top_n=TOP_N_RANKED):
    sub = atlas_group_analysis.loc[
        atlas_group_analysis.condition_id.eq(case_id)
        & atlas_group_analysis.n_genes_in_metric_universe.ge(MIN_GROUP_SIZE_PLOTS)
        & atlas_group_analysis.n_genes_passing_in_group.ge(MIN_OVERLAP_FOR_FLAG)
    ].copy()
    significant = sub.loc[sub.well_predicted_group].copy()
    chosen = significant if not significant.empty else sub
    chosen = chosen.assign(
        rank_fdr=chosen.fdr_within_source_condition.fillna(1.0),
        rank_fold=chosen.fold_enrichment.fillna(0),
    )
    return chosen.sort_values(
        ['well_predicted_group', 'rank_fdr', 'rank_fold', 'n_genes_passing_in_group'],
        ascending=[False, True, False, False],
    ).head(top_n)

def plot_ranked_enrichment_grid():
    fig, axes, _, _ = compact_grid(len(condition_table), panel_size=(7.2, 5.6), max_cols=2)
    for ax, case in zip(axes, condition_table.itertuples(index=False)):
        sub = ranked_groups_for_case(case.condition_id).iloc[::-1]
        if sub.empty:
            ax.text(0.5, 0.5, 'No groups met the plotting filters', ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
            continue
        labels = [wrap_label(f'{r.source} | {r.group_name}', 48) for r in sub.itertuples(index=False)]
        y = np.arange(len(sub))
        x = np.log2(sub.fold_enrichment.clip(lower=1e-9))
        sizes = 34 + 13 * np.sqrt(sub.n_genes_passing_in_group.clip(lower=1))
        ax.scatter(
            x, y, s=sizes,
            c=[SOURCE_COLORS.get(source, '#7F7F7F') for source in sub.source],
            edgecolors='white', linewidth=0.55, alpha=0.92,
        )
        ax.axvline(np.log2(FOLD_ENRICHMENT_CUTOFF), color='black', linestyle='--', linewidth=0.8, alpha=0.6)
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.set_xlabel('log2 fold enrichment')
        ax.set_ylabel('Atlas group')
        ax.set_title(case.condition + (' — significant groups' if sub.well_predicted_group.any() else ' — best available groups'))
        ax.grid(axis='x', alpha=0.16, linewidth=0.6)
    handles = [mpl.patches.Patch(facecolor=color, label=source) for source, color in SOURCE_COLORS.items() if source in atlas_group_analysis.source.unique()]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.005), ncol=min(5, len(handles)), frameon=False)
    fig.suptitle('Ranked atlas-group enrichment; marker size reflects passing overlap', y=0.997, fontsize=12, fontweight='bold')
    fig.tight_layout(rect=(0, 0.08, 1, 0.96), h_pad=1.3, w_pad=1.0)
    finish_figure(fig, 'main_03_ranked_enrichment_grid')

plot_ranked_enrichment_grid()

In [ ]:
def plot_source_capture_heatmap_grid():
    sources = sorted(atlas_group_analysis.source.unique())
    fig, axes, _, _ = compact_grid(len(sources), panel_size=(7.0, 5.3), max_cols=2)
    image = None
    for ax, source in zip(axes, sources):
        sub = atlas_group_analysis.loc[
            atlas_group_analysis.source.eq(source)
            & atlas_group_analysis.n_genes_in_metric_universe.ge(MIN_GROUP_SIZE_PLOTS)
        ].copy()
        if sub.empty:
            ax.set_axis_off()
            continue
        ranked = (
            sub.groupby(['group_id', 'group_name'], as_index=False)
            .agg(max_capture=('pct_metric_universe_group_captured', 'max'), max_overlap=('n_genes_passing_in_group', 'max'))
            .sort_values(['max_capture', 'max_overlap'], ascending=False)
            .head(TOP_N_CAPTURE_PER_SOURCE)
        )
        matrix = (
            sub.loc[sub.group_id.isin(ranked.group_id)]
            .pivot_table(index=['group_id', 'group_name'], columns='condition', values='pct_metric_universe_group_captured', aggfunc='max', fill_value=0)
            .reindex(columns=condition_order, fill_value=0)
            .reset_index()
            .merge(ranked[['group_id', 'max_capture']], on='group_id')
            .sort_values('max_capture', ascending=True)
            .set_index(['group_id', 'group_name'])
            .drop(columns='max_capture')
        )
        image = ax.imshow(matrix.to_numpy(dtype=float), aspect='auto', vmin=0, vmax=100, interpolation='nearest', cmap='viridis')
        ax.set_xticks(np.arange(len(matrix.columns)))
        ax.set_xticklabels(matrix.columns, rotation=32, ha='right')
        ax.set_yticks(np.arange(len(matrix.index)))
        ax.set_yticklabels([wrap_label(name, 42) for _, name in matrix.index])
        ax.set_xlabel('Threshold set')
        ax.set_ylabel('Atlas group')
        ax.set_title(source)
        ax.tick_params(axis='both', length=0)
    if image is not None:
        cbar = fig.colorbar(image, ax=list(axes), fraction=0.018, pad=0.012)
        cbar.set_label('Metric-universe group captured (%)')
    fig.suptitle('Top captured groups within each atlas source', y=0.997, fontsize=12, fontweight='bold')
    fig.subplots_adjust(left=0.28, right=0.92, top=0.94, bottom=0.10, hspace=0.38, wspace=0.52)
    finish_figure(fig, 'main_04_source_capture_heatmap_grid')

plot_source_capture_heatmap_grid()

In [ ]:
def plot_capture_enrichment_landscape_grid():
    fig, axes, _, _ = compact_grid(len(condition_table), panel_size=(6.2, 4.6), max_cols=2)
    for ax, case in zip(axes, condition_table.itertuples(index=False)):
        sub = atlas_group_analysis.loc[
            atlas_group_analysis.condition_id.eq(case.condition_id)
            & atlas_group_analysis.n_genes_in_metric_universe.ge(MIN_GROUP_SIZE_PLOTS)
            & atlas_group_analysis.fold_enrichment.gt(0)
        ].copy()
        for source, source_sub in sub.groupby('source'):
            ax.scatter(
                source_sub.pct_metric_universe_group_captured,
                np.log2(source_sub.fold_enrichment),
                s=12 + 4.0 * np.sqrt(source_sub.n_genes_passing_in_group.clip(lower=0)),
                c=SOURCE_COLORS.get(source, '#7F7F7F'),
                alpha=0.45, edgecolors='none', label=source,
            )
        ax.axhline(np.log2(FOLD_ENRICHMENT_CUTOFF), color='black', linestyle='--', linewidth=0.8, alpha=0.6)
        ax.set_xlim(0, 100)
        ax.set_xlabel('Metric-universe group captured (%)')
        ax.set_ylabel('log2 fold enrichment')
        ax.set_title(case.condition)
        ax.grid(alpha=0.14, linewidth=0.6)
    handles = [mpl.patches.Patch(facecolor=color, label=source) for source, color in SOURCE_COLORS.items() if source in atlas_group_analysis.source.unique()]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.005), ncol=min(5, len(handles)), frameon=False)
    fig.suptitle('Capture and enrichment answer different biological questions', y=0.997, fontsize=12, fontweight='bold')
    fig.tight_layout(rect=(0, 0.09, 1, 0.95), h_pad=1.2, w_pad=0.9)
    finish_figure(fig, 'main_05_capture_enrichment_landscape_grid')

plot_capture_enrichment_landscape_grid()

## 8. Resource-quality and composition checks

These figures are diagnostic rather than principal biological results. They are retained because multi-atlas conclusions can be distorted by source redundancy and very different group-size distributions.


In [ ]:
def plot_resource_gene_jaccard():
    sources = sorted(atlas_unique.source.unique())
    gene_sets = {source: set(atlas_unique.loc[atlas_unique.source.eq(source), 'gene_symbol']) for source in sources}
    matrix = np.zeros((len(sources), len(sources)), dtype=float)
    for i, left in enumerate(sources):
        for j, right in enumerate(sources):
            union = gene_sets[left] | gene_sets[right]
            matrix[i, j] = len(gene_sets[left] & gene_sets[right]) / len(union) if union else np.nan
    resource_jaccard = pd.DataFrame(matrix, index=sources, columns=sources)
    resource_jaccard.to_csv(TABLE_DIR / 'resource_gene_jaccard.csv')
    fig, ax = plt.subplots(figsize=(7.0, 5.8))
    image = ax.imshow(matrix, vmin=0, vmax=1, cmap='viridis')
    ax.set_xticks(np.arange(len(sources)))
    ax.set_xticklabels(sources, rotation=32, ha='right')
    ax.set_yticks(np.arange(len(sources)))
    ax.set_yticklabels(sources)
    ax.set_xlabel('Atlas source')
    ax.set_ylabel('Atlas source')
    ax.set_title('Jaccard overlap of genes annotated by each atlas source')
    for i in range(len(sources)):
        for j in range(len(sources)):
            ax.text(j, i, f'{matrix[i, j]:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if matrix[i, j] < 0.45 else 'black')
    cbar = fig.colorbar(image, ax=ax, fraction=0.042, pad=0.025)
    cbar.set_label('Gene-set Jaccard index')
    fig.tight_layout()
    finish_figure(fig, 'qc_01_resource_gene_jaccard', show=DISPLAY_MAIN_FIGURES)

def plot_group_size_distribution_grid():
    sources = sorted(group_reference.source.unique())
    fig, axes, _, _ = compact_grid(len(sources), panel_size=(6.0, 4.0), max_cols=2)
    bins = np.linspace(0, max(1, np.log10(group_reference.n_genes_atlas_loaded.max())), 28)
    for ax, source in zip(axes, sources):
        sub = group_reference.loc[group_reference.source.eq(source)]
        loaded = np.log10(sub.n_genes_atlas_loaded.clip(lower=1))
        tested = np.log10(sub.n_genes_tested_any_metric.clip(lower=1))
        ax.hist(loaded, bins=bins, alpha=0.55, label='Loaded atlas group')
        ax.hist(tested, bins=bins, alpha=0.55, label='Tested-any intersection')
        ax.set_xlabel('log10 unique genes per group')
        ax.set_ylabel('Groups (n)')
        ax.set_title(source)
        ax.grid(axis='y', alpha=0.14, linewidth=0.6)
    handles = [
        mpl.patches.Patch(alpha=0.55, label='Loaded atlas group'),
        mpl.patches.Patch(alpha=0.55, label='Tested-any intersection'),
    ]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.005), ncol=2, frameon=False)
    fig.suptitle('Atlas group-size distributions before and after model-universe restriction', y=0.997, fontsize=12, fontweight='bold')
    fig.tight_layout(rect=(0, 0.09, 1, 0.95), h_pad=1.2, w_pad=0.9)
    finish_figure(fig, 'qc_02_group_size_distribution_grid', show=DISPLAY_MAIN_FIGURES)

if MAKE_QC_FIGURES:
    plot_resource_gene_jaccard()
    plot_group_size_distribution_grid()

## 9. Interpretation and output guide

### Recommended interpretation order

1. Inspect `threshold_summary.csv` and `atlas_source_universe_summary.csv` before interpreting any enrichment.
2. Use `atlas_group_universe_by_threshold.csv` to verify that a group's metric-specific denominator is sufficiently large.
3. Use `pan_atlas_group_analysis_all.csv` as the authoritative combined result table.
4. Treat `well_predicted_group` as a screening flag, not a biological truth. It requires source-specific FDR, fold enrichment and minimum overlap simultaneously.
5. Compare `pct_metric_universe_group_captured` with `pct_loaded_atlas_group_captured`. A high metric-universe capture can coexist with low loaded-atlas capture when much of the atlas group was never evaluated.
6. Use the capture–enrichment landscape to distinguish broad group recovery from statistical overrepresentation.

### Main tables

- `atlas_membership_long.csv`: normalized memberships from all parsed atlases;
- `gene_annotation_burden_by_source.csv` and `goa_evidence_summary.csv`: resource-composition audits;
- `best_pearson_mse_per_gene.csv`: independently selected best metric values and provenance;
- `threshold_membership_long.csv`: passing genes for Pearson and MSE cases;
- `atlas_group_universe_by_threshold.csv`: complete denominator audit, including zero-member groups;
- `model_genes_unmatched_to_any_atlas.csv`: identifier and coverage audit;
- `pan_atlas_group_analysis_all.csv`: capture and enrichment for every retained group-condition pair;
- `well_predicted_groups.csv`: rows passing the configured screening criteria;
- `pearson_mse_overlap_joint_universe.csv`: cross-metric overlap audit.

### Deliberately omitted repetitive figures

The merged notebook does not generate separate top-group tiles, lollipop charts, duplicate bubble matrices, 50% and 75% capture heatmaps, or multiple representations of the same ranked groups. Those views can be regenerated from the master table if a presentation requires them.


In [ ]:
# Compact final audit
required_tables = [
    'atlas_membership_long.csv',
    'best_pearson_mse_per_gene.csv',
    'threshold_membership_long.csv',
    'atlas_group_universe_by_threshold.csv',
    'pan_atlas_group_analysis_all.csv',
    'well_predicted_groups.csv',
    'pearson_mse_overlap_joint_universe.csv',
]
final_audit = pd.DataFrame({
    'table': required_tables,
    'exists': [(TABLE_DIR / name).exists() for name in required_tables],
})
display(final_audit)
print('6. pan_atlas analysis complete.')
print('Tables:', TABLE_DIR.resolve())
print('Figures:', FIGURE_DIR.resolve())

---
# Part B — HPA class heatmaps (was `5_HPA_heatmaps.ipynb`)

Coverage of HPA macro classes per cascade stage. Writes `4.2_HPA_heatmaps_outputs/`.

## 4.2 - HPA heatmaps


## 1. Configuration

In [ ]:
import gzip, io, math, os, re, textwrap, warnings, zipfile

from IPython.display import Markdown, display

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

# -----------------------------------------------------------------------------
# Shared paths and target groups
# -----------------------------------------------------------------------------
BASE_DIR = Path.cwd()
DOWNLOAD_DIR = BASE_DIR / 'pan_atlas_downloads'
OUTPUT_ROOT = BASE_DIR / '4.2_HPA_heatmaps_outputs'
SHARED_OUTPUT_DIR = OUTPUT_ROOT / 'shared'
SHARED_TABLE_DIR = SHARED_OUTPUT_DIR / 'tables'
SHARED_FIGURE_DIR = SHARED_OUTPUT_DIR / 'figures'

# These globals are switched by configure_target() before each target is run.
OUTPUT_DIR = SHARED_OUTPUT_DIR
TABLE_DIR = SHARED_TABLE_DIR
FIGURE_DIR = SHARED_FIGURE_DIR
TARGET_GROUP_NAME = 'Shared setup'
TARGET_STAGE_GROUPS = []

for folder in [OUTPUT_ROOT, SHARED_OUTPUT_DIR, SHARED_TABLE_DIR, SHARED_FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# All five uploaded variants are run sequentially. Remove an entry here if needed.
TARGET_SPECS = [
    {
        'key': 'proxy',
        'target_group_name': 'Proxy model',
        'target_stage_groups': ['Proxy model'],
        'output_folder_name': 'hpa_macroclass_proxy_single_model_mse020_030_outputs',
    },
    {
        'key': 'elasticnet',
        'target_group_name': 'ElasticNet',
        'target_stage_groups': ['ElasticNet'],
        'output_folder_name': 'hpa_macroclass_elasticnet_mse020_030_outputs',
    },
    {
        'key': 'xgb',
        'target_group_name': 'XGB',
        'target_stage_groups': ['XGB'],
        'output_folder_name': 'hpa_macroclass_xgb_mse020_030_outputs',
    },
    {
        'key': 'still_not_good',
        'target_group_name': 'Still not good',
        'target_stage_groups': ['Still not good'],
        'output_folder_name': 'hpa_macroclass_still_not_good_mse020_030_outputs',
    },
    {
        'key': 'model_good_union',
        'target_group_name': 'Model-good union: Proxy + ElasticNet + XGB',
        'target_stage_groups': ['Proxy model', 'ElasticNet', 'XGB'],
        'output_folder_name': 'hpa_macroclass_all_model_threshold_good_union_mse020_030_outputs',
    },
]

HPA_MACRO_CLASSES = ['Enzymes', 'Metabolic proteins', 'CD markers', 'Blood group antigen proteins', 'Nuclear receptors', 'Transporters', 'Ribosomal proteins', 'G-protein coupled receptors', 'Voltage-gated ion channels', 'Predicted membrane proteins', 'Predicted secreted proteins', 'Predicted intracellular proteins', 'Plasma proteins', 'Transcription factors', 'RNA polymerase related proteins', 'RAS pathway related proteins', 'Citric acid cycle related proteins', 'Essential proteins', 'Immunoglobulin genes', 'T-cell receptor genes']

# Local-first HPA file search. No internet/download is used.
HPA_CANDIDATE_FILES = [
    DOWNLOAD_DIR / 'hpa_proteinatlas.tsv.zip', DOWNLOAD_DIR / 'proteinatlas.tsv.zip',
    DOWNLOAD_DIR / 'hpa_proteinatlas.tsv.gz', DOWNLOAD_DIR / 'proteinatlas.tsv.gz',
    DOWNLOAD_DIR / 'hpa_proteinatlas.tsv', DOWNLOAD_DIR / 'proteinatlas.tsv',
    BASE_DIR / 'hpa_proteinatlas.tsv.zip', BASE_DIR / 'proteinatlas.tsv.zip',
]

# Manual override: paste one or more cascade_partition_proteins_long.csv paths here if auto-search is too broad.
CASCADE_LONG_FILES = [str(p) for p in [BASE_DIR / 'procedural_unified_tables' / 'train_linear' / 'protein_names_by_cascade_partition' / 'cascade_partition_proteins_long.csv'] if p.exists()]  # paper primary proxy mode (train-fitted linear); empty -> auto-search
AUTO_SEARCH_CASCADE_FILES = True

# Explicitly exclude old procedural and procedural-linear/proxy-linear exports.
EXCLUDED_SOURCE_IDS = {'procedural', 'proxy_linear'}
EXCLUDED_PATH_KEYWORDS = {'procedural_tables', 'procedural_proxy_norm_tables'}

SELECTED_METRICS = ['Pearson', 'Cosine', 'MSE']
SELECTED_THRESHOLDS_BY_METRIC = {
    'Pearson': [0.50, 0.70],
    'Cosine': [0.50, 0.70],
    'MSE': [0.20, 0.30],
}
THRESHOLD_TOLERANCE = 1e-7
MIN_HIGH_COVERAGE_PCT = 70.0

SAVE_PNG = True
SAVE_SVG = True
SAVE_PDF = False
DPI = 300

# The merged notebook displays every plot sequentially by default.
DISPLAY_ALL_PLOTS = True
MAX_HIGH_COVERAGE_ROWS_PER_PLOT = 30

mpl.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': DPI,
    'font.size': 11,
    'axes.titlesize': 15,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'xtick.labelsize': 9.5,
    'ytick.labelsize': 9.5,
    'legend.fontsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

STAGE_COLORS = OrderedDict([
    ('Proxy model', '#4E79A7'),
    ('ElasticNet', '#F28E2B'),
    ('XGB', '#E15759'),
    ('Still not good', '#B07AA1'),
    ('scTranslator', '#76B7B2'),
    ('Other', '#9D9D9D'),
])
METRIC_ORDER = {'Pearson': 0, 'Cosine': 1, 'MSE': 2}

print(f'Base directory: {BASE_DIR}')
print(f'Merged output root: {OUTPUT_ROOT}')
print(f'Target groups configured: {len(TARGET_SPECS)}')
print(f'HPA macro protein classes configured: {len(HPA_MACRO_CLASSES)}')
print(f'Display every generated plot: {DISPLAY_ALL_PLOTS}')
print('Selected thresholds by metric:', SELECTED_THRESHOLDS_BY_METRIC)

## 2. Shared helper functions

In [ ]:
def wrap_label(text, width=30):
    return '\n'.join(textwrap.wrap(str(text), width=width, break_long_words=False))

def clean_gene_symbol(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if not value or value.lower() in {'nan', 'none', 'na', 'null'}:
        return np.nan
    # Partition proteins are gene symbols in the cascade exports. If a compound field appears, keep the first symbol.
    return re.split(r'[;|,/\s]+', value)[0].upper()

def split_hpa_terms(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none', 'na', 'null'}:
        return []
    out, seen = [], set()
    for part in re.split(r'[,;|]+', text):
        term = re.sub(r'\s+', ' ', part.strip())
        key = normalize_text(term)
        if term and key not in seen:
            out.append(term)
            seen.add(key)
    return out

def pick_column(df, candidates, required=True):
    norm_to_col = {normalize_text(c): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        if normalize_text(cand) in norm_to_col:
            return norm_to_col[normalize_text(cand)]
    for cand in candidates:
        words = normalize_text(cand).split()
        for col in df.columns:
            if all(w in normalize_text(col) for w in words):
                return col
    if required:
        raise KeyError(f'Could not find any of {candidates}. Available columns include {list(df.columns)[:40]}')
    return None

def read_table_any(path):
    path = Path(path)
    suffixes = [s.lower() for s in path.suffixes]
    if suffixes and suffixes[-1] == '.zip':
        with zipfile.ZipFile(path) as zf:
            names = [n for n in zf.namelist() if not n.endswith('/') and n.lower().endswith(('.tsv', '.txt', '.csv'))]
            if not names:
                raise ValueError(f'No delimited .tsv/.txt/.csv file was found inside {path}')
            inner = names[0]
            sep = ',' if inner.lower().endswith('.csv') else '\t'
            with zf.open(inner) as fh:
                return pd.read_csv(fh, sep=sep, dtype=str, low_memory=False)
    if suffixes and suffixes[-1] == '.gz':
        sep = ',' if len(suffixes) >= 2 and suffixes[-2] == '.csv' else '\t'
        return pd.read_csv(path, sep=sep, dtype=str, compression='gzip', low_memory=False)
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path, dtype=str, low_memory=False)
    return pd.read_csv(path, sep='\t', dtype=str, low_memory=False)

def save_figure(fig, stem):
    stem = safe_filename(stem)
    paths = []
    if SAVE_PNG:
        p = FIGURE_DIR / f'{stem}.png'
        fig.savefig(p, bbox_inches='tight', dpi=DPI)
        paths.append(p)
    if SAVE_SVG:
        p = FIGURE_DIR / f'{stem}.svg'
        fig.savefig(p, bbox_inches='tight')
        paths.append(p)
    if SAVE_PDF:
        p = FIGURE_DIR / f'{stem}.pdf'
        fig.savefig(p, bbox_inches='tight')
        paths.append(p)
    return paths

def show_or_close(fig, show=False):
    if show:
        plt.show()
    else:
        plt.close(fig)

def canonical_stage_group(raw_stage):
    text = normalize_text(raw_stage)
    if not text or text in {'nan', 'none'}:
        return 'Other'
    # Proxy is treated as one model; Proxy Pearson and Proxy Spearman are intentionally not split.
    if 'proxy' in text:
        return 'Proxy model'
    if 'enet' in text or 'elastic' in text:
        return 'ElasticNet'
    if 'xgb' in text or 'xgboost' in text:
        return 'XGB'
    if ('still' in text and 'good' in text) or 'no valid mse' in text:
        return 'Still not good'
    if 'sctranslator' in text:
        return 'scTranslator'
    return str(raw_stage).strip() or 'Other'

def parse_metric_family(analysis, label):
    text = f'{analysis} {label}'.casefold()
    if 'pearson' in text:
        return 'Pearson'
    if 'cosine' in text:
        return 'Cosine'
    if 'mse' in text:
        return 'MSE'
    return 'Other'

def parse_threshold_value(row):
    # Prefer explicit numeric threshold columns when available.
    for col in ['threshold', 'prediction_threshold', 'corr_threshold', 'mse_threshold', 'metric_threshold']:
        if col in row.index and pd.notna(row[col]):
            try:
                value = float(str(row[col]).strip())
                if math.isfinite(value):
                    return value
            except Exception:
                pass
    label = str(row.get('threshold_label', ''))
    nums = re.findall(r'[-+]?\d*\.\d+|[-+]?\d+', label)
    if not nums:
        return np.nan
    value = float(nums[-1])
    metric = parse_metric_family(row.get('analysis', ''), label)
    if metric in {'Pearson', 'Cosine'} and 1 < value <= 100:
        value /= 100.0
    return value

def threshold_name(metric, value):
    if pd.isna(value):
        return 'no fixed threshold'
    if metric in {'Pearson', 'Cosine'}:
        return f'>= {value:.2f}'
    if metric == 'MSE':
        return f'<= {value:.2f}'
    return f'{value:.2f}'

def preferred_analysis_rank(analysis, metric):
    a = normalize_text(analysis)
    if metric == 'Pearson':
        if a == 'pearson_threshold_sweep': return 0
        if a == 'main_pearson_cascade': return 1
        if 'pearson' in a and 'threshold' in a: return 2
        if 'pearson' in a: return 3
    if metric == 'Cosine':
        if a == 'cosine_threshold_sweep': return 0
        if a == 'main_cosine_cascade': return 1
        if 'cosine' in a and 'threshold' in a: return 2
        if 'cosine' in a: return 3
    if metric == 'MSE':
        if a == 'mse_threshold_sweep': return 0
        if a == 'main_mse_cascade': return 1
        if 'mse' in a and 'threshold' in a and 'percentile' not in a: return 2
        if 'mse' in a: return 3
    return 99

def infer_source_id(path):
    p = Path(path)
    parts = [x.lower() for x in p.parts]
    if 'procedural_proxy_z_tables' in parts:
        return 'proxy_z'
    if 'procedural_proxy_norm_tables' in parts:
        return 'proxy_linear'
    if 'procedural_tables' in parts:
        return 'procedural'
    for part in reversed(p.parts):
        if part.endswith('_tables') or part.startswith('procedural'):
            return safe_filename(part)
    return safe_filename(p.parent.parent.name if len(p.parts) >= 3 else p.parent.name)

def find_cascade_long_files():
    manual = [Path(p) for p in CASCADE_LONG_FILES]
    files = [p for p in manual if p.exists()]
    if AUTO_SEARCH_CASCADE_FILES:
        standard = [
            BASE_DIR / 'procedural_proxy_z_tables' / 'protein_names_by_cascade_partition' / 'cascade_partition_proteins_long.csv',
            BASE_DIR / 'protein_names_by_cascade_partition' / 'cascade_partition_proteins_long.csv',
        ]
        for p in standard:
            if p.exists():
                files.append(p)
        files.extend(BASE_DIR.rglob('protein_names_by_cascade_partition/cascade_partition_proteins_long.csv'))
    # De-duplicate while preserving order, and drop old procedural/proxy-linear exports.
    out, seen = [], set()
    for p in files:
        sid = infer_source_id(p)
        path_text = str(p).lower()
        if sid in EXCLUDED_SOURCE_IDS or any(k in path_text for k in EXCLUDED_PATH_KEYWORDS):
            continue
        rp = p.resolve()
        if rp not in seen:
            out.append(p)
            seen.add(rp)
    return out

## 3. Load HPA macro protein classes and cascade partitions once

The HPA membership table and standardized cascade partition table are shared by every target.

In [ ]:
# -----------------------------
# HPA macro protein-class memberships
# -----------------------------
hpa_path = next((p for p in HPA_CANDIDATE_FILES if p.exists()), None)
if hpa_path is None:
    raise FileNotFoundError(
        'Could not find a local HPA protein atlas file. Expected pan_atlas_downloads/hpa_proteinatlas.tsv.zip or proteinatlas.tsv.zip.'
    )

hpa_raw = read_table_any(hpa_path)
gene_col = pick_column(hpa_raw, ['Gene', 'gene', 'Gene name', 'Gene symbol', 'gene_symbol', 'Gene name according to HGNC'])
protein_class_col = pick_column(hpa_raw, ['Protein class', 'protein_class', 'Protein classes'])
macro_map = {normalize_text(x): x for x in HPA_MACRO_CLASSES}
rows = []
for gene, protein_class in zip(hpa_raw[gene_col], hpa_raw[protein_class_col]):
    g = clean_gene_symbol(gene)
    if pd.isna(g):
        continue
    for term in split_hpa_terms(protein_class):
        macro_class = macro_map.get(normalize_text(term))
        if macro_class is not None:
            rows.append((g, macro_class, str(hpa_path)))

hpa_macro = pd.DataFrame(rows, columns=['gene_symbol', 'hpa_macro_class', 'hpa_source_file']).drop_duplicates(['gene_symbol', 'hpa_macro_class'])
if hpa_macro.empty:
    raise ValueError('No HPA macro protein classes matched. Check the Protein class column and HPA_MACRO_CLASSES.')
hpa_macro['hpa_macro_class'] = pd.Categorical(hpa_macro['hpa_macro_class'], categories=HPA_MACRO_CLASSES, ordered=True)
hpa_macro = hpa_macro.sort_values(['hpa_macro_class', 'gene_symbol']).reset_index(drop=True)
hpa_macro.to_csv(TABLE_DIR / 'hpa_macro_class_membership_long.csv', index=False)

hpa_group_sizes = hpa_macro.groupby('hpa_macro_class', observed=True)['gene_symbol'].nunique().reset_index(name='n_hpa_genes')
hpa_group_sizes.to_csv(TABLE_DIR / 'hpa_macro_class_sizes_in_hpa_file.csv', index=False)

print('HPA file loaded:', hpa_path)
print(f'HPA macro-class memberships retained: {len(hpa_macro):,}')
print(f'HPA genes with at least one retained macro class: {hpa_macro.gene_symbol.nunique():,}')
print(f'HPA macro classes observed in the local file: {hpa_macro.hpa_macro_class.nunique():,} / {len(HPA_MACRO_CLASSES)}')
print('Largest retained HPA macro classes:')
print(hpa_group_sizes.sort_values('n_hpa_genes', ascending=False).head(8).to_string(index=False))

# -----------------------------
# Cascade partition exports
# -----------------------------
cascade_files = find_cascade_long_files()
if not cascade_files:
    raise FileNotFoundError(
        'No cascade_partition_proteins_long.csv file found. Run the procedural export cell first, or set CASCADE_LONG_FILES manually.'
    )

all_parts = []
for path in cascade_files:
    df = pd.read_csv(path, dtype=str, low_memory=False)
    df['source_path'] = str(path)
    df['source_id'] = infer_source_id(path)
    all_parts.append(df)

raw_partitions = pd.concat(all_parts, ignore_index=True, sort=False)
required = ['protein', 'stage']
missing_required = [c for c in required if c not in raw_partitions.columns]
if missing_required:
    raise KeyError(f'Missing required columns in cascade export: {missing_required}')
if 'dataset' not in raw_partitions.columns:
    raw_partitions['dataset'] = 'dataset'
if 'analysis' not in raw_partitions.columns:
    raw_partitions['analysis'] = 'analysis_unknown'
if 'threshold_label' not in raw_partitions.columns:
    raw_partitions['threshold_label'] = ''

partitions = raw_partitions.copy()
partitions['gene_symbol'] = partitions['protein'].apply(clean_gene_symbol)
partitions = partitions.dropna(subset=['gene_symbol']).copy()
partitions['stage_group'] = partitions['stage'].apply(canonical_stage_group)
partitions['metric_family'] = partitions.apply(lambda r: parse_metric_family(r.get('analysis', ''), r.get('threshold_label', '')), axis=1)
partitions['threshold_value'] = partitions.apply(parse_threshold_value, axis=1)
partitions['threshold_value_rounded'] = partitions['threshold_value'].round(2)

# Keep Pearson/Cosine at 0.50 and 0.70, but MSE at 0.20 and 0.30.
thr_mask = pd.Series(False, index=partitions.index)
thr_values = partitions['threshold_value'].astype(float)
for metric, thresholds in SELECTED_THRESHOLDS_BY_METRIC.items():
    metric_thr_mask = pd.Series(False, index=partitions.index)
    for thr in thresholds:
        metric_thr_mask = metric_thr_mask | np.isclose(thr_values, thr, atol=THRESHOLD_TOLERANCE, equal_nan=False)
    thr_mask = thr_mask | (partitions['metric_family'].eq(metric) & metric_thr_mask)
selected = partitions[thr_mask].copy()
if selected.empty:
    raise ValueError('No selected threshold rows were found: Pearson/Cosine 0.50/0.70 and MSE 0.20/0.30.')

# Avoid double-counting when both main_*_cascade and *_threshold_sweep exist for the same metric and threshold.
selected['analysis_rank'] = selected.apply(lambda r: preferred_analysis_rank(r['analysis'], r['metric_family']), axis=1)
rank_cols = ['source_id', 'dataset', 'metric_family', 'threshold_value_rounded']
best_rank = selected.groupby(rank_cols, dropna=False)['analysis_rank'].min().reset_index(name='best_analysis_rank')
selected = selected.merge(best_rank, on=rank_cols, how='left')
selected = selected[selected['analysis_rank'] == selected['best_analysis_rank']].copy()

selected['threshold_name'] = selected.apply(lambda r: threshold_name(r['metric_family'], r['threshold_value_rounded']), axis=1)
# Clean plot/export labels: no procedural/proxy-linear source names and no CCLE prefix.
# Internal source/dataset columns are kept in tables for traceability, but figure labels and
# per-case file names use only the biological threshold case.
selected['case_key'] = selected.apply(
    lambda r: safe_filename(f"{r['metric_family']}__{r['threshold_value_rounded']:.2f}"), axis=1
)
selected['case_label'] = selected.apply(
    lambda r: f"{r['metric_family']} {r['threshold_name']}", axis=1
)
selected = selected.drop_duplicates(['source_id', 'dataset', 'metric_family', 'threshold_value_rounded', 'analysis', 'stage', 'gene_symbol'])
selected.to_csv(TABLE_DIR / 'selected_standardized_cascade_partitions.csv', index=False)

print('\nCascade partition files loaded:')
for path in cascade_files:
    print('-', path)
print(f'Raw partition rows: {len(raw_partitions):,}')
print(f'Selected Pearson/Cosine 0.50/0.70 and MSE 0.20/0.30 rows after duplicate-analysis preference: {len(selected):,}')
print(f'Selected unique genes: {selected.gene_symbol.nunique():,}')
print('Selected cases:')
case_preview = selected[['analysis', 'metric_family', 'threshold_name', 'case_label', 'case_key']].drop_duplicates().sort_values(['metric_family', 'threshold_name', 'analysis'])
print(case_preview.to_string(index=False))
print('\nObserved canonical stage groups in selected rows:')
stage_counts = selected.groupby('stage_group')['gene_symbol'].nunique().sort_values(ascending=False)
print(stage_counts.to_string())

## 4. Compute shared case denominators and stage splits once

In [ ]:
CASE_COLS = ['source_id', 'dataset', 'metric_family', 'threshold_value_rounded', 'threshold_name', 'case_key', 'case_label']

# Denominator: genes from each HPA macro class present anywhere in each case's partition universe.
case_universe = selected[CASE_COLS + ['gene_symbol']].drop_duplicates()
case_macro = case_universe.merge(hpa_macro, on='gene_symbol', how='inner')
macro_universe_counts = (
    case_macro
    .groupby(CASE_COLS + ['hpa_macro_class'], observed=True)['gene_symbol']
    .nunique()
    .reset_index(name='n_macro_genes_in_partition_universe')
)
if macro_universe_counts.empty:
    raise ValueError('No selected partition genes matched the retained HPA macro protein classes.')

# Full partition split, shared by every target analysis.
stage_macro = selected[CASE_COLS + ['stage_group', 'gene_symbol']].drop_duplicates().merge(
    hpa_macro, on='gene_symbol', how='inner'
)
stage_counts = (
    stage_macro
    .groupby(CASE_COLS + ['stage_group', 'hpa_macro_class'], observed=True)['gene_symbol']
    .nunique()
    .reset_index(name='n_stage_group_genes_in_macro')
)
stage_coverage = stage_counts.merge(
    macro_universe_counts, on=CASE_COLS + ['hpa_macro_class'], how='left'
)
stage_coverage['pct_macro_captured_by_stage_group'] = (
    100 * stage_coverage['n_stage_group_genes_in_macro']
    / stage_coverage['n_macro_genes_in_partition_universe']
)
stage_coverage['hpa_macro_class'] = pd.Categorical(
    stage_coverage['hpa_macro_class'], categories=HPA_MACRO_CLASSES, ordered=True
)
stage_coverage = stage_coverage.sort_values(
    CASE_COLS + ['hpa_macro_class', 'stage_group']
).reset_index(drop=True)
stage_coverage.to_csv(
    SHARED_TABLE_DIR / 'hpa_macro_class_coverage_by_all_partition_stage_groups.csv', index=False
)

print(f'Shared case/macro denominator rows: {len(macro_universe_counts):,}')
print(f'Shared stage-coverage rows: {len(stage_coverage):,}')

## 5. Target-specific coverage function

In [ ]:
def configure_target(target_group_name, target_stage_groups, output_folder_name):
    """Switch the active target context used by saving and plotting functions."""
    global TARGET_GROUP_NAME, TARGET_STAGE_GROUPS, OUTPUT_DIR, TABLE_DIR, FIGURE_DIR

    TARGET_GROUP_NAME = target_group_name
    TARGET_STAGE_GROUPS = list(target_stage_groups)
    OUTPUT_DIR = OUTPUT_ROOT / output_folder_name
    TABLE_DIR = OUTPUT_DIR / 'tables'
    FIGURE_DIR = OUTPUT_DIR / 'figures'
    for folder in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR]:
        folder.mkdir(parents=True, exist_ok=True)

def compute_target_results(spec):
    """Compute and save target-specific HPA macro-class coverage tables."""
    configure_target(
        spec['target_group_name'],
        spec['target_stage_groups'],
        spec['output_folder_name'],
    )

    target_rows = selected[selected['stage_group'].isin(TARGET_STAGE_GROUPS)].copy()
    if target_rows.empty:
        raise ValueError(
            f'No selected rows matched target stage groups: {TARGET_STAGE_GROUPS}. '
            f'Observed groups: {sorted(selected.stage_group.unique())}'
        )

    target_macro = (
        target_rows[CASE_COLS + ['gene_symbol']]
        .drop_duplicates()
        .merge(hpa_macro, on='gene_symbol', how='inner')
    )
    target_counts = (
        target_macro
        .groupby(CASE_COLS + ['hpa_macro_class'], observed=True)['gene_symbol']
        .nunique()
        .reset_index(name='n_target_genes_in_macro')
    )

    coverage = macro_universe_counts.merge(
        target_counts, on=CASE_COLS + ['hpa_macro_class'], how='left'
    )
    coverage['n_target_genes_in_macro'] = coverage['n_target_genes_in_macro'].fillna(0).astype(int)
    coverage['pct_macro_captured_by_target'] = (
        100 * coverage['n_target_genes_in_macro']
        / coverage['n_macro_genes_in_partition_universe']
    )
    coverage['target_group'] = TARGET_GROUP_NAME
    coverage['hpa_macro_class'] = pd.Categorical(
        coverage['hpa_macro_class'], categories=HPA_MACRO_CLASSES, ordered=True
    )
    coverage = coverage.sort_values(CASE_COLS + ['hpa_macro_class']).reset_index(drop=True)
    coverage.to_csv(TABLE_DIR / 'target_hpa_macro_class_coverage.csv', index=False)

    target_gene_lists = (
        target_macro
        .groupby(CASE_COLS + ['hpa_macro_class'], observed=True)['gene_symbol']
        .apply(lambda s: ';'.join(sorted(set(s))))
        .reset_index(name='target_genes_in_macro')
    )
    target_gene_lists['target_group'] = TARGET_GROUP_NAME
    target_gene_lists.to_csv(TABLE_DIR / 'target_hpa_macro_class_gene_lists.csv', index=False)

    high_coverage = coverage[
        (coverage['n_target_genes_in_macro'] > 0)
        & (coverage['pct_macro_captured_by_target'] >= MIN_HIGH_COVERAGE_PCT)
    ].copy()
    high_coverage = high_coverage.sort_values(
        ['pct_macro_captured_by_target', 'n_target_genes_in_macro'],
        ascending=[False, False],
    )
    high_coverage.to_csv(
        TABLE_DIR / f'high_coverage_hpa_macro_classes_ge_{int(MIN_HIGH_COVERAGE_PCT)}pct.csv',
        index=False,
    )

    summary = coverage.groupby(CASE_COLS, observed=True).agg(
        macro_classes_with_denominator=(
            'n_macro_genes_in_partition_universe', lambda s: int((s > 0).sum())
        ),
        target_gene_macro_memberships=('n_target_genes_in_macro', 'sum'),
        median_pct_macro_captured=('pct_macro_captured_by_target', 'median'),
        mean_pct_macro_captured=('pct_macro_captured_by_target', 'mean'),
        max_pct_macro_captured=('pct_macro_captured_by_target', 'max'),
        macro_classes_ge_70pct=(
            'pct_macro_captured_by_target',
            lambda s: int((s >= MIN_HIGH_COVERAGE_PCT).sum()),
        ),
        macro_classes_ge_90pct=(
            'pct_macro_captured_by_target', lambda s: int((s >= 90).sum())
        ),
    ).reset_index()
    summary['target_group'] = TARGET_GROUP_NAME
    summary = summary.sort_values(
        ['source_id', 'dataset', 'metric_family', 'threshold_value_rounded']
    )
    summary.to_csv(TABLE_DIR / 'compact_summary_by_threshold_case.csv', index=False)

    print('\nCompact HPA macro-class coverage summary')
    show_cols = [
        'source_id', 'dataset', 'metric_family', 'threshold_name',
        'macro_classes_with_denominator', 'target_gene_macro_memberships',
        'median_pct_macro_captured', 'max_pct_macro_captured',
        'macro_classes_ge_70pct',
    ]
    print(summary[show_cols].round(2).to_string(index=False))
    print(
        f'\nHigh-coverage rows: {len(high_coverage):,} '
        f'(coverage >= {MIN_HIGH_COVERAGE_PCT:.0f}%)'
    )

    return {
        'spec': spec,
        'coverage': coverage,
        'summary': summary,
        'high_coverage': high_coverage,
        'target_gene_lists': target_gene_lists,
        'stage_coverage': stage_coverage,
        'output_dir': OUTPUT_DIR,
        'table_dir': TABLE_DIR,
        'figure_dir': FIGURE_DIR,
    }

## 6. Plotting functions defined once

In [ ]:
def ordered_cases(df):
    out = df[CASE_COLS].drop_duplicates().copy()
    out['metric_order'] = out['metric_family'].map(METRIC_ORDER).fillna(99)
    return out.sort_values(['source_id', 'dataset', 'metric_order', 'threshold_value_rounded']).drop(columns='metric_order')

def case_axis_labels(cases):
    # Clean labels: show only metric and threshold, never source names such as procedural/proxy_linear and never CCLE.
    labels = []
    for r in cases.itertuples(index=False):
        labels.append(f'{r.metric_family} {r.threshold_name}')
    return labels

def plot_overall_heatmap(cov, show=False):
    cases = ordered_cases(cov)
    if cases.empty:
        return
    case_order = cases['case_key'].tolist()
    labels = case_axis_labels(cases)
    mat = cov.pivot_table(
        index='hpa_macro_class', columns='case_key', values='pct_macro_captured_by_target', aggfunc='mean', observed=True
    ).reindex(index=HPA_MACRO_CLASSES, columns=case_order)
    fig, ax = plt.subplots(figsize=(max(9.5, 0.74 * len(case_order) + 3.2), 9.2))
    im = ax.imshow(mat.values.astype(float), aspect='auto', vmin=0, vmax=100, cmap='viridis')
    ax.set_xticks(range(len(case_order)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticks(range(len(HPA_MACRO_CLASSES)))
    ax.set_yticklabels([wrap_label(x, 31) for x in HPA_MACRO_CLASSES])
    ax.set_title(f'HPA macro-class coverage by {TARGET_GROUP_NAME}')
    ax.set_xlabel('Partition threshold case')
    ax.set_ylabel('HPA protein-class macro group')
    cbar = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.02)
    cbar.set_label('HPA macro group covered by target (%)')
    if len(case_order) <= 14:
        vals = mat.values
        for i in range(vals.shape[0]):
            for j in range(vals.shape[1]):
                v = vals[i, j]
                if pd.notna(v):
                    ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=7.5, color=('white' if v >= 55 else 'black'))
    fig.tight_layout()
    save_figure(fig, f'heatmap_hpa_macro_coverage_{TARGET_GROUP_NAME}')
    show_or_close(fig, show)

def plot_summary_case_bars(stats, show=False):
    if stats.empty:
        return
    stats = stats.copy()
    stats['metric_order'] = stats['metric_family'].map(METRIC_ORDER).fillna(99)
    stats = stats.sort_values(['source_id', 'dataset', 'metric_order', 'threshold_value_rounded'])
    labels = case_axis_labels(stats[CASE_COLS].drop_duplicates())
    vals = stats['macro_classes_ge_70pct'].astype(float).values
    med = stats['median_pct_macro_captured'].astype(float).values
    fig, ax1 = plt.subplots(figsize=(max(9.5, 0.70 * len(vals) + 3), 5.7))
    bar_color = STAGE_COLORS.get(TARGET_STAGE_GROUPS[0], '#4E79A7')
    bars = ax1.bar(range(len(vals)), vals, color=bar_color, alpha=0.88, label=f'Macro groups >= {MIN_HIGH_COVERAGE_PCT:.0f}%')
    ax1.set_ylabel(f'Number of HPA macro groups >= {MIN_HIGH_COVERAGE_PCT:.0f}% covered')
    ax1.set_xticks(range(len(vals)))
    ax1.set_xticklabels(labels, rotation=45, ha='right')
    ax1.grid(axis='y', alpha=0.22)
    for b, v in zip(bars, vals):
        ax1.text(b.get_x() + b.get_width()/2, v + 0.18, f'{int(v)}', ha='center', va='bottom', fontsize=9)
    ax2 = ax1.twinx()
    ax2.plot(range(len(med)), med, marker='o', linewidth=2.2, color='#222222', label='Median coverage')
    ax2.set_ylim(0, 100)
    ax2.set_ylabel('Median HPA macro coverage (%)')
    ax1.set_title(f'High-coverage HPA macro groups by threshold case: {TARGET_GROUP_NAME}')
    lines, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels1 + labels2, loc='upper left', frameon=False)
    fig.tight_layout()
    save_figure(fig, f'summary_high_coverage_counts_{TARGET_GROUP_NAME}')
    show_or_close(fig, show)

def plot_high_coverage_matrix(cov, show=False):
    high = cov.copy()
    high.loc[high['pct_macro_captured_by_target'] < MIN_HIGH_COVERAGE_PCT, 'pct_macro_captured_by_target'] = np.nan
    cases = ordered_cases(high)
    if cases.empty:
        return
    case_order = cases['case_key'].tolist()
    labels = case_axis_labels(cases)
    mat = high.pivot_table(index='hpa_macro_class', columns='case_key', values='pct_macro_captured_by_target', aggfunc='mean', observed=True).reindex(index=HPA_MACRO_CLASSES, columns=case_order)
    if mat.notna().sum().sum() == 0:
        return
    cmap = mpl.cm.get_cmap('magma').copy()
    cmap.set_bad('#F2F2F2')
    fig, ax = plt.subplots(figsize=(max(9.5, 0.74 * len(case_order) + 3.2), 9.2))
    im = ax.imshow(mat.values.astype(float), aspect='auto', vmin=MIN_HIGH_COVERAGE_PCT, vmax=100, cmap=cmap)
    ax.set_xticks(range(len(case_order)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticks(range(len(HPA_MACRO_CLASSES)))
    ax.set_yticklabels([wrap_label(x, 31) for x in HPA_MACRO_CLASSES])
    ax.set_title(f'HPA macro groups with >= {MIN_HIGH_COVERAGE_PCT:.0f}% coverage by {TARGET_GROUP_NAME}')
    ax.set_xlabel('Partition threshold case')
    ax.set_ylabel('HPA protein-class macro group')
    cbar = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.02)
    cbar.set_label('Coverage among high-covered groups (%)')
    if len(case_order) <= 14:
        vals = mat.values
        for i in range(vals.shape[0]):
            for j in range(vals.shape[1]):
                v = vals[i, j]
                if pd.notna(v):
                    ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=7.5, color='white')
    fig.tight_layout()
    save_figure(fig, f'high_coverage_matrix_ge_{int(MIN_HIGH_COVERAGE_PCT)}pct_{TARGET_GROUP_NAME}')
    show_or_close(fig, show)

def plot_high_coverage_bar_for_case(cov, case_key, show=False):
    sub = cov[(cov['case_key'] == case_key) & (cov['pct_macro_captured_by_target'] >= MIN_HIGH_COVERAGE_PCT) & (cov['n_target_genes_in_macro'] > 0)].copy()
    if sub.empty:
        return False
    sub = sub.sort_values(['pct_macro_captured_by_target', 'n_target_genes_in_macro'], ascending=[True, True]).tail(MAX_HIGH_COVERAGE_ROWS_PER_PLOT)
    desc = sub[CASE_COLS].drop_duplicates().iloc[0]
    y = np.arange(len(sub))
    vals = sub['pct_macro_captured_by_target'].astype(float).values
    n = sub['n_target_genes_in_macro'].astype(int).values
    d = sub['n_macro_genes_in_partition_universe'].astype(int).values
    colors = mpl.cm.viridis(np.linspace(0.32, 0.92, len(sub)))
    fig, ax = plt.subplots(figsize=(11.0, max(4.8, 0.43 * len(sub) + 2.0)))
    bars = ax.barh(y, vals, color=colors, alpha=0.95)
    ax.axvline(MIN_HIGH_COVERAGE_PCT, color='#333333', linestyle='--', linewidth=1.0, alpha=0.65)
    ax.set_xlim(max(0, MIN_HIGH_COVERAGE_PCT - 8), 103)
    ax.set_yticks(y)
    ax.set_yticklabels([wrap_label(x, 34) for x in sub['hpa_macro_class'].astype(str)])
    ax.set_xlabel('HPA macro group covered by target (%)')
    ax.set_ylabel('HPA protein-class macro group')
    ax.set_title(f'{TARGET_GROUP_NAME}: HPA macro groups >= {MIN_HIGH_COVERAGE_PCT:.0f}% covered\n{desc.case_label}')
    ax.grid(axis='x', alpha=0.22)
    for yi, v, nn, dd in zip(y, vals, n, d):
        ax.text(min(v + 0.8, 101.5), yi, f'{v:.0f}%  ({nn}/{dd})', va='center', ha='left', fontsize=9)
    fig.tight_layout()
    save_figure(fig, f'bar_high_coverage_ge_{int(MIN_HIGH_COVERAGE_PCT)}pct_{TARGET_GROUP_NAME}_{case_key}')
    show_or_close(fig, show)
    return True

def plot_high_coverage_dot_for_case(cov, case_key, show=False):
    sub = cov[(cov['case_key'] == case_key) & (cov['pct_macro_captured_by_target'] >= MIN_HIGH_COVERAGE_PCT) & (cov['n_target_genes_in_macro'] > 0)].copy()
    if sub.empty:
        return False
    sub = sub.sort_values(['pct_macro_captured_by_target', 'n_target_genes_in_macro'], ascending=[True, True]).tail(MAX_HIGH_COVERAGE_ROWS_PER_PLOT)
    desc = sub[CASE_COLS].drop_duplicates().iloc[0]
    y = np.arange(len(sub))
    vals = sub['pct_macro_captured_by_target'].astype(float).values
    denom = sub['n_macro_genes_in_partition_universe'].astype(float).values
    numer = sub['n_target_genes_in_macro'].astype(float).values
    sizes = 80 + 26 * np.sqrt(np.maximum(denom, 1))
    fig, ax = plt.subplots(figsize=(11.0, max(4.8, 0.43 * len(sub) + 2.0)))
    sc = ax.scatter(vals, y, s=sizes, c=numer, cmap='plasma', alpha=0.86, edgecolor='white', linewidth=0.8)
    ax.axvline(MIN_HIGH_COVERAGE_PCT, color='#333333', linestyle='--', linewidth=1.0, alpha=0.65)
    ax.set_xlim(max(0, MIN_HIGH_COVERAGE_PCT - 8), 103)
    ax.set_yticks(y)
    ax.set_yticklabels([wrap_label(x, 34) for x in sub['hpa_macro_class'].astype(str)])
    ax.set_xlabel('HPA macro group covered by target (%)')
    ax.set_ylabel('HPA protein-class macro group')
    ax.set_title(f'Dot/bubble view of high-covered HPA macro groups\n{TARGET_GROUP_NAME} - {desc.case_label}')
    ax.grid(axis='x', alpha=0.22)
    for yi, v, nn, dd in zip(y, vals, numer.astype(int), denom.astype(int)):
        ax.text(min(v + 0.8, 101.5), yi, f'{int(nn)}/{int(dd)}', va='center', ha='left', fontsize=9)
    cbar = fig.colorbar(sc, ax=ax, shrink=0.78, pad=0.02)
    cbar.set_label('Target genes in macro group')
    fig.tight_layout()
    save_figure(fig, f'dot_high_coverage_ge_{int(MIN_HIGH_COVERAGE_PCT)}pct_{TARGET_GROUP_NAME}_{case_key}')
    show_or_close(fig, show)
    return True

def plot_stage_split_for_top_high_groups(stage_cov, cov, case_key, show=False):
    top = cov[(cov['case_key'] == case_key) & (cov['pct_macro_captured_by_target'] >= MIN_HIGH_COVERAGE_PCT)].copy()
    if top.empty:
        return False
    top = top.sort_values(['pct_macro_captured_by_target', 'n_target_genes_in_macro'], ascending=[False, False]).head(12)
    macros = top['hpa_macro_class'].astype(str).tolist()
    sub = stage_cov[(stage_cov['case_key'] == case_key) & (stage_cov['hpa_macro_class'].astype(str).isin(macros))].copy()
    if sub.empty:
        return False
    desc = top[CASE_COLS].drop_duplicates().iloc[0]
    pivot = sub.pivot_table(index='hpa_macro_class', columns='stage_group', values='pct_macro_captured_by_stage_group', aggfunc='sum', observed=True).reindex(index=macros)
    stage_cols = [s for s in STAGE_COLORS.keys() if s in pivot.columns]
    if not stage_cols:
        return False
    pivot = pivot[stage_cols].fillna(0)
    fig, ax = plt.subplots(figsize=(12.5, max(5.2, 0.50 * len(pivot) + 2.2)))
    left = np.zeros(len(pivot))
    y = np.arange(len(pivot))
    for stage in stage_cols:
        vals = pivot[stage].astype(float).values
        ax.barh(y, vals, left=left, color=STAGE_COLORS.get(stage, '#999999'), label=stage, alpha=0.90)
        left += vals
    ax.set_xlim(0, 100)
    ax.set_yticks(y)
    ax.set_yticklabels([wrap_label(x, 34) for x in pivot.index.astype(str)])
    ax.invert_yaxis()
    ax.set_xlabel('Share of HPA macro group in each partition/model stage (%)')
    ax.set_ylabel('HPA protein-class macro group')
    ax.set_title(f'Partition split for high-covered HPA macro groups\n{desc.case_label}')
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.22), ncol=min(5, len(stage_cols)), frameon=False)
    ax.grid(axis='x', alpha=0.22)
    fig.tight_layout()
    save_figure(fig, f'stage_split_top_high_coverage_groups_{TARGET_GROUP_NAME}_{case_key}')
    show_or_close(fig, show)
    return True

## 7. Sequential target runner

In [ ]:
def generate_target_plots(results, display_all=True):
    """Generate, save, and sequentially display every available plot for one target."""
    coverage = results['coverage']
    summary = results['summary']

    # Overview plots.
    plot_overall_heatmap(coverage, show=True)
    plot_summary_case_bars(summary, show=True)
    plot_high_coverage_matrix(coverage, show=True)

    # Per-case plots. With display_all=True, every generated plot is displayed.
    created_bar = 0
    created_dot = 0
    created_split = 0
    cases = ordered_cases(coverage)
    for case_key in cases['case_key'].tolist():
        if plot_high_coverage_bar_for_case(coverage, case_key, show=display_all):
            created_bar += 1
        if plot_high_coverage_dot_for_case(coverage, case_key, show=display_all):
            created_dot += 1
        if plot_stage_split_for_top_high_groups(
            results['stage_coverage'], coverage, case_key, show=display_all
        ):
            created_split += 1

    print('\nFigures saved to:', results['figure_dir'])
    print(f'High-coverage bar plots created: {created_bar}')
    print(f'High-coverage dot/bubble plots created: {created_dot}')
    print(f'High-coverage partition-split plots created: {created_split}')
    print('Tables saved to:', results['table_dir'])

def run_target(spec, display_all=True):
    """Run one configured target from computation through plotting."""
    display(Markdown(f"## {spec['target_group_name']}"))
    print('Stage groups used as numerator:', spec['target_stage_groups'])
    results = compute_target_results(spec)
    generate_target_plots(results, display_all=display_all)
    return results

## 8. Run all heatmap analyses sequentially

This cell computes each target and displays its overview and per-case plots before moving to the next target.

In [ ]:
all_results = OrderedDict()

for spec in TARGET_SPECS:
    all_results[spec['key']] = run_target(
        spec,
        display_all=DISPLAY_ALL_PLOTS,
    )

print('\nAll target analyses completed.')
print('Merged output root:', OUTPUT_ROOT)
print('Completed targets:', ', '.join(all_results.keys()))

## Outputs

Shared tables are saved under `4.2_HPA_heatmaps_outputs/shared/tables`. Each target retains its original output-folder name under `4.2_HPA_heatmaps_outputs/`, with separate `tables` and `figures` subfolders.

---
# Part C — HPA enrichment and Pearson–MSE overlap, Fig. 4 / S5 (was `5_1_hpa_barplots.ipynb`)

Writes `4.0_hpa_v4_outputs/`.

## 4.0 — HPA model comparison v4

This version uses only **Pearson ≥ 0.50**, **Pearson ≥ 0.70**, **MSE ≤ 0.20**, and **MSE ≤ 0.30**.

The central correction is the denominator. Every HPA percentage is now divided by the **full HPA class size supplied for that class**, rather than by the smaller number of class members that happened to enter a model-specific evaluation universe.

All figures are:

- percentage-based;
- vertical bar plots;
- arranged in compact grids;
- explicit about proteins that were evaluated but not predicted well and proteins that were not evaluated at all.

No `0/0` HPA rows are produced because all 20 reference HPA classes have a strictly positive full-class denominator.

## Readable figures and PDF export

Run this notebook from the first cell to the final export cell using the original data files. Fonts, subplot spacing, title wrapping, and statistics boxes have been enlarged. All figure and subplot titles are removed; axis labels, legends and data annotations are retained. Previous outputs were cleared so they cannot be mistaken for the updated plots.

The final cell writes **every generated figure** as a vector PDF into **`figures_pdf` beside this notebook**, including figures closed by earlier cells. Filenames contain a notebook prefix and figure number to prevent collisions. Standard Jupyter starts in the notebook folder; if your kernel was launched elsewhere and cannot report its notebook path, set `NOTEBOOK_DIRECTORY_OVERRIDE` in the setup cell.


In [ ]:
# Larger plot text and deferred PDF export.
# Keep this cell before the analysis/plotting cells.
from io import BytesIO
import os
from matplotlib._pylab_helpers import Gcf
from matplotlib.ticker import MaxNLocator

NOTEBOOK_SLUG = 'hpa_barplots'
REMOVE_PLOT_TITLES = True
NOTEBOOK_DIRECTORY_OVERRIDE = None  # Set a Path here only if the kernel runs elsewhere.

def _notebook_directory():
    if NOTEBOOK_DIRECTORY_OVERRIDE is not None:
        return Path(NOTEBOOK_DIRECTORY_OVERRIDE).expanduser().resolve()
    # VS Code and Jupyter may provide the notebook's actual path.
    candidates = [globals().get('__vsc_ipynb_file__'), os.environ.get('JPY_SESSION_NAME')]
    for candidate in candidates:
        if candidate:
            path = Path(candidate).expanduser()
            if path.suffix == '.ipynb' and path.is_file():
                return path.resolve().parent
    try:
        import ipynbname
        return ipynbname.path().resolve().parent
    except (ImportError, FileNotFoundError, RuntimeError, OSError):
        # Standard Jupyter kernels start in the notebook's own directory.
        return Path.cwd().resolve()

NOTEBOOK_DIRECTORY = _notebook_directory()
FIGURES_PDF_DIR = FIG_DIR
_pdf_figure_cache = OrderedDict()
_pdf_next_number = 1

mpl.rcParams.update({
    'font.size': 13,
    'axes.titlesize': 17,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'legend.title_fontsize': 12,
    'figure.titlesize': 22,
    'figure.figsize': (8, 6.5),
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.25,
})

def set_readable_theme(*args, **kwargs):
    """Keep Seaborn's own implicit label/tick fonts larger as well."""
    import seaborn as sns
    kwargs.setdefault('font_scale', 1.20)
    return sns.set_theme(*args, **kwargs)

def readable_figsize(original_size):
    """Increase physical space together with fonts, including dynamic grids."""
    width, height = original_size
    if NOTEBOOK_SLUG == 'hpa_barplots':
        return float(width) * 1.18, float(height) * 1.35
    return float(width) * 1.10, float(height) * 1.25

def _wrap_to_width(artist, renderer, max_width):
    """Wrap whole words using measured text extents, without shortening labels."""
    original = getattr(artist, '_readable_original_text', artist.get_text())
    artist._readable_original_text = original
    if not original or '$' in original:
        return
    artist.set_text(original)
    if artist.get_window_extent(renderer).width <= max_width:
        return
    for width in range(max(12, max(map(len, original.splitlines())) - 1), 11, -1):
        candidate = '\n'.join(
            textwrap.fill(line, width=width, break_long_words=False, break_on_hyphens=False)
            for line in original.split('\n')
        )
        artist.set_text(candidate)
        if artist.get_window_extent(renderer).width <= max_width:
            break

def prepare_readable_figure(fig):
    """Apply minimum font sizes and measure margins before display or export."""
    if not fig.axes or getattr(fig, '_readable_preparing', False):
        return fig
    fig._readable_preparing = True
    try:
        # Remove every main/subplot title before measuring layout or exporting.
        if REMOVE_PLOT_TITLES:
            for axis in fig.axes:
                for location in ('left', 'center', 'right'):
                    title_artist = axis.set_title('', loc=location)
                    title_artist._readable_original_text = ''
            if fig._suptitle is not None:
                fig._suptitle.remove()
                fig._suptitle = None
        for ax in fig.axes:
            if not ax.get_visible():
                continue
            ax.title.set_fontsize(max(16, ax.title.get_fontsize()))
            ax.xaxis.label.set_fontsize(max(14, ax.xaxis.label.get_fontsize()))
            ax.yaxis.label.set_fontsize(max(14, ax.yaxis.label.get_fontsize()))
            ax.xaxis.labelpad = max(8, ax.xaxis.labelpad)
            ax.yaxis.labelpad = max(9, ax.yaxis.labelpad)
            ax.tick_params(axis='both', which='major', pad=5)
            for label in ax.get_xticklabels() + ax.get_yticklabels():
                label.set_fontsize(max(11.5, label.get_fontsize()))
            for offset in (ax.xaxis.get_offset_text(), ax.yaxis.get_offset_text()):
                offset.set_fontsize(max(11.5, offset.get_fontsize()))
            for artist in ax.texts:
                artist.set_fontsize(max(11.5, artist.get_fontsize()))
                if re.fullmatch(r'\(?[A-Za-z]\)', artist.get_text().strip()):
                    # Panel letters sit outside the data and statistics boxes.
                    artist.set_position((-0.06, 1.03))
                    artist.set_ha('right')
                    artist.set_va('bottom')
                    artist.set_clip_on(False)
                if not ax.axison and artist.get_text().count('\n') >= 8:
                    # Keep the full statistics box within its own panel.
                    artist.set_position((0.03, 0.96))
                    artist.set_ha('left')
                    artist.set_va('top')
                    artist.set_linespacing(1.10)
            legend = ax.get_legend()
            if legend is not None:
                for label in legend.get_texts():
                    label.set_fontsize(max(11.5, label.get_fontsize()))
                legend.get_title().set_fontsize(max(12, legend.get_title().get_fontsize()))
        if fig._suptitle is not None:
            fig._suptitle.set_fontsize(max(20, fig._suptitle.get_fontsize()))
            fig._suptitle.set_y(0.985)
            fig._suptitle.set_va('top')
        for legend in fig.legends:
            for label in legend.get_texts():
                label.set_fontsize(max(11.5, label.get_fontsize()))
        # Repeat after layout: wrapping changes title height and required margins.
        for _ in range(3):
            fig.canvas.draw()
            renderer = fig.canvas.get_renderer()
            if fig._suptitle is not None:
                _wrap_to_width(fig._suptitle, renderer, fig.bbox.width * 0.94)
            for ax in fig.axes:
                if ax.get_visible():
                    _wrap_to_width(ax.title, renderer, ax.bbox.width * 0.96)
            bottom, top = 0.02, 0.98
            if fig._suptitle is not None:
                title_height = fig._suptitle.get_window_extent(renderer).height / fig.bbox.height
                top = min(top, 0.97 - title_height - 0.018)
            for legend in fig.legends:
                bounds = legend.get_window_extent(renderer)
                fraction = bounds.height / fig.bbox.height
                if bounds.y0 + bounds.height / 2 < fig.bbox.y0 + fig.bbox.height / 2:
                    bottom = max(bottom, fraction + 0.045)
                else:
                    top = min(top, 0.97 - fraction - 0.025)
            # Let tight_layout control GridSpec spacing instead of skipping grids
            # whose hspace/wspace was explicitly fixed by older plotting code.
            engine = fig.get_layout_engine()
            if engine is None or engine.__class__.__name__ != 'ConstrainedLayoutEngine':
                grids = set()
                for ax in fig.axes:
                    if hasattr(ax, 'get_subplotspec'):
                        spec = ax.get_subplotspec()
                        if spec is not None:
                            grids.add(spec.get_gridspec())
                for grid in grids:
                    grid.update(hspace=None, wspace=None)
            if engine is not None and engine.__class__.__name__ == 'ConstrainedLayoutEngine':
                engine.set(w_pad=0.12, h_pad=0.16, wspace=0.10, hspace=0.15)
            else:
                with warnings.catch_warnings():
                    warnings.filterwarnings('ignore', message='.*not compatible with tight_layout.*')
                    fig.tight_layout(pad=1.25, h_pad=2.1, w_pad=2.2, rect=(0.01, bottom, 0.99, top))
            fig.canvas.draw()
            renderer = fig.canvas.get_renderer()
            # A long statistics block must not extend below its allocated panel.
            required_growth = 1.0
            for ax in fig.axes:
                for artist in ax.texts:
                    # Reserve room inside axes for bar-end numbers and log-scale
                    # percentage annotations; do not move their data coordinates.
                    if artist.get_transform() is ax.transData and ax.axison:
                        bounds = artist.get_window_extent(renderer)
                        x0, x1 = ax.get_xlim()
                        y0, y1 = ax.get_ylim()
                        if x1 > x0 and bounds.x1 > ax.bbox.x1 + 1:
                            right = ax.transData.inverted().transform((bounds.x1 + 8, ax.bbox.y0))[0]
                            if np.isfinite(right):
                                ax.set_xlim(x0, right)
                        if y1 > y0 and bounds.y1 > ax.bbox.y1 + 1:
                            upper = ax.transData.inverted().transform((ax.bbox.x0, bounds.y1 + 8))[1]
                            if np.isfinite(upper):
                                ax.set_ylim(y0, upper)
                    if not ax.axison and artist.get_text().count('\n') >= 8:
                        bounds = artist.get_window_extent(renderer)
                        required_growth = max(required_growth, bounds.height / max(ax.bbox.height * 0.94, 1))
                # Numeric ticks can use fewer positions while preserving their scale.
                labels = [t for t in ax.get_xticklabels() if t.get_visible() and t.get_text()]
                boxes = [t.get_window_extent(renderer) for t in labels]
                overlaps = any(a.overlaps(b) for a, b in zip(boxes, boxes[1:]))
                if overlaps and not ax.xaxis.get_units() and len(labels) > 5:
                    if all(t.get_rotation() == 0 for t in labels):
                        locator = ax.xaxis.get_major_locator()
                        if locator.__class__.__name__ in {'AutoLocator', 'MaxNLocator'}:
                            ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
            if required_growth > 1.01:
                width, height = fig.get_size_inches()
                fig.set_size_inches(width, height * min(required_growth * 1.05, 1.6), forward=True)
        # Avoid collisions between annotation text and an axes legend.
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        for ax in fig.axes:
            legend = ax.get_legend()
            if legend is None:
                continue
            legend_box = legend.get_window_extent(renderer)
            if any(t.get_visible() and t.get_text() and legend_box.overlaps(t.get_window_extent(renderer))
                   for t in ax.texts):
                if hasattr(legend, 'set_loc'):
                    for location in ('upper left', 'lower right', 'lower left'):
                        legend.set_loc(location)
                        fig.canvas.draw()
                        legend_box = legend.get_window_extent(renderer)
                        if not any(t.get_visible() and t.get_text() and legend_box.overlaps(t.get_window_extent(renderer))
                                   for t in ax.texts):
                            break
        fig.canvas.draw()
        return fig
    finally:
        fig._readable_preparing = False

def _cache_figure_pdf(fig, source_name=None):
    """Snapshot vector PDFs in memory so closed figures survive until the last cell."""
    global _pdf_next_number
    if not fig.axes or not any(ax.get_visible() for ax in fig.axes):
        return
    key = getattr(fig, '_readable_pdf_key', None)
    if key is None or not str(key).startswith(NOTEBOOK_SLUG + '__'):
        key = f'{NOTEBOOK_SLUG}__{_pdf_next_number:03d}'
        _pdf_next_number += 1
        fig._readable_pdf_key = key
    previous = _pdf_figure_cache.get(key, {})
    name = previous.get('name')
    if source_name is not None and isinstance(source_name, (str, os.PathLike)):
        name = Path(source_name).stem
    if not name:
        title = fig._suptitle.get_text() if fig._suptitle is not None else ''
        if not title:
            title = next((ax.get_title() for ax in fig.axes if ax.get_title()), 'figure')
        name = title
    name = re.sub(r'[^A-Za-z0-9._-]+', '_', name).strip('._')[:120] or 'figure'
    buffer = BytesIO()
    fig.savefig(buffer, format='pdf', bbox_inches='tight', pad_inches=0.25,
                metadata={'Creator': 'Matplotlib', 'Title': name})
    _pdf_figure_cache[key] = {'name': name, 'pdf': buffer.getvalue()}

def save_readable_figure(fig, filename, *args, **kwargs):
    prepare_readable_figure(fig)
    kwargs.setdefault('bbox_inches', 'tight')
    kwargs.setdefault('pad_inches', 0.25)
    result = fig.savefig(filename, *args, **kwargs)
    _cache_figure_pdf(fig, source_name=filename)
    return result

def save_readable_current_figure(filename, *args, **kwargs):
    return save_readable_figure(plt.gcf(), filename, *args, **kwargs)

def show_readable_figures(*args, **kwargs):
    for manager in list(Gcf.get_all_fig_managers()):
        fig = manager.canvas.figure
        prepare_readable_figure(fig)
        _cache_figure_pdf(fig)
    return plt.show(*args, **kwargs)

def close_readable_figures(fig=None):
    if fig == 'all':
        figures = [m.canvas.figure for m in Gcf.get_all_fig_managers()]
    elif fig is None:
        manager = Gcf.get_active()
        figures = [manager.canvas.figure] if manager is not None else []
    elif hasattr(fig, 'savefig'):
        figures = [fig]
    else:
        manager = Gcf.get_fig_manager(fig) if isinstance(fig, int) else None
        figures = [manager.canvas.figure] if manager is not None else []
    for figure in figures:
        prepare_readable_figure(figure)
        _cache_figure_pdf(figure)
    return plt.close(fig)

def export_notebook_figures():
    """Write every figure created during this run, including already-closed figures."""
    for manager in list(Gcf.get_all_fig_managers()):
        fig = manager.canvas.figure
        prepare_readable_figure(fig)
        _cache_figure_pdf(fig)
    if not _pdf_figure_cache:
        raise RuntimeError('No figures captured. Run the setup and plotting cells first.')
    FIGURES_PDF_DIR.mkdir(parents=True, exist_ok=True)
    saved = []
    for key, record in _pdf_figure_cache.items():
        path = FIGURES_PDF_DIR / f'{key}__{record["name"]}.pdf'
        path.write_bytes(record['pdf'])
        saved.append(path)
    print(f'Saved {len(saved)} PDF figures to: {FIGURES_PDF_DIR}')
    for path in saved:
        print('  ' + path.name)
    return saved

print('Readable plotting enabled. Final PDF destination:', FIGURES_PDF_DIR)

In [ ]:
warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)
pd.set_option('display.max_colwidth', 140)

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
BASE_DIR = Path.cwd()
DOWNLOAD_DIR = BASE_DIR / 'pan_atlas_downloads'
OUTPUT_DIR = BASE_DIR / '4.0_hpa_v4_outputs'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
for folder in (OUTPUT_DIR, TABLE_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Manual overrides. Leave empty to use automatic discovery.
CASCADE_LONG_FILES = [str(p) for p in [BASE_DIR / 'procedural_unified_tables' / 'train_linear' / 'protein_names_by_cascade_partition' / 'cascade_partition_proteins_long.csv'] if p.exists()]  # paper primary proxy mode (train-fitted linear); empty -> auto-search
INDEPENDENT_SCORE_FILES = []
AUTO_SEARCH_CASCADE_FILES = True
AUTO_SEARCH_INDEPENDENT_SCORE_FILES = True
MAX_AUTO_SCORE_FILES = 100

DATASET_FILTER = None
SOURCE_ID_FILTER = None
INDEPENDENT_UNIVERSE_MODE = 'union'  # 'union' or 'intersection'
DUPLICATE_SCORE_AGGREGATION = 'median'
REQUIRE_INDEPENDENT_SCORES = False

MODELS = ['Proxy model', 'ElasticNet', 'XGB']
SUCCESS_STAGES = ['Proxy model', 'ElasticNet', 'XGB']
CASCADE_STAGES = ['Proxy model', 'ElasticNet', 'XGB', 'Still not good', 'Other']
SELECTED_THRESHOLDS_BY_METRIC = OrderedDict([
    ('Pearson', [0.50, 0.70]),
    ('MSE', [0.20, 0.30]),
])
PEARSON_THRESHOLDS = [0.50, 0.70]
MSE_THRESHOLDS = [0.20, 0.30]
THRESHOLD_TOLERANCE = 1e-7
METRIC_ORDER = {'Pearson': 0, 'MSE': 1}
MODEL_ORDER_FOR_MARGINAL_GAIN = MODELS.copy()

# Authoritative full HPA class sizes supplied by the user.
HPA_REFERENCE_COUNTS = OrderedDict([
    ('Enzymes', 3777),
    ('Metabolic proteins', 2882),
    ('CD markers', 384),
    ('Blood group antigen proteins', 31),
    ('Nuclear receptors', 49),
    ('Transporters', 2137),
    ('Ribosomal proteins', 180),
    ('G-protein coupled receptors', 743),
    ('Voltage-gated ion channels', 132),
    ('Predicted membrane proteins', 5571),
    ('Predicted secreted proteins', 1902),
    ('Predicted intracellular proteins', 15906),
    ('Plasma proteins', 3750),
    ('Transcription factors', 1485),
    ('RNA polymerase related proteins', 34),
    ('RAS pathway related proteins', 235),
    ('Citric acid cycle related proteins', 30),
    ('Essential proteins', 2022),
    ('Immunoglobulin genes', 214),
    ('T-cell receptor genes', 196),
])
HPA_MACRO_CLASSES = list(HPA_REFERENCE_COUNTS)
HPA_SHORT_LABELS = [
    'Enzymes', 'Metabolic', 'CD markers', 'Blood group', 'Nuclear rec.',
    'Transporters', 'Ribosomal', 'GPCR', 'Voltage-gated', 'Pred. membrane',
    'Pred. secreted', 'Pred. intracellular', 'Plasma', 'TFs',
    'RNA polymerase', 'RAS pathway', 'TCA cycle', 'Essential',
    'Immunoglobulin', 'T-cell receptor'
]

STAGE_COLORS = OrderedDict([
    ('Proxy model', '#4E79A7'),
    ('ElasticNet', '#F28E2B'),
    ('XGB', '#E15759'),
    ('Still not good', '#B07AA1'),
    ('Other', '#9D9D9D'),
])
MODEL_COLORS = {k: STAGE_COLORS[k] for k in MODELS}
FULL_DECOMPOSITION_COLORS = OrderedDict([
    ('Proxy-covered', '#4E79A7'),
    ('ElasticNet additional', '#F28E2B'),
    ('XGB additional', '#E15759'),
    ('Evaluated but not predicted well', '#B07AA1'),
    ('Not evaluated / absent from universe', '#E6E6E6'),
])
MODEL_OVERLAP_COLORS = OrderedDict([
    ('Proxy only', '#4E79A7'),
    ('ElasticNet only', '#F28E2B'),
    ('XGB only', '#E15759'),
    ('Shared by 2 models', '#76B7B2'),
    ('Shared by 3 models', '#59A14F'),
    ('Evaluated, no model covers', '#B07AA1'),
    ('Not evaluated / absent from universe', '#E6E6E6'),
])
PEARSON_MSE_COLORS = OrderedDict([
    ('Pearson only', '#6F4E9C'),
    ('Shared', '#59A14F'),
    ('MSE only', '#E15759'),
    ('Jointly evaluated, neither passes', '#B07AA1'),
    ('Not jointly evaluated', '#E6E6E6'),
])

COMPACT_GRID_MAX_COLS = 2
VERTICAL_PANEL_SIZE = (6.2, 4.2)
VERTICAL_LABEL_ROTATION = 63
BAR_WIDTH = 0.78
SAVE_PNG = True
SAVE_SVG = False
SAVE_PDF = False
DPI = 180
DISPLAY_MAIN_FIGURES = True

mpl.rcParams.update({
    'figure.dpi': 125,
    'savefig.dpi': DPI,
    'font.size': 13.0,
    'axes.titlesize': 15.2,
    'axes.titleweight': 'bold',
    'axes.labelsize': 13.3,
    'xtick.labelsize': 10.4,
    'ytick.labelsize': 11.3,
    'legend.fontsize': 11.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

print('Base directory:', BASE_DIR)
print('Output directory:', OUTPUT_DIR)
print('Thresholds:', dict(SELECTED_THRESHOLDS_BY_METRIC))

## Percentage convention used throughout

For each HPA class \(c\), the denominator is the full reference size \(N_c\) shown in the table supplied with the analysis.

For example, if one GPCR is covered by Proxy, the plotted percentage is:

\[
100	imes
rac{1}{743}=0.135\%,
\]

not \(1/1=100\%\). The remaining GPCRs are represented explicitly as evaluated-but-not-covered or not-evaluated.

In [ ]:
BAR_HEIGHT = globals().get("BAR_HEIGHT", 0.72)            # was undefined in 5_1_hpa_barplots
SHOW_SEGMENT_LABELS = globals().get("SHOW_SEGMENT_LABELS", True)
def wrap_label(text, width=29):
    return '\n'.join(textwrap.wrap(str(text), width=width, break_long_words=False))

def clean_gene_symbol(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if not value or value.casefold() in {'nan', 'none', 'na', 'null'}:
        return np.nan
    return re.split(r'[;|,/\s]+', value)[0].upper()

def split_hpa_terms(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.casefold() in {'nan', 'none', 'na', 'null'}:
        return []
    out, seen = [], set()
    for part in re.split(r'[,;|]+', text):
        term = re.sub(r'\s+', ' ', part.strip())
        key = normalize_text(term)
        if term and key not in seen:
            out.append(term)
            seen.add(key)
    return out

def pick_column(df, candidates, required=True):
    norm_to_col = {normalize_text(c): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        key = normalize_text(cand)
        if key in norm_to_col:
            return norm_to_col[key]
    for cand in candidates:
        words = normalize_text(cand).split()
        for col in df.columns:
            norm_col = normalize_text(col)
            if words and all(w in norm_col for w in words):
                return col
    if required:
        raise KeyError(f'Could not find any of {candidates}. Columns include: {list(df.columns)[:50]}')
    return None

def read_table_any(path):
    path = Path(path)
    suffixes = [s.lower() for s in path.suffixes]
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    if suffixes and suffixes[-1] == '.zip':
        with zipfile.ZipFile(path) as zf:
            names = [n for n in zf.namelist() if not n.endswith('/') and n.lower().endswith(('.tsv', '.txt', '.csv'))]
            if not names:
                raise ValueError(f'No delimited table found inside {path}')
            inner = names[0]
            sep = ',' if inner.lower().endswith('.csv') else '\t'
            with zf.open(inner) as fh:
                return pd.read_csv(fh, sep=sep, dtype=str, low_memory=False)
    if suffixes and suffixes[-1] == '.gz':
        sep = ',' if len(suffixes) >= 2 and suffixes[-2] == '.csv' else '\t'
        return pd.read_csv(path, sep=sep, dtype=str, compression='gzip', low_memory=False)
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path, dtype=str, low_memory=False)
    return pd.read_csv(path, sep='\t', dtype=str, low_memory=False)

def save_figure(fig, stem):
    stem = safe_filename(stem)
    saved = []
    if SAVE_PNG:
        p = FIGURE_DIR / f'{stem}.png'
        save_readable_figure(fig, p, bbox_inches='tight', dpi=DPI)
        saved.append(p)
    if SAVE_SVG:
        p = FIGURE_DIR / f'{stem}.svg'
        save_readable_figure(fig, p, bbox_inches='tight')
        saved.append(p)
    if SAVE_PDF:
        p = FIGURE_DIR / f'{stem}.pdf'
        save_readable_figure(fig, p, bbox_inches='tight')
        saved.append(p)
    return saved

def apply_grid_axis_label_policy(fig):
    """Avoid repeated axis titles in multi-column subplot grids.

    For two-column grids, keep the y-axis title only on the left panel of
    each row and the x-axis title only on the right panel of each row.
    Tick labels are left unchanged.
    """
    for ax in fig.axes:
        if not ax.get_visible():
            continue
        try:
            spec = ax.get_subplotspec()
            grid = spec.get_gridspec()
        except Exception:
            continue
        if grid.ncols <= 1:
            continue
        col = spec.colspan.start
        if col != 0:
            ax.set_ylabel('')
        if col != grid.ncols - 1:
            ax.set_xlabel('')

def finish_figure(fig, stem, show=False):
    apply_grid_axis_label_policy(fig)
    save_figure(fig, stem)
    if show:
        show_readable_figures()
    else:
        close_readable_figures(fig)

def canonical_stage_group(raw_stage):
    text = normalize_text(raw_stage)
    if not text or text in {'nan', 'none'}:
        return 'Other'
    if 'still' in text and 'good' in text:
        return 'Still not good'
    if 'no valid mse' in text:
        return 'Still not good'
    if 'proxy' in text:
        return 'Proxy model'
    if 'enet' in text or 'elastic' in text:
        return 'ElasticNet'
    if 'xgb' in text or 'xgboost' in text:
        return 'XGB'
    return 'Other'

def canonical_model(value):
    text = normalize_text(value)
    if not text or text in {'nan', 'none'}:
        return None
    if 'proxy' in text:
        return 'Proxy model'
    if 'enet' in text or 'elastic' in text:
        return 'ElasticNet'
    if 'xgb' in text or 'xgboost' in text or 'boosted tree' in text:
        return 'XGB'
    return None

def parse_metric_family(*values):
    text = ' '.join(str(v) for v in values).casefold()
    if 'pearson' in text or re.search(r'(^|[^a-z])pcc([^a-z]|$)', text):
        return 'Pearson'
    if 'cosine' in text or 'cos sim' in text or 'cos_sim' in text:
        return 'Cosine'
    if 'mean squared error' in text or re.search(r'(^|[^a-z])mse([^a-z]|$)', text):
        return 'MSE'
    return None

def parse_threshold_value(row):
    for col in ['threshold', 'prediction_threshold', 'corr_threshold', 'mse_threshold', 'metric_threshold']:
        if col in row.index and pd.notna(row[col]):
            try:
                value = float(str(row[col]).strip())
                if math.isfinite(value):
                    return value
            except Exception:
                pass
    label = str(row.get('threshold_label', ''))
    nums = re.findall(r'[-+]?\d*\.\d+|[-+]?\d+', label)
    if not nums:
        return np.nan
    value = float(nums[-1])
    metric = parse_metric_family(row.get('analysis', ''), label)
    if metric in {'Pearson', 'Cosine'} and 1 < value <= 100:
        value /= 100.0
    return value

def threshold_label(metric, value):
    return f'{metric} {threshold_operator(metric)} {float(value):.2f}'

def ordered_cases(df):
    cols = ['case_id', 'context_id', 'source_id', 'dataset', 'metric_family', 'threshold_value', 'case_label']
    out = df[[c for c in cols if c in df.columns]].drop_duplicates().copy()
    out['metric_order'] = out['metric_family'].map(METRIC_ORDER).fillna(99)
    return out.sort_values(['context_id', 'metric_order', 'threshold_value']).drop(columns='metric_order')

def compact_grid(n_panels, panel_size=(3.75, 5.35), max_cols=None, **subplot_kwargs):
    """Create a compact grid and hide unused axes."""
    n_panels = int(n_panels)
    if n_panels < 1:
        raise ValueError('n_panels must be at least 1')
    max_cols = COMPACT_GRID_MAX_COLS if max_cols is None else max_cols
    ncols = min(max_cols, n_panels)
    nrows = int(math.ceil(n_panels / ncols))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=readable_figsize((panel_size[0] * ncols, panel_size[1] * nrows)),
        squeeze=False,
        **subplot_kwargs,
    )
    flat = axes.ravel()
    for ax in flat[n_panels:]:
        ax.set_visible(False)
    return fig, flat[:n_panels], nrows, ncols

def set_hpa_axis_labels(ax, orientation='horizontal', show_labels=True):
    positions = np.arange(len(HPA_MACRO_CLASSES))
    if orientation == 'horizontal':
        ax.set_yticks(positions)
        ax.set_yticklabels([wrap_label(x, 24) for x in HPA_MACRO_CLASSES] if show_labels else [])
        if show_labels:
            ax.set_ylabel('HPA macro group')
        ax.invert_yaxis()
    else:
        ax.set_xticks(positions)
        ax.set_xticklabels(
            HPA_SHORT_LABELS if show_labels else [],
            rotation=VERTICAL_LABEL_ROTATION,
            ha='right',
        )
        if show_labels:
            ax.set_xlabel('HPA macro group')

def add_zero_denominator_background(ax, zero_mask, orientation='horizontal', extent=100):
    """Mark undefined 0/0 HPA groups explicitly instead of leaving white space."""
    zero_mask = np.asarray(zero_mask, dtype=bool)
    positions = np.arange(len(zero_mask))
    if orientation == 'horizontal':
        if zero_mask.any():
            ax.barh(
                positions[zero_mask], np.repeat(extent, zero_mask.sum()),
                color='#F2F2F2', edgecolor='#BDBDBD', hatch='///',
                height=BAR_HEIGHT, linewidth=0.5, zorder=0,
            )
        for pos in positions[zero_mask]:
            ax.text(extent / 2, pos, '0/0', ha='center', va='center', fontsize=9.9, color='#555555', zorder=4)
    else:
        for pos in positions[zero_mask]:
            ax.axvspan(pos - 0.42, pos + 0.42, color='#F2F2F2', hatch='///', edgecolor='#BDBDBD', linewidth=0.4, zorder=0)
            ax.text(pos, extent * 0.5, '0/0', ha='center', va='center', rotation=90, fontsize=9.4, color='#555555', zorder=4)

def label_stacked_segment(ax, orientation, base, value, position, text, threshold):
    if not SHOW_SEGMENT_LABELS or value < threshold or value <= 0:
        return
    if orientation == 'horizontal':
        ax.text(base + value / 2, position, text, ha='center', va='center', fontsize=9.0, color='white', fontweight='bold')
    else:
        ax.text(position, base + value / 2, text, ha='center', va='center', fontsize=8.5, color='white', fontweight='bold', rotation=90)

def ordered_independent_cases():
    out = independent_cases.copy()
    out['metric_order'] = out['metric_family'].map(METRIC_ORDER).fillna(99)
    return out.sort_values(['metric_order', 'threshold_value']).drop(columns='metric_order')

HPA_SHORT_MAP = dict(zip(HPA_MACRO_CLASSES, HPA_SHORT_LABELS))

def short_hpa_labels(classes):
    return [HPA_SHORT_MAP.get(str(c), str(c)) for c in classes]

def set_hpa_axis_labels_dynamic(ax, classes, orientation='horizontal', show_labels=True):
    classes = [str(c) for c in classes]
    positions = np.arange(len(classes))
    if orientation == 'horizontal':
        ax.set_yticks(positions)
        ax.set_yticklabels([wrap_label(x, 24) for x in classes] if show_labels else [])
        if show_labels:
            ax.set_ylabel('HPA macro group')
        ax.invert_yaxis()
    else:
        ax.set_xticks(positions)
        ax.set_xticklabels(
            short_hpa_labels(classes) if show_labels else [],
            rotation=VERTICAL_LABEL_ROTATION,
            ha='right',
        )
        if show_labels:
            ax.set_xlabel('HPA macro group')

def probability_zero_overlap(n_universe, n_a, n_b):
    """P(|A∩B|=0) for two random sets of fixed sizes in one universe."""
    n_universe, n_a, n_b = map(int, (n_universe, n_a, n_b))
    if n_universe <= 0 or n_a < 0 or n_b < 0 or n_a > n_universe or n_b > n_universe:
        return np.nan
    if n_a + n_b > n_universe:
        return 0.0
    # log-combinations avoid integer overflow for large protein universes.
    log_p = (
        math.lgamma(n_universe - n_a + 1)
        - math.lgamma(n_b + 1)
        - math.lgamma(n_universe - n_a - n_b + 1)
        - math.lgamma(n_universe + 1)
        + math.lgamma(n_b + 1)
        + math.lgamma(n_universe - n_b + 1)
    )
    return float(math.exp(log_p))

def zero_overlap_diagnosis(n_universe, n_a, n_b, n_shared, p_zero):
    if n_universe == 0:
        return 'No common evaluated universe: overlap is not interpretable.'
    if n_a == 0 or n_b == 0:
        return 'One selected set is empty: zero overlap is automatic.'
    if n_shared > 0:
        return 'The selected sets overlap; use Jaccard and class composition to quantify disagreement.'
    if pd.notna(p_zero) and p_zero < 1e-6:
        return 'Exact zero is extremely unlikely for random sets of these sizes; audit partition construction, identifiers, preprocessing, and threshold logic first.'
    if pd.notna(p_zero) and p_zero < 0.05:
        return 'Exact zero is statistically unusual for sets of these sizes; technical separation is more plausible than chance alone.'
    return 'Exact zero is possible at this selectivity, but still requires universe and pipeline checks before biological interpretation.'

## Load HPA membership and validate the supplied full-class denominators

The supplied class sizes are used as the plotting denominators. The loaded Protein Atlas file is used to map individual gene symbols to the 20 macroclasses. The validation table below makes any HPA-version or identifier discrepancy visible instead of silently changing the denominator.

In [ ]:
HPA_CANDIDATE_FILES = [
    DOWNLOAD_DIR / 'hpa_proteinatlas.tsv.zip', DOWNLOAD_DIR / 'proteinatlas.tsv.zip',
    DOWNLOAD_DIR / 'hpa_proteinatlas.tsv.gz', DOWNLOAD_DIR / 'proteinatlas.tsv.gz',
    DOWNLOAD_DIR / 'hpa_proteinatlas.tsv', DOWNLOAD_DIR / 'proteinatlas.tsv',
    BASE_DIR / 'hpa_proteinatlas.tsv.zip', BASE_DIR / 'proteinatlas.tsv.zip',
    BASE_DIR / 'hpa_proteinatlas.tsv', BASE_DIR / 'proteinatlas.tsv',
]
if not any(p.exists() for p in HPA_CANDIDATE_FILES):
    HPA_CANDIDATE_FILES.extend(sorted(
        [p for p in BASE_DIR.rglob('*proteinatlas*') if p.is_file() and p.suffix.lower() in {'.zip', '.gz', '.tsv', '.csv'}],
        key=lambda p: len(p.parts)
    ))

hpa_path = next((p for p in HPA_CANDIDATE_FILES if p.exists()), None)
if hpa_path is None:
    raise FileNotFoundError('HPA file not found. Put proteinatlas.tsv(.zip/.gz) under the project root or pan_atlas_downloads/.')

hpa_raw = read_table_any(hpa_path)
gene_col = pick_column(hpa_raw, ['Gene', 'gene', 'Gene name', 'Gene symbol', 'gene_symbol', 'Gene name according to HGNC'])
protein_class_col = pick_column(hpa_raw, ['Protein class', 'protein_class', 'Protein classes'])

macro_map = {normalize_text(x): x for x in HPA_MACRO_CLASSES}
rows = []
for gene, protein_class in zip(hpa_raw[gene_col], hpa_raw[protein_class_col]):
    g = clean_gene_symbol(gene)
    if pd.isna(g):
        continue
    for term in split_hpa_terms(protein_class):
        macro_class = macro_map.get(normalize_text(term))
        if macro_class is not None:
            rows.append((g, macro_class))

hpa_macro = pd.DataFrame(rows, columns=['gene_symbol', 'hpa_macro_class']).drop_duplicates()
if hpa_macro.empty:
    raise ValueError('No configured HPA macro classes matched the Protein class column.')
hpa_macro['hpa_macro_class'] = pd.Categorical(hpa_macro['hpa_macro_class'], categories=HPA_MACRO_CLASSES, ordered=True)
hpa_macro = hpa_macro.sort_values(['hpa_macro_class', 'gene_symbol']).reset_index(drop=True)
hpa_macro.to_csv(TABLE_DIR / 'hpa_macro_class_membership_long.csv', index=False)

hpa_reference = pd.DataFrame({
    'hpa_macro_class': HPA_MACRO_CLASSES,
    'n_hpa_genes': [HPA_REFERENCE_COUNTS[c] for c in HPA_MACRO_CLASSES],
})
hpa_loaded_counts = (
    hpa_macro.groupby('hpa_macro_class', observed=False)['gene_symbol']
    .nunique().reindex(HPA_MACRO_CLASSES, fill_value=0)
    .rename('n_hpa_genes_in_loaded_file').reset_index()
)
hpa_denominator_validation = hpa_reference.merge(hpa_loaded_counts, on='hpa_macro_class', how='left')
hpa_denominator_validation['loaded_minus_reference'] = (
    hpa_denominator_validation['n_hpa_genes_in_loaded_file'] - hpa_denominator_validation['n_hpa_genes']
)
hpa_denominator_validation.to_csv(TABLE_DIR / 'hpa_reference_denominator_validation.csv', index=False)

print('HPA file:', hpa_path)
print(f'Unique mapped genes in the 20 retained classes: {hpa_macro.gene_symbol.nunique():,}')
display(hpa_denominator_validation)

if (hpa_denominator_validation['loaded_minus_reference'] > 0).any():
    display(Markdown(
        '**Warning:** the loaded HPA file contains more unique members than the supplied reference count for at least one class. '
        'Before interpreting percentages, reconcile the HPA release and identifier-cleaning rules. The notebook will stop if an evaluated count exceeds its reference denominator.'
    ))

## Part 1 — Sequential cascade with full-HPA denominators

In [ ]:
EXCLUDED_SOURCE_IDS = {'procedural', 'proxy_linear'}
EXCLUDED_PATH_KEYWORDS = {'procedural_tables', 'procedural_proxy_norm_tables'}

def find_cascade_long_files():
    files = [Path(p) for p in CASCADE_LONG_FILES if Path(p).exists()]
    if AUTO_SEARCH_CASCADE_FILES:
        standard = [
            BASE_DIR / 'procedural_proxy_z_tables' / 'protein_names_by_cascade_partition' / 'cascade_partition_proteins_long.csv',
            BASE_DIR / 'protein_names_by_cascade_partition' / 'cascade_partition_proteins_long.csv',
        ]
        files.extend([p for p in standard if p.exists()])
        files.extend(BASE_DIR.rglob('protein_names_by_cascade_partition/cascade_partition_proteins_long.csv'))
    out, seen = [], set()
    for p in files:
        sid = infer_source_id(p)
        path_text = str(p).lower()
        if sid in EXCLUDED_SOURCE_IDS or any(k in path_text for k in EXCLUDED_PATH_KEYWORDS):
            continue
        rp = p.resolve()
        if rp not in seen:
            out.append(p)
            seen.add(rp)
    return out

cascade_files = find_cascade_long_files()
if not cascade_files:
    raise FileNotFoundError('No cascade_partition_proteins_long.csv was found. Set CASCADE_LONG_FILES in the configuration cell.')

parts = []
for path in cascade_files:
    df = pd.read_csv(path, dtype=str, low_memory=False)
    df['source_path'] = str(path)
    df['source_id'] = infer_source_id(path)
    parts.append(df)
raw_partitions = pd.concat(parts, ignore_index=True, sort=False)

for required in ['protein', 'stage']:
    if required not in raw_partitions.columns:
        raise KeyError(f'Cascade export is missing required column: {required}')
for col, default in [('dataset', 'dataset'), ('analysis', 'analysis_unknown'), ('threshold_label', '')]:
    if col not in raw_partitions.columns:
        raw_partitions[col] = default

partitions = raw_partitions.copy()
partitions['gene_symbol'] = partitions['protein'].apply(clean_gene_symbol)
partitions = partitions.dropna(subset=['gene_symbol']).copy()
partitions['stage_group'] = partitions['stage'].apply(canonical_stage_group)
partitions['metric_family'] = partitions.apply(
    lambda r: parse_metric_family(r.get('analysis', ''), r.get('threshold_label', '')), axis=1
)
partitions['threshold_value'] = partitions.apply(parse_threshold_value, axis=1)
partitions['threshold_value'] = pd.to_numeric(partitions['threshold_value'], errors='coerce').round(2)

if DATASET_FILTER is not None:
    partitions = partitions[partitions['dataset'].astype(str).eq(str(DATASET_FILTER))].copy()
if SOURCE_ID_FILTER is not None:
    partitions = partitions[partitions['source_id'].astype(str).eq(str(SOURCE_ID_FILTER))].copy()

mask = pd.Series(False, index=partitions.index)
for metric, thresholds in SELECTED_THRESHOLDS_BY_METRIC.items():
    metric_mask = partitions['metric_family'].eq(metric)
    threshold_mask = pd.Series(False, index=partitions.index)
    for threshold in thresholds:
        threshold_mask |= np.isclose(partitions['threshold_value'].astype(float), threshold, atol=THRESHOLD_TOLERANCE, equal_nan=False)
    mask |= metric_mask & threshold_mask
selected = partitions[mask].copy()
if selected.empty:
    raise ValueError('No cascade rows matched Pearson 0.50/0.70 or MSE 0.20/0.30.')

selected['analysis_rank'] = selected.apply(lambda r: preferred_analysis_rank(r['analysis'], r['metric_family']), axis=1)
rank_cols = ['source_id', 'dataset', 'metric_family', 'threshold_value']
best_rank = selected.groupby(rank_cols, dropna=False)['analysis_rank'].transform('min')
selected = selected[selected['analysis_rank'].eq(best_rank)].copy()

selected['context_id'] = selected['source_id'].astype(str) + '__' + selected['dataset'].astype(str)
multiple_contexts = selected['context_id'].nunique() > 1
selected['case_id'] = selected.apply(
    lambda r: safe_filename(f"{r['context_id']}__{r['metric_family']}__{r['threshold_value']:.2f}"), axis=1
)
selected['case_label'] = selected.apply(
    lambda r: (
        f"{r['source_id']} | {r['dataset']} | {threshold_label(r['metric_family'], r['threshold_value'])}"
        if multiple_contexts else threshold_label(r['metric_family'], r['threshold_value'])
    ), axis=1
)

partition_key = ['case_id', 'gene_symbol']
stage_multiplicity = selected.groupby(partition_key)['stage_group'].nunique()
ambiguous = stage_multiplicity[stage_multiplicity > 1]
if len(ambiguous):
    print(f'Warning: {len(ambiguous):,} case/gene pairs occur in more than one stage. Keeping the earliest cascade stage.')
    stage_rank = {stage: i for i, stage in enumerate(CASCADE_STAGES)}
    selected['stage_rank'] = selected['stage_group'].map(stage_rank).fillna(999)
    selected = selected.sort_values(partition_key + ['stage_rank']).drop_duplicates(partition_key, keep='first')
else:
    selected = selected.drop_duplicates(partition_key + ['stage_group'])

selected.to_csv(TABLE_DIR / 'part1_selected_cascade_partitions.csv', index=False)
print('Cascade files:')
for path in cascade_files:
    print(' -', path)
print(f'Rows after threshold selection: {len(selected):,}')
display(ordered_cases(selected))

## HPA groups inside each threshold-set universe

For every retained threshold case, the notebook now displays a compact table with exactly two columns:

1. `hpa_macro_class`;
2. `n_genes_in_threshold_universe`.

The count is the number of unique genes belonging to that HPA macro class that are present in the evaluated universe for that threshold set. The complete audit table, including the full HPA denominator, percentages, and absent counts, is still exported to `part1_full_hpa_vs_case_universe_long.csv`.


In [ ]:
CASE_COLS = ['case_id', 'context_id', 'source_id', 'dataset', 'metric_family', 'threshold_value', 'case_label']
case_table = ordered_cases(selected)
case_class_grid = (
    case_table.assign(_key=1)
    .merge(hpa_reference.assign(_key=1), on='_key')
    .drop(columns='_key')
    .merge(hpa_loaded_counts, on='hpa_macro_class', how='left')
)
case_universe = selected[CASE_COLS + ['gene_symbol']].drop_duplicates()
case_macro = case_universe.merge(hpa_macro, on='gene_symbol', how='inner')
case_evaluated_counts = (
    case_macro.groupby(CASE_COLS + ['hpa_macro_class'], observed=True)['gene_symbol']
    .nunique().reset_index(name='n_hpa_genes_in_evaluated_universe')
)
case_hpa_universe = case_class_grid.merge(
    case_evaluated_counts, on=CASE_COLS + ['hpa_macro_class'], how='left'
)
case_hpa_universe['n_hpa_genes_in_evaluated_universe'] = case_hpa_universe['n_hpa_genes_in_evaluated_universe'].fillna(0).astype(int)
case_hpa_universe['pct_full_hpa_in_evaluated_universe'] = safe_percent(
    case_hpa_universe['n_hpa_genes_in_evaluated_universe'], case_hpa_universe['n_hpa_genes']
)
case_hpa_universe['n_hpa_genes_not_in_evaluated_universe'] = (
    case_hpa_universe['n_hpa_genes'] - case_hpa_universe['n_hpa_genes_in_evaluated_universe']
)

impossible = case_hpa_universe[case_hpa_universe['n_hpa_genes_not_in_evaluated_universe'] < 0]
if len(impossible):
    display(impossible[['case_label', 'hpa_macro_class', 'n_hpa_genes', 'n_hpa_genes_in_evaluated_universe']])
    raise ValueError(
        'At least one evaluated HPA count exceeds the supplied full-class denominator. '
        'Reconcile the HPA release, aliases, and deduplication before plotting percentages.'
    )

case_hpa_universe.to_csv(TABLE_DIR / 'part1_full_hpa_vs_case_universe_long.csv', index=False)

threshold_universe_group_counts = (
    case_hpa_universe[
        CASE_COLS + ['hpa_macro_class', 'n_hpa_genes_in_evaluated_universe']
    ]
    .rename(columns={
        'n_hpa_genes_in_evaluated_universe': 'n_genes_in_threshold_universe'
    })
    .copy()
)
threshold_universe_group_counts.to_csv(
    TABLE_DIR / 'part1_hpa_group_counts_by_threshold_universe.csv', index=False
)

for case in case_table.itertuples(index=False):
    two_col = (
        threshold_universe_group_counts[
            threshold_universe_group_counts['case_id'].eq(case.case_id)
        ][['hpa_macro_class', 'n_genes_in_threshold_universe']]
        .set_index('hpa_macro_class')
        .reindex(HPA_MACRO_CLASSES, fill_value=0)
        .reset_index()
    )
    two_col['n_genes_in_threshold_universe'] = (
        two_col['n_genes_in_threshold_universe'].fillna(0).astype(int)
    )
    display(Markdown(
        f"### Threshold universe — `{case.context_id}` — {case.case_label}"
    ))
    display(two_col.style.hide(axis='index').format({
        'n_genes_in_threshold_universe': '{:,}'
    }))

## Corrected sequential percentage decomposition

In [ ]:
SEQUENTIAL_COMPONENTS = list(FULL_DECOMPOSITION_COLORS)
stage_to_component = {
    'Proxy model': 'Proxy-covered',
    'ElasticNet': 'ElasticNet additional',
    'XGB': 'XGB additional',
    'Still not good': 'Evaluated but not predicted well',
    'Other': 'Evaluated but not predicted well',
}

stage_macro = (
    selected[CASE_COLS + ['stage_group', 'gene_symbol']].drop_duplicates()
    .merge(hpa_macro, on='gene_symbol', how='inner')
)
stage_macro['component'] = stage_macro['stage_group'].map(stage_to_component)
stage_component_counts = (
    stage_macro.groupby(CASE_COLS + ['hpa_macro_class', 'component'], observed=True)['gene_symbol']
    .nunique().reset_index(name='n_genes')
)

base_case_class = case_hpa_universe[
    CASE_COLS + ['hpa_macro_class', 'n_hpa_genes', 'n_hpa_genes_in_loaded_file',
                 'n_hpa_genes_in_evaluated_universe', 'n_hpa_genes_not_in_evaluated_universe']
].drop_duplicates()
component_grid = (
    base_case_class.assign(_key=1)
    .merge(pd.DataFrame({'component': SEQUENTIAL_COMPONENTS, '_key': 1}), on='_key')
    .drop(columns='_key')
)
sequential_full_hpa = component_grid.merge(
    stage_component_counts,
    on=CASE_COLS + ['hpa_macro_class', 'component'],
    how='left'
)
sequential_full_hpa['n_genes'] = sequential_full_hpa['n_genes'].fillna(0).astype(int)
not_eval_mask = sequential_full_hpa['component'].eq('Not evaluated / absent from universe')
sequential_full_hpa.loc[not_eval_mask, 'n_genes'] = sequential_full_hpa.loc[
    not_eval_mask, 'n_hpa_genes_not_in_evaluated_universe'
].astype(int)
sequential_full_hpa['pct_of_full_hpa_class'] = safe_percent(
    sequential_full_hpa['n_genes'], sequential_full_hpa['n_hpa_genes']
)
sequential_full_hpa['hpa_macro_class'] = pd.Categorical(
    sequential_full_hpa['hpa_macro_class'], categories=HPA_MACRO_CLASSES, ordered=True
)

validation = (
    sequential_full_hpa.groupby(CASE_COLS + ['hpa_macro_class', 'n_hpa_genes'], observed=True)['n_genes']
    .sum().reset_index(name='component_sum')
)
validation['difference_from_full_hpa'] = validation['component_sum'] - validation['n_hpa_genes']
if validation['difference_from_full_hpa'].ne(0).any():
    display(validation[validation['difference_from_full_hpa'].ne(0)])
    raise ValueError('Sequential decomposition does not sum to the full HPA denominator.')

sequential_full_hpa.to_csv(TABLE_DIR / 'part1_sequential_full_hpa_percentage_decomposition.csv', index=False)
print('All sequential class decompositions sum exactly to 100% of the supplied full HPA class size.')

In [ ]:
def _case_order_for_context(context_id):
    return case_table[case_table['context_id'].eq(context_id)].sort_values(
        ['metric_family', 'threshold_value'],
        key=lambda s: s.map(METRIC_ORDER) if s.name == 'metric_family' else s
    )

def _format_vertical_hpa_axis(ax):
    x = np.arange(len(HPA_MACRO_CLASSES))
    ax.set_xticks(x)
    ax.set_xticklabels(HPA_SHORT_LABELS, rotation=VERTICAL_LABEL_ROTATION, ha='right')
    ax.set_xlabel('HPA macro class')
    ax.grid(axis='y', alpha=0.16, linewidth=0.6)

def plot_full_decomposition_grid(data, context_id, stem_prefix):
    cases = _case_order_for_context(context_id)
    fig, axes, _, _ = compact_grid(len(cases), panel_size=VERTICAL_PANEL_SIZE, max_cols=2)
    for ax, case in zip(axes, cases.itertuples(index=False)):
        sub = data[data['case_id'].eq(case.case_id)]
        pivot = sub.pivot_table(
            index='hpa_macro_class', columns='component', values='pct_of_full_hpa_class',
            aggfunc='sum', fill_value=0, observed=False
        ).reindex(index=HPA_MACRO_CLASSES, columns=SEQUENTIAL_COMPONENTS, fill_value=0)
        x = np.arange(len(pivot))
        bottom = np.zeros(len(pivot))
        for component in SEQUENTIAL_COMPONENTS:
            values = pivot[component].to_numpy(dtype=float)
            ax.bar(
                x, values, bottom=bottom, width=BAR_WIDTH,
                color=FULL_DECOMPOSITION_COLORS[component],
                edgecolor='white', linewidth=0.25, label=component
            )
            bottom += values
        ax.set_ylim(0, 100)
        ax.set_ylabel('Share of full HPA class (%)')
        None
        _format_vertical_hpa_axis(ax)
    handles = [mpl.patches.Patch(facecolor=color, label=label) for label, color in FULL_DECOMPOSITION_COLORS.items()]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.005), ncol=2, frameon=False)
    None
    fig.tight_layout(rect=(0, 0.10, 1, 0.95), h_pad=1.68, w_pad=1.5)
    finish_figure(fig, f'{stem_prefix}_full_hpa_100pct_vertical_grid', show=DISPLAY_MAIN_FIGURES)

def plot_success_only_grid(data, context_id, stem_prefix):
    cases = _case_order_for_context(context_id)
    success_components = ['Proxy-covered', 'ElasticNet additional', 'XGB additional']
    max_value = 0.0
    pivots = {}
    for case in cases.itertuples(index=False):
        sub = data[data['case_id'].eq(case.case_id)]
        pivot = sub.pivot_table(
            index='hpa_macro_class', columns='component', values='pct_of_full_hpa_class',
            aggfunc='sum', fill_value=0, observed=False
        ).reindex(index=HPA_MACRO_CLASSES, columns=success_components, fill_value=0)
        pivots[case.case_id] = pivot
        max_value = max(max_value, float(pivot.sum(axis=1).max()))
    y_max = min(100, max(1.0, max_value * 1.18))

    fig, axes, _, _ = compact_grid(len(cases), panel_size=VERTICAL_PANEL_SIZE, max_cols=2)
    for ax, case in zip(axes, cases.itertuples(index=False)):
        pivot = pivots[case.case_id]
        x = np.arange(len(pivot))
        bottom = np.zeros(len(pivot))
        for component in success_components:
            values = pivot[component].to_numpy(dtype=float)
            ax.bar(
                x, values, bottom=bottom, width=BAR_WIDTH,
                color=FULL_DECOMPOSITION_COLORS[component],
                edgecolor='white', linewidth=0.25, label=component
            )
            bottom += values
        ax.set_ylim(0, y_max)
        ax.set_ylabel('Successfully covered full HPA class (%)')
        None
        _format_vertical_hpa_axis(ax)
    handles = [mpl.patches.Patch(facecolor=FULL_DECOMPOSITION_COLORS[c], label=c) for c in success_components]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.01), ncol=2, frameon=False)
    None
    fig.tight_layout(rect=(0, 0.10, 1, 0.95), h_pad=1.68, w_pad=1.5)
    finish_figure(fig, f'{stem_prefix}_successful_coverage_percentage_vertical_grid', show=DISPLAY_MAIN_FIGURES)

part1_figures = []
for context_id in case_table['context_id'].drop_duplicates():
    stem = safe_filename(f'part1_{context_id}')
    plot_full_decomposition_grid(sequential_full_hpa, context_id, stem)
    plot_success_only_grid(sequential_full_hpa, context_id, stem)
    part1_figures.extend([
        f'{stem}_full_hpa_100pct_vertical_grid',
        f'{stem}_successful_coverage_percentage_vertical_grid',
    ])

## Pearson–MSE overlap on a common evaluated universe

In [ ]:
def find_case_id(case_df, context_id, metric, threshold):
    hit = case_df[
        case_df['context_id'].eq(context_id)
        & case_df['metric_family'].eq(metric)
        & np.isclose(case_df['threshold_value'].astype(float), threshold)
    ]
    return None if hit.empty else hit.iloc[0]['case_id']

success_by_case = {
    case_id: set(group.loc[group['stage_group'].isin(SUCCESS_STAGES), 'gene_symbol'])
    for case_id, group in selected.groupby('case_id')
}
universe_by_case = {
    case_id: set(group['gene_symbol'])
    for case_id, group in selected.groupby('case_id')
}

# Expected threshold nesting.
nesting_rows = []
for context_id in case_table['context_id'].drop_duplicates():
    for metric, loose, strict in [('Pearson', 0.50, 0.70), ('MSE', 0.30, 0.20)]:
        loose_id = find_case_id(case_table, context_id, metric, loose)
        strict_id = find_case_id(case_table, context_id, metric, strict)
        if loose_id is None or strict_id is None:
            continue
        loose_set = success_by_case.get(loose_id, set())
        strict_set = success_by_case.get(strict_id, set())
        violations = strict_set - loose_set
        nesting_rows.append((context_id, metric, loose, strict, len(loose_set), len(strict_set), len(violations), len(violations) == 0))
sequential_nesting_audit = pd.DataFrame(
    nesting_rows,
    columns=['context_id', 'metric_family', 'loose_threshold', 'strict_threshold', 'n_loose_good', 'n_strict_good',
             'n_strict_only_violations', 'nested_as_expected']
)
sequential_nesting_audit.to_csv(TABLE_DIR / 'part1_threshold_nesting_audit.csv', index=False)
display(sequential_nesting_audit)

# Pairwise Pearson/MSE overlap, always restricted to the intersection of evaluated universes.
audit_rows = []
overlap_rows = []
for context_id in case_table['context_id'].drop_duplicates():
    for p_thr in PEARSON_THRESHOLDS:
        for m_thr in MSE_THRESHOLDS:
            p_id = find_case_id(case_table, context_id, 'Pearson', p_thr)
            m_id = find_case_id(case_table, context_id, 'MSE', m_thr)
            if p_id is None or m_id is None:
                continue
            common = universe_by_case[p_id] & universe_by_case[m_id]
            p_good = success_by_case[p_id] & common
            m_good = success_by_case[m_id] & common
            shared = p_good & m_good
            union = p_good | m_good
            n = len(common)
            expected = len(p_good) * len(m_good) / n if n else np.nan
            p_zero = probability_zero_overlap(n, len(p_good), len(m_good))
            audit_rows.append((
                context_id, p_thr, m_thr, n, len(p_good), len(m_good), len(shared), len(union),
                len(shared) / len(union) if union else 0.0, expected, p_zero,
                zero_overlap_diagnosis(n, len(p_good), len(m_good), len(shared), p_zero)
            ))

            for hpa_class in HPA_MACRO_CLASSES:
                class_genes = set(hpa_macro.loc[hpa_macro['hpa_macro_class'].eq(hpa_class), 'gene_symbol'])
                c_common = class_genes & common
                categories = {
                    'Pearson only': len((p_good - m_good) & class_genes),
                    'Shared': len(shared & class_genes),
                    'MSE only': len((m_good - p_good) & class_genes),
                    'Jointly evaluated, neither passes': len(c_common - union),
                    'Not jointly evaluated': HPA_REFERENCE_COUNTS[hpa_class] - len(c_common),
                }
                if categories['Not jointly evaluated'] < 0:
                    raise ValueError(f'Common-universe HPA count exceeds reference denominator for {hpa_class}.')
                for component, count in categories.items():
                    overlap_rows.append((context_id, p_thr, m_thr, hpa_class, component, count, 100.0 * count / HPA_REFERENCE_COUNTS[hpa_class]))

sequential_zero_overlap_audit = pd.DataFrame(
    audit_rows,
    columns=['context_id', 'pearson_threshold', 'mse_threshold', 'n_common_evaluated_universe',
             'n_pearson_good', 'n_mse_good', 'n_shared', 'n_union', 'jaccard',
             'expected_shared_if_random', 'probability_zero_if_random_fixed_sizes', 'diagnosis']
)
sequential_zero_overlap_audit.to_csv(TABLE_DIR / 'part1_pearson_mse_common_universe_overlap_audit.csv', index=False)
display(sequential_zero_overlap_audit)

sequential_pearson_mse_hpa = pd.DataFrame(
    overlap_rows,
    columns=['context_id', 'pearson_threshold', 'mse_threshold', 'hpa_macro_class', 'component', 'n_genes', 'pct_of_full_hpa_class']
)
sequential_pearson_mse_hpa.to_csv(TABLE_DIR / 'part1_pearson_mse_hpa_full_denominator_overlap.csv', index=False)

for context_id in case_table['context_id'].drop_duplicates():
    pairs = [(p, m) for p in PEARSON_THRESHOLDS for m in MSE_THRESHOLDS]
    fig, axes, _, _ = compact_grid(len(pairs), panel_size=VERTICAL_PANEL_SIZE, max_cols=2)
    for ax, (p_thr, m_thr) in zip(axes, pairs):
        sub = sequential_pearson_mse_hpa[
            sequential_pearson_mse_hpa['context_id'].eq(context_id)
            & np.isclose(sequential_pearson_mse_hpa['pearson_threshold'], p_thr)
            & np.isclose(sequential_pearson_mse_hpa['mse_threshold'], m_thr)
        ]
        pivot = sub.pivot_table(
            index='hpa_macro_class', columns='component', values='pct_of_full_hpa_class',
            aggfunc='sum', fill_value=0
        ).reindex(index=HPA_MACRO_CLASSES, columns=list(PEARSON_MSE_COLORS), fill_value=0)
        x = np.arange(len(pivot))
        bottom = np.zeros(len(pivot))
        for component, color in PEARSON_MSE_COLORS.items():
            vals = pivot[component].to_numpy(dtype=float)
            ax.bar(x, vals, bottom=bottom, width=BAR_WIDTH, color=color, edgecolor='white', linewidth=0.25)
            bottom += vals
        ax.set_ylim(0, 100)
        ax.set_ylabel('Share of full HPA class (%)')
        None
        _format_vertical_hpa_axis(ax)
    handles = [mpl.patches.Patch(facecolor=color, label=label) for label, color in PEARSON_MSE_COLORS.items()]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.005), ncol=2, frameon=False)
    None
    fig.tight_layout(rect=(0, 0.10, 1, 0.95), h_pad=1.68, w_pad=1.5)
    finish_figure(fig, f'part1_{safe_filename(context_id)}_pearson_mse_overlap_percentage_vertical_grid', show=DISPLAY_MAIN_FIGURES)
    part1_figures.append(f'part1_{safe_filename(context_id)}_pearson_mse_overlap_percentage_vertical_grid')

## Part 2 — Standalone model scores, using the full HPA denominator

In [ ]:
GENE_COLUMN_CANDIDATES = [
    'gene_symbol', 'gene', 'Gene', 'protein', 'protein_name', 'target', 'feature', 'Unnamed: 0'
]
MODEL_COLUMN_CANDIDATES = ['model', 'model_name', 'estimator', 'method', 'algorithm']
METRIC_COLUMN_CANDIDATES = ['metric', 'metric_family', 'measure', 'score_name']
VALUE_COLUMN_CANDIDATES = ['value', 'score', 'metric_value', 'performance', 'result']

def metric_from_column_name(column):
    text = normalize_text(column)
    if 'threshold' in text or 'cutoff' in text:
        return None
    return parse_metric_family(text)

def model_from_column_name(column):
    return canonical_model(column)

def infer_gene_column(df):
    col = pick_column(df, GENE_COLUMN_CANDIDATES, required=False)
    if col is not None:
        return col
    # Conservative fallback: first mostly non-numeric column with many unique values.
    for candidate in list(df.columns)[:5]:
        series = df[candidate]
        numeric_fraction = pd.to_numeric(series, errors='coerce').notna().mean()
        if numeric_fraction < 0.25 and series.nunique(dropna=True) > 10:
            return candidate
    return None

def standardize_score_frame(df, source_path):
    source_path = str(source_path)
    if df is None or df.empty:
        return pd.DataFrame(columns=['gene_symbol', 'model', 'metric_family', 'score', 'source_file'])
    gene_col = infer_gene_column(df)
    if gene_col is None:
        return pd.DataFrame(columns=['gene_symbol', 'model', 'metric_family', 'score', 'source_file'])

    model_col = pick_column(df, MODEL_COLUMN_CANDIDATES, required=False)
    metric_col = pick_column(df, METRIC_COLUMN_CANDIDATES, required=False)
    value_col = pick_column(df, VALUE_COLUMN_CANDIDATES, required=False)
    file_model = canonical_model(source_path)
    rows = []

    # Long layout.
    if metric_col is not None and value_col is not None:
        temp = pd.DataFrame({
            'gene_symbol': df[gene_col].map(clean_gene_symbol),
            'metric_family': df[metric_col].map(parse_metric_family),
            'score': pd.to_numeric(df[value_col], errors='coerce'),
        })
        if model_col is not None:
            temp['model'] = df[model_col].map(canonical_model)
        else:
            temp['model'] = file_model
        rows.append(temp)

    # Semi-wide layout with an explicit model column, or one file per model.
    explicit_or_file_model = model_col is not None or file_model is not None
    if explicit_or_file_model:
        for col in df.columns:
            metric = metric_from_column_name(col)
            if metric is None:
                continue
            # In semi-wide mode, do not use columns that encode a different model.
            encoded_model = model_from_column_name(col)
            if model_col is not None and encoded_model is not None:
                continue
            temp = pd.DataFrame({
                'gene_symbol': df[gene_col].map(clean_gene_symbol),
                'model': df[model_col].map(canonical_model) if model_col is not None else file_model,
                'metric_family': metric,
                'score': pd.to_numeric(df[col], errors='coerce'),
            })
            rows.append(temp)

    # Fully wide layout: model and metric both encoded in each column name.
    for col in df.columns:
        metric = metric_from_column_name(col)
        model = model_from_column_name(col)
        if metric is None or model is None:
            continue
        temp = pd.DataFrame({
            'gene_symbol': df[gene_col].map(clean_gene_symbol),
            'model': model,
            'metric_family': metric,
            'score': pd.to_numeric(df[col], errors='coerce'),
        })
        rows.append(temp)

    if not rows:
        return pd.DataFrame(columns=['gene_symbol', 'model', 'metric_family', 'score', 'source_file'])
    out = pd.concat(rows, ignore_index=True)
    out['source_file'] = source_path
    out = out.dropna(subset=['gene_symbol', 'model', 'metric_family', 'score'])
    out = out[out['model'].isin(MODELS) & out['metric_family'].isin(SELECTED_THRESHOLDS_BY_METRIC)]
    return out.drop_duplicates()

def discover_independent_score_files():
    manual = [Path(p) for p in INDEPENDENT_SCORE_FILES if Path(p).exists()]
    if manual:
        return manual
    if not AUTO_SEARCH_INDEPENDENT_SCORE_FILES:
        return []
    allowed_suffixes = {'.csv', '.tsv', '.txt', '.parquet', '.gz', '.zip'}
    name_tokens = ('proxy', 'elastic', 'enet', 'xgb', 'xgboost', 'model', 'prediction', 'metric', 'score', 'performance', 'evaluation')
    excluded_tokens = (
        '4.0_hpa_outputs', '4.0_hpa_v3_outputs', '4.0_hpa_v4_outputs', 'hpa_macroclass_', 'protein_names_by_cascade_partition',
        'proteinatlas', 'selected_standardized_cascade_partitions'
    )
    candidates = []
    for p in BASE_DIR.rglob('*'):
        if not p.is_file() or p.suffix.lower() not in allowed_suffixes:
            continue
        text = str(p).casefold()
        if any(token in text for token in excluded_tokens):
            continue
        if any(token in text for token in name_tokens):
            candidates.append(p)
    candidates = sorted(set(candidates), key=lambda p: (len(p.parts), str(p)))
    return candidates[:MAX_AUTO_SCORE_FILES]

score_files = discover_independent_score_files()
standardized_frames = []
read_failures = []
for path in score_files:
    try:
        frame = read_table_any(path)
        standardized = standardize_score_frame(frame, path)
        if not standardized.empty:
            standardized_frames.append(standardized)
    except Exception as exc:
        read_failures.append((str(path), str(exc)))

# Also inspect the cascade export for genuine wide score columns. This does not
# use the stage assignments; it only accepts columns that explicitly encode a model+metric.
cascade_wide_scores = standardize_score_frame(raw_partitions, 'cascade_export_wide_score_columns')
if not cascade_wide_scores.empty:
    standardized_frames.append(cascade_wide_scores)

if standardized_frames:
    independent_scores_raw = pd.concat(standardized_frames, ignore_index=True)
    duplicate_profile = (
        independent_scores_raw.groupby(['gene_symbol', 'model', 'metric_family'])['score']
        .agg(n_values='size', min_score='min', max_score='max')
        .reset_index()
    )
    duplicates = duplicate_profile[duplicate_profile['n_values'] > 1]
    if len(duplicates):
        print(f'Duplicate score groups: {len(duplicates):,}; reducing with {DUPLICATE_SCORE_AGGREGATION}.')
    agg_func = {'median': 'median', 'mean': 'mean', 'min': 'min', 'max': 'max'}[DUPLICATE_SCORE_AGGREGATION]
    independent_scores = (
        independent_scores_raw.groupby(['gene_symbol', 'model', 'metric_family'], as_index=False)['score']
        .agg(agg_func)
    )
    independent_scores.to_csv(TABLE_DIR / 'part2_independent_scores_standardized.csv', index=False)
    duplicate_profile.to_csv(TABLE_DIR / 'part2_independent_score_duplicate_profile.csv', index=False)
    INDEPENDENT_AVAILABLE = True
else:
    independent_scores_raw = pd.DataFrame(columns=['gene_symbol', 'model', 'metric_family', 'score', 'source_file'])
    independent_scores = independent_scores_raw.copy()
    INDEPENDENT_AVAILABLE = False

print(f'Candidate score files inspected: {len(score_files)}')
for path in score_files[:20]:
    print(' -', path)
if len(score_files) > 20:
    print(f' ... plus {len(score_files)-20} more')
if read_failures:
    print(f'Files that could not be read: {len(read_failures)} (saved to a table)')
    pd.DataFrame(read_failures, columns=['source_file', 'error']).to_csv(TABLE_DIR / 'part2_score_file_read_failures.csv', index=False)

if INDEPENDENT_AVAILABLE:
    score_inventory = (
        independent_scores.groupby(['model', 'metric_family'])['gene_symbol']
        .nunique().unstack(fill_value=0).reindex(index=MODELS, columns=list(SELECTED_THRESHOLDS_BY_METRIC), fill_value=0)
    )
    print(f'Standardized independent score rows: {len(independent_scores):,}')
    display(score_inventory)
else:
    message = (
        '**Standalone score tables were not found.** Part 1 is complete, but Part 2 cannot be reconstructed from the cascade stage column. '
        'Set `INDEPENDENT_SCORE_FILES` to one or more per-protein score tables and rerun from this cell.'
    )
    display(Markdown(message))
    if REQUIRE_INDEPENDENT_SCORES:
        raise FileNotFoundError(message)

## Standalone-score universe and percentage analyses

This section is executed only when genuine per-protein scores are available. Cascade stage labels are never treated as standalone model scores.

In [ ]:
part2_figures = []

if INDEPENDENT_AVAILABLE:
    independent_cases = pd.DataFrame([
        {
            'metric_family': metric,
            'threshold_value': float(threshold),
            'case_id': safe_filename(f'{metric}__{threshold:.2f}'),
            'case_label': threshold_label(metric, threshold),
        }
        for metric, thresholds in SELECTED_THRESHOLDS_BY_METRIC.items()
        for threshold in thresholds
    ])
    independent_cases['metric_order'] = independent_cases['metric_family'].map(METRIC_ORDER)
    independent_cases = independent_cases.sort_values(['metric_order', 'threshold_value']).drop(columns='metric_order')

    hit_rows, evaluated_rows = [], []
    for case in independent_cases.itertuples(index=False):
        metric_scores = independent_scores[independent_scores['metric_family'].eq(case.metric_family)]
        for model in MODELS:
            model_scores = metric_scores[metric_scores['model'].eq(model)]
            evaluated = set(model_scores['gene_symbol'])
            hits = set(model_scores.loc[
                passes_threshold(model_scores['score'], case.metric_family, case.threshold_value), 'gene_symbol'
            ])
            hit_rows.extend((case.case_id, case.case_label, case.metric_family, case.threshold_value, model, g) for g in hits)
            evaluated_rows.extend((case.case_id, case.metric_family, case.threshold_value, model, g) for g in evaluated)

    independent_hits = pd.DataFrame(
        hit_rows, columns=['case_id', 'case_label', 'metric_family', 'threshold_value', 'model', 'gene_symbol']
    ).drop_duplicates()
    independent_evaluated = pd.DataFrame(
        evaluated_rows, columns=['case_id', 'metric_family', 'threshold_value', 'model', 'gene_symbol']
    ).drop_duplicates()
    independent_hits.to_csv(TABLE_DIR / 'part2_independent_threshold_hits_long.csv', index=False)

    universe_rows = []
    for case in independent_cases.itertuples(index=False):
        model_sets = {
            model: set(independent_evaluated.loc[
                independent_evaluated['case_id'].eq(case.case_id) & independent_evaluated['model'].eq(model), 'gene_symbol'
            ]) for model in MODELS
        }
        if INDEPENDENT_UNIVERSE_MODE == 'intersection':
            universe = set.intersection(*(model_sets[m] for m in MODELS)) if all(model_sets[m] for m in MODELS) else set()
        else:
            universe = set.union(*(model_sets[m] for m in MODELS))
        universe_rows.extend((case.case_id, gene) for gene in universe)
    independent_universe = pd.DataFrame(universe_rows, columns=['case_id', 'gene_symbol']).drop_duplicates()

    # Full HPA versus standalone universe table.
    standalone_grid = (
        independent_cases.assign(_key=1)
        .merge(hpa_reference.assign(_key=1), on='_key').drop(columns='_key')
        .merge(hpa_loaded_counts, on='hpa_macro_class', how='left')
    )
    standalone_universe_counts = (
        independent_universe.merge(hpa_macro, on='gene_symbol', how='inner')
        .groupby(['case_id', 'hpa_macro_class'], observed=True)['gene_symbol']
        .nunique().reset_index(name='n_hpa_genes_in_standalone_universe')
    )
    standalone_universe_table = standalone_grid.merge(standalone_universe_counts, on=['case_id', 'hpa_macro_class'], how='left')
    standalone_universe_table['n_hpa_genes_in_standalone_universe'] = standalone_universe_table['n_hpa_genes_in_standalone_universe'].fillna(0).astype(int)
    standalone_universe_table['pct_full_hpa_in_standalone_universe'] = safe_percent(
        standalone_universe_table['n_hpa_genes_in_standalone_universe'], standalone_universe_table['n_hpa_genes']
    )
    standalone_universe_table['n_hpa_genes_not_in_standalone_universe'] = (
        standalone_universe_table['n_hpa_genes'] - standalone_universe_table['n_hpa_genes_in_standalone_universe']
    )
    if (standalone_universe_table['n_hpa_genes_not_in_standalone_universe'] < 0).any():
        raise ValueError('Standalone universe exceeds a supplied HPA class denominator.')
    standalone_universe_table.to_csv(TABLE_DIR / 'part2_full_hpa_vs_standalone_universe.csv', index=False)

    standalone_wide = hpa_reference.merge(hpa_loaded_counts, on='hpa_macro_class', how='left').set_index('hpa_macro_class')
    pieces = [standalone_wide]
    for case in independent_cases.itertuples(index=False):
        one = standalone_universe_table[standalone_universe_table['case_id'].eq(case.case_id)].set_index('hpa_macro_class')
        part = one[['n_hpa_genes_in_standalone_universe', 'pct_full_hpa_in_standalone_universe', 'n_hpa_genes_not_in_standalone_universe']].copy()
        part.columns = pd.MultiIndex.from_product([[case.case_label], ['intersection n', 'intersection %', 'not in universe n']])
        pieces.append(part)
    display(Markdown('### Full HPA sets versus the standalone-score universe'))
    display(pd.concat(pieces, axis=1).reindex(HPA_MACRO_CLASSES).style.format(
        {c: '{:.2f}%' for c in pd.concat(pieces, axis=1).columns if isinstance(c, tuple) and c[1] == 'intersection %'},
        na_rep='0'
    ))

    # Per-model standalone coverage as a percentage of the full HPA class.
    coverage_grid = (
        independent_cases.assign(_key=1)
        .merge(hpa_reference.assign(_key=1), on='_key')
        .merge(pd.DataFrame({'model': MODELS, '_key': 1}), on='_key')
        .drop(columns='_key')
    )
    hit_counts = (
        independent_hits.merge(hpa_macro, on='gene_symbol', how='inner')
        .groupby(['case_id', 'model', 'hpa_macro_class'], observed=True)['gene_symbol']
        .nunique().reset_index(name='n_good_genes')
    )
    independent_coverage = coverage_grid.merge(hit_counts, on=['case_id', 'model', 'hpa_macro_class'], how='left')
    independent_coverage['n_good_genes'] = independent_coverage['n_good_genes'].fillna(0).astype(int)
    independent_coverage['pct_of_full_hpa_class'] = safe_percent(independent_coverage['n_good_genes'], independent_coverage['n_hpa_genes'])
    independent_coverage.to_csv(TABLE_DIR / 'part2_standalone_model_coverage_full_hpa_percentage.csv', index=False)

    fig, axes, _, _ = compact_grid(len(independent_cases), panel_size=VERTICAL_PANEL_SIZE, max_cols=2)
    width = 0.24
    offsets = np.array([-1, 0, 1]) * width
    max_cov = float(independent_coverage.groupby(['case_id', 'hpa_macro_class'])['pct_of_full_hpa_class'].max().max())
    y_max = min(100, max(1.0, max_cov * 1.20))
    for ax, case in zip(axes, independent_cases.itertuples(index=False)):
        sub = independent_coverage[independent_coverage['case_id'].eq(case.case_id)]
        pivot = sub.pivot_table(index='hpa_macro_class', columns='model', values='pct_of_full_hpa_class', fill_value=0)
        pivot = pivot.reindex(index=HPA_MACRO_CLASSES, columns=MODELS, fill_value=0)
        x = np.arange(len(pivot))
        for offset, model in zip(offsets, MODELS):
            ax.bar(x + offset, pivot[model], width=width, color=MODEL_COLORS[model], label=model)
        ax.set_ylim(0, y_max)
        ax.set_ylabel('Covered full HPA class (%)')
        None
        _format_vertical_hpa_axis(ax)
    handles = [mpl.patches.Patch(facecolor=MODEL_COLORS[m], label=m) for m in MODELS]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.01), ncol=2, frameon=False)
    None
    fig.tight_layout(rect=(0, 0.10, 1, 0.95), h_pad=1.68, w_pad=1.5)
    finish_figure(fig, 'part2_standalone_model_coverage_percentage_vertical_grid', show=DISPLAY_MAIN_FIGURES)
    part2_figures.append('part2_standalone_model_coverage_percentage_vertical_grid')

    # Exact overlap composition and order-dependent marginal composition.
    overlap_rows, marginal_rows = [], []
    for case in independent_cases.itertuples(index=False):
        universe = set(independent_universe.loc[independent_universe['case_id'].eq(case.case_id), 'gene_symbol'])
        sets = {
            model: set(independent_hits.loc[
                independent_hits['case_id'].eq(case.case_id) & independent_hits['model'].eq(model), 'gene_symbol'
            ]) & universe for model in MODELS
        }
        union_hits = set.union(*(sets[m] for m in MODELS))
        marginal_sets = {
            'Proxy-covered': sets['Proxy model'],
            'ElasticNet additional': sets['ElasticNet'] - sets['Proxy model'],
            'XGB additional': sets['XGB'] - (sets['Proxy model'] | sets['ElasticNet']),
        }
        for hpa_class in HPA_MACRO_CLASSES:
            class_genes = set(hpa_macro.loc[hpa_macro['hpa_macro_class'].eq(hpa_class), 'gene_symbol'])
            c_universe = class_genes & universe
            overlap_categories = {
                'Proxy only': 0,
                'ElasticNet only': 0,
                'XGB only': 0,
                'Shared by 2 models': 0,
                'Shared by 3 models': 0,
                'Evaluated, no model covers': len(c_universe - union_hits),
                'Not evaluated / absent from universe': HPA_REFERENCE_COUNTS[hpa_class] - len(c_universe),
            }
            for gene in c_universe & union_hits:
                members = [m for m in MODELS if gene in sets[m]]
                if len(members) == 3:
                    overlap_categories['Shared by 3 models'] += 1
                elif len(members) == 2:
                    overlap_categories['Shared by 2 models'] += 1
                elif members[0] == 'Proxy model':
                    overlap_categories['Proxy only'] += 1
                elif members[0] == 'ElasticNet':
                    overlap_categories['ElasticNet only'] += 1
                else:
                    overlap_categories['XGB only'] += 1
            for component, count in overlap_categories.items():
                overlap_rows.append((case.case_id, case.case_label, hpa_class, component, count, 100.0 * count / HPA_REFERENCE_COUNTS[hpa_class]))

            marginal_categories = {
                'Proxy-covered': len(marginal_sets['Proxy-covered'] & class_genes),
                'ElasticNet additional': len(marginal_sets['ElasticNet additional'] & class_genes),
                'XGB additional': len(marginal_sets['XGB additional'] & class_genes),
                'Evaluated but not predicted well': len(c_universe - union_hits),
                'Not evaluated / absent from universe': HPA_REFERENCE_COUNTS[hpa_class] - len(c_universe),
            }
            for component, count in marginal_categories.items():
                marginal_rows.append((case.case_id, case.case_label, hpa_class, component, count, 100.0 * count / HPA_REFERENCE_COUNTS[hpa_class]))

    independent_overlap_hpa = pd.DataFrame(
        overlap_rows, columns=['case_id', 'case_label', 'hpa_macro_class', 'component', 'n_genes', 'pct_of_full_hpa_class']
    )
    independent_marginal_hpa = pd.DataFrame(
        marginal_rows, columns=['case_id', 'case_label', 'hpa_macro_class', 'component', 'n_genes', 'pct_of_full_hpa_class']
    )
    independent_overlap_hpa.to_csv(TABLE_DIR / 'part2_standalone_overlap_full_hpa_percentage.csv', index=False)
    independent_marginal_hpa.to_csv(TABLE_DIR / 'part2_standalone_marginal_full_hpa_percentage.csv', index=False)

    for data, colors, title, stem in [
        (independent_overlap_hpa, MODEL_OVERLAP_COLORS, 'Standalone model overlap — full HPA denominator', 'part2_model_overlap_percentage_vertical_grid'),
        (independent_marginal_hpa, FULL_DECOMPOSITION_COLORS, 'Standalone marginal additions (Proxy → ElasticNet → XGB)', 'part2_marginal_additions_percentage_vertical_grid'),
    ]:
        fig, axes, _, _ = compact_grid(len(independent_cases), panel_size=VERTICAL_PANEL_SIZE, max_cols=2)
        for ax, case in zip(axes, independent_cases.itertuples(index=False)):
            sub = data[data['case_id'].eq(case.case_id)]
            pivot = sub.pivot_table(index='hpa_macro_class', columns='component', values='pct_of_full_hpa_class', fill_value=0)
            pivot = pivot.reindex(index=HPA_MACRO_CLASSES, columns=list(colors), fill_value=0)
            x = np.arange(len(pivot))
            bottom = np.zeros(len(pivot))
            for component, color in colors.items():
                vals = pivot[component].to_numpy(dtype=float)
                ax.bar(x, vals, bottom=bottom, width=BAR_WIDTH, color=color, edgecolor='white', linewidth=0.25)
                bottom += vals
            ax.set_ylim(0, 100)
            ax.set_ylabel('Share of full HPA class (%)')
            None
            _format_vertical_hpa_axis(ax)
        handles = [mpl.patches.Patch(facecolor=color, label=label) for label, color in colors.items()]
        fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.005), ncol=2, frameon=False)
        None
        fig.tight_layout(rect=(0, 0.10, 1, 0.95), h_pad=1.68, w_pad=1.5)
        finish_figure(fig, stem, show=DISPLAY_MAIN_FIGURES)
        part2_figures.append(stem)
else:
    display(Markdown('Standalone-score plots are skipped because no valid per-protein score table was found.'))

## Standalone Pearson–MSE overlap and zero-overlap audit

In [ ]:
if INDEPENDENT_AVAILABLE:
    score_wide = independent_scores.pivot_table(
        index=['gene_symbol', 'model'], columns='metric_family', values='score', aggfunc='first'
    ).reset_index()

    raw_nesting_rows = []
    for model in MODELS:
        sub = score_wide[score_wide['model'].eq(model)]
        p_valid = sub.dropna(subset=['Pearson'])
        m_valid = sub.dropna(subset=['MSE'])
        p50 = set(p_valid.loc[p_valid['Pearson'].ge(0.50), 'gene_symbol'])
        p70 = set(p_valid.loc[p_valid['Pearson'].ge(0.70), 'gene_symbol'])
        m30 = set(m_valid.loc[m_valid['MSE'].le(0.30), 'gene_symbol'])
        m20 = set(m_valid.loc[m_valid['MSE'].le(0.20), 'gene_symbol'])
        raw_nesting_rows.extend([
            (model, 'Pearson', 0.50, 0.70, len(p50), len(p70), len(p70 - p50), len(p70 - p50) == 0),
            (model, 'MSE', 0.30, 0.20, len(m30), len(m20), len(m20 - m30), len(m20 - m30) == 0),
        ])
    independent_threshold_audit = pd.DataFrame(
        raw_nesting_rows,
        columns=['model', 'metric_family', 'loose_threshold', 'strict_threshold', 'n_loose_good', 'n_strict_good',
                 'n_strict_only_violations', 'nested_as_expected']
    )
    independent_threshold_audit.to_csv(TABLE_DIR / 'part2_raw_score_threshold_nesting_audit.csv', index=False)
    display(independent_threshold_audit)

    audit_rows, class_rows = [], []
    for model in MODELS:
        model_scores = score_wide[score_wide['model'].eq(model)].dropna(subset=['Pearson', 'MSE']).copy()
        common = set(model_scores['gene_symbol'])
        for p_thr in PEARSON_THRESHOLDS:
            for m_thr in MSE_THRESHOLDS:
                p_good = set(model_scores.loc[model_scores['Pearson'].ge(p_thr), 'gene_symbol'])
                m_good = set(model_scores.loc[model_scores['MSE'].le(m_thr), 'gene_symbol'])
                shared = p_good & m_good
                union = p_good | m_good
                n = len(common)
                expected = len(p_good) * len(m_good) / n if n else np.nan
                p_zero = probability_zero_overlap(n, len(p_good), len(m_good))
                audit_rows.append((
                    model, p_thr, m_thr, n, len(p_good), len(m_good), len(shared), len(union),
                    len(shared) / len(union) if union else 0.0, expected, p_zero,
                    zero_overlap_diagnosis(n, len(p_good), len(m_good), len(shared), p_zero)
                ))
                for hpa_class in HPA_MACRO_CLASSES:
                    class_genes = set(hpa_macro.loc[hpa_macro['hpa_macro_class'].eq(hpa_class), 'gene_symbol'])
                    c_common = class_genes & common
                    categories = {
                        'Pearson only': len((p_good - m_good) & class_genes),
                        'Shared': len(shared & class_genes),
                        'MSE only': len((m_good - p_good) & class_genes),
                        'Jointly evaluated, neither passes': len(c_common - union),
                        'Not jointly evaluated': HPA_REFERENCE_COUNTS[hpa_class] - len(c_common),
                    }
                    if categories['Not jointly evaluated'] < 0:
                        raise ValueError(f'Raw-score common universe exceeds reference HPA denominator for {hpa_class}.')
                    for component, count in categories.items():
                        class_rows.append((model, p_thr, m_thr, hpa_class, component, count, 100.0 * count / HPA_REFERENCE_COUNTS[hpa_class]))

    independent_zero_overlap_audit = pd.DataFrame(
        audit_rows,
        columns=['model', 'pearson_threshold', 'mse_threshold', 'n_common_score_rows', 'n_pearson_good', 'n_mse_good',
                 'n_shared', 'n_union', 'jaccard', 'expected_shared_if_random',
                 'probability_zero_if_random_fixed_sizes', 'diagnosis']
    )
    independent_zero_overlap_audit.to_csv(TABLE_DIR / 'part2_raw_score_pearson_mse_overlap_audit.csv', index=False)
    display(independent_zero_overlap_audit)

    independent_pearson_mse_hpa = pd.DataFrame(
        class_rows,
        columns=['model', 'pearson_threshold', 'mse_threshold', 'hpa_macro_class', 'component', 'n_genes', 'pct_of_full_hpa_class']
    )
    independent_pearson_mse_hpa.to_csv(TABLE_DIR / 'part2_raw_score_pearson_mse_hpa_overlap_percentage.csv', index=False)

    for model in MODELS:
        pairs = [(p, m) for p in PEARSON_THRESHOLDS for m in MSE_THRESHOLDS]
        fig, axes, _, _ = compact_grid(len(pairs), panel_size=VERTICAL_PANEL_SIZE, max_cols=2)
        for ax, (p_thr, m_thr) in zip(axes, pairs):
            sub = independent_pearson_mse_hpa[
                independent_pearson_mse_hpa['model'].eq(model)
                & np.isclose(independent_pearson_mse_hpa['pearson_threshold'], p_thr)
                & np.isclose(independent_pearson_mse_hpa['mse_threshold'], m_thr)
            ]
            pivot = sub.pivot_table(index='hpa_macro_class', columns='component', values='pct_of_full_hpa_class', fill_value=0)
            pivot = pivot.reindex(index=HPA_MACRO_CLASSES, columns=list(PEARSON_MSE_COLORS), fill_value=0)
            x = np.arange(len(pivot))
            bottom = np.zeros(len(pivot))
            for component, color in PEARSON_MSE_COLORS.items():
                vals = pivot[component].to_numpy(dtype=float)
                ax.bar(x, vals, bottom=bottom, width=BAR_WIDTH, color=color, edgecolor='white', linewidth=0.25)
                bottom += vals
            ax.set_ylim(0, 100)
            ax.set_ylabel('Share of full HPA class (%)')
            None
            _format_vertical_hpa_axis(ax)
        handles = [mpl.patches.Patch(facecolor=color, label=label) for label, color in PEARSON_MSE_COLORS.items()]
        fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.005), ncol=2, frameon=False)
        None
        fig.tight_layout(rect=(0, 0.10, 1, 0.95), h_pad=1.68, w_pad=1.5)
        stem = f'part2_{safe_filename(model)}_pearson_mse_overlap_percentage_vertical_grid'
        finish_figure(fig, stem, show=DISPLAY_MAIN_FIGURES)
        part2_figures.append(stem)

In [ ]:
# =============================================================================
# LOW-VARIANCE DIAGNOSTIC FOR ALL THREE MODELS
#
# Models:
#   - Proxy model
#   - ElasticNet
#   - XGB
#
# File:
#   data/proteomics_matched.joblib
#
# Main question:
#   Do MSE-only proteins have lower observed cross-sample variability than
#   Pearson-only and Shared proteins?
# =============================================================================

import joblib

from scipy.stats import mannwhitneyu

# -----------------------------------------------------------------------------
# Settings
# -----------------------------------------------------------------------------
BASE_DIR = Path(globals().get("BASE_DIR", Path.cwd()))

PROTEOMICS_PATH = (
    BASE_DIR
    / "data"
    / "proteomics_matched.joblib"
)

MODELS_TO_CHECK = [
    "Proxy model",
    "ElasticNet",
    "XGB",
]

PEARSON_THRESHOLDS_DIAG = [
    0.50,
    0.70,
]

MSE_THRESHOLDS_DIAG = [
    0.20,
    0.30,
]

MIN_N_FOR_TEST = 5
TOP_RED_CLASSES_PER_CASE = 8

# Set to True only when proteomics_matched.joblib is on exactly the same
# scale and transformation used to calculate the reported MSE values.
PROTEOMICS_IS_ON_MSE_SCALE = False

# Optional exact test-sample identifiers.
#
# Example:
# PROFILE_SAMPLE_IDS = test_sample_names
#
# When None, the cell tries common variable names and otherwise uses all rows.
PROFILE_SAMPLE_IDS = globals().get(
    "PROFILE_SAMPLE_IDS",
    None,
)

OUTPUT_TABLE_DIR = Path(
    globals().get(
        "TABLE_DIR",
        BASE_DIR / "diagnostic_tables",
    )
)

OUTPUT_FIGURE_DIR = Path(
    globals().get(
        "FIGURE_DIR",
        BASE_DIR / "diagnostic_figures",
    )
)

OUTPUT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# -----------------------------------------------------------------------------
# Check required notebook objects
# -----------------------------------------------------------------------------
if "score_wide" not in globals():

    if "independent_scores" not in globals():

        raise NameError(
            "Run the notebook from the beginning through the independent-score "
            "section first. The diagnostic requires `independent_scores` or "
            "`score_wide`."
        )

    score_wide = (
        independent_scores
        .pivot_table(
            index=[
                "gene_symbol",
                "model",
            ],
            columns="metric_family",
            values="score",
            aggfunc="first",
        )
        .reset_index()
    )

for required_name in [
    "hpa_macro",
    "HPA_MACRO_CLASSES",
]:

    if required_name not in globals():

        raise NameError(
            f"Missing `{required_name}`. "
            "Run the preceding HPA-loading cells first."
        )

required_score_columns = {
    "gene_symbol",
    "model",
    "Pearson",
    "MSE",
}

if not required_score_columns.issubset(
    score_wide.columns
):

    raise ValueError(
        "score_wide does not contain the required columns. "
        f"Available columns: {list(score_wide.columns)}"
    )

available_models = sorted(
    score_wide["model"]
    .dropna()
    .astype(str)
    .unique()
)

models_found = [
    model
    for model in MODELS_TO_CHECK
    if model in available_models
]

missing_models = [
    model
    for model in MODELS_TO_CHECK
    if model not in available_models
]

print(
    "Available models:",
    available_models,
)

print(
    "Models included:",
    models_found,
)

if missing_models:

    print(
        "Models not found and skipped:",
        missing_models,
    )

if not models_found:

    raise ValueError(
        "None of the requested models was found in score_wide."
    )

# -----------------------------------------------------------------------------
# Identifier cleaning
# -----------------------------------------------------------------------------
def diagnostic_clean_gene(value):

    if "clean_gene_symbol" in globals():

        return clean_gene_symbol(value)

    if pd.isna(value):

        return np.nan

    text = str(value).strip()

    if not text:

        return np.nan

    return re.split(
        r"[;|,/\s]+",
        text,
    )[0].upper()

score_wide = score_wide.copy()

score_wide["gene_symbol"] = (
    score_wide["gene_symbol"]
    .map(diagnostic_clean_gene)
)

score_wide["Pearson"] = pd.to_numeric(
    score_wide["Pearson"],
    errors="coerce",
)

score_wide["MSE"] = pd.to_numeric(
    score_wide["MSE"],
    errors="coerce",
)

# -----------------------------------------------------------------------------
# Load the proteomics matrix
# -----------------------------------------------------------------------------
if not PROTEOMICS_PATH.exists():

    candidates = sorted(
        BASE_DIR.rglob(
            "proteomics_matched.joblib"
        ),
        key=lambda path: (
            len(path.parts),
            str(path),
        ),
    )

    if not candidates:

        raise FileNotFoundError(
            "proteomics_matched.joblib was not found. "
            "Expected location:\n"
            f"{PROTEOMICS_PATH}"
        )

    PROTEOMICS_PATH = candidates[0]

loaded_object = joblib.load(
    PROTEOMICS_PATH
)

print(
    "\nProteomics file:",
    PROTEOMICS_PATH,
)

print(
    "Stored object type:",
    type(loaded_object).__name__,
)

def extract_dataframe(obj):
    """
    Extract a labeled DataFrame from common joblib structures.
    """

    if isinstance(
        obj,
        pd.DataFrame,
    ):

        return obj.copy()

    if isinstance(
        obj,
        dict,
    ):

        preferred_keys = [
            "proteomics_matched",
            "proteomics",
            "matrix",
            "data",
            "values",
            "Y",
            "y",
        ]

        key_map = {
            str(key).casefold(): key
            for key in obj
        }

        for candidate in preferred_keys:

            key = key_map.get(
                candidate.casefold()
            )

            if (
                key is not None
                and isinstance(
                    obj[key],
                    pd.DataFrame,
                )
            ):

                return obj[key].copy()

        for value in obj.values():

            if isinstance(
                value,
                pd.DataFrame,
            ):

                return value.copy()

    if isinstance(
        obj,
        (list, tuple),
    ):

        for value in obj:

            if isinstance(
                value,
                pd.DataFrame,
            ):

                return value.copy()

    raise TypeError(
        "The joblib must contain a labeled pandas DataFrame. "
        "The expected orientation is samples × proteins."
    )

proteomics = extract_dataframe(
    loaded_object
)

# -----------------------------------------------------------------------------
# Detect orientation
# -----------------------------------------------------------------------------
all_score_genes = set(
    score_wide["gene_symbol"]
    .dropna()
)

clean_columns = [
    diagnostic_clean_gene(value)
    for value in proteomics.columns
]

clean_index = [
    diagnostic_clean_gene(value)
    for value in proteomics.index
]

column_overlap = sum(
    gene in all_score_genes
    for gene in clean_columns
    if pd.notna(gene)
)

index_overlap = sum(
    gene in all_score_genes
    for gene in clean_index
    if pd.notna(gene)
)

if index_overlap > column_overlap:

    proteomics = proteomics.T

    orientation_message = (
        "Transposed because proteins were detected on rows."
    )

else:

    orientation_message = (
        "Kept unchanged because proteins were detected on columns."
    )

proteomics.columns = [
    diagnostic_clean_gene(value)
    for value in proteomics.columns
]

valid_columns = [
    pd.notna(value)
    for value in proteomics.columns
]

proteomics = proteomics.loc[
    :,
    valid_columns,
]

proteomics = proteomics.apply(
    pd.to_numeric,
    errors="coerce",
)

# Multiple protein columns can map to one cleaned gene symbol.
if proteomics.columns.duplicated().any():

    duplicate_count = int(
        proteomics.columns
        .duplicated()
        .sum()
    )

    print(
        f"Aggregating {duplicate_count:,} duplicate "
        "gene-symbol columns by median."
    )

    proteomics = (
        proteomics.T
        .groupby(level=0)
        .median()
        .T
    )

# -----------------------------------------------------------------------------
# Restrict to test samples when identifiers are available
# -----------------------------------------------------------------------------
if PROFILE_SAMPLE_IDS is None:

    for candidate_name in [
        "TEST_SAMPLE_IDS",
        "test_sample_ids",
        "test_samples",
        "H5_TEST_SAMPLES",
        "h5_test_samples",
    ]:

        if (
            candidate_name in globals()
            and globals()[candidate_name] is not None
        ):

            PROFILE_SAMPLE_IDS = list(
                globals()[candidate_name]
            )

            print(
                f"Using test samples from `{candidate_name}`."
            )

            break

if PROFILE_SAMPLE_IDS is not None:

    requested_samples = [
        str(value)
        for value in PROFILE_SAMPLE_IDS
    ]

    matrix_samples = [
        str(value)
        for value in proteomics.index
    ]

    position_by_sample = {
        sample: position
        for position, sample
        in enumerate(matrix_samples)
    }

    matched_samples = [
        sample
        for sample in requested_samples
        if sample in position_by_sample
    ]

    if len(matched_samples) < 3:

        raise ValueError(
            "Fewer than three requested test samples matched "
            "the proteomics matrix row index."
        )

    proteomics = proteomics.iloc[
        [
            position_by_sample[sample]
            for sample in matched_samples
        ]
    ]

    sample_scope = (
        f"{len(matched_samples):,} matched test samples"
    )

else:

    sample_scope = (
        f"all {len(proteomics):,} samples"
    )

    display(
        Markdown(
            "**Sample-scope warning:** no test-sample list was found. "
            "The profile statistics therefore use every row in "
            "`proteomics_matched.joblib`. This tests intrinsic observed "
            "variability, but exact agreement with the evaluation metrics "
            "requires the same held-out samples."
        )
    )

print(
    "Orientation:",
    orientation_message,
)

print(
    "Proteomics shape:",
    proteomics.shape,
)

print(
    "Samples used:",
    sample_scope,
)

# -----------------------------------------------------------------------------
# Compute one observed-profile table shared by all models
# -----------------------------------------------------------------------------
matched_genes = sorted(
    all_score_genes
    & set(proteomics.columns)
)

if not matched_genes:

    raise ValueError(
        "No score-table genes matched the proteomics columns. "
        "Check whether the matrix columns contain gene symbols."
    )

profile_rows = []

for gene in matched_genes:

    values = (
        proteomics[gene]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) < 3:

        continue

    q05, q25, q75, q95 = np.quantile(
        values,
        [
            0.05,
            0.25,
            0.75,
            0.95,
        ],
    )

    mean_value = float(
        np.mean(values)
    )

    median_value = float(
        np.median(values)
    )

    # Population variance equals the MSE of predicting the protein mean
    # for every sample.
    observed_variance = float(
        np.mean(
            (
                values
                - mean_value
            ) ** 2
        )
    )

    profile_rows.append({
        "gene_symbol": gene,

        "n_profile_samples": len(values),

        "missing_fraction": float(
            proteomics[gene]
            .isna()
            .mean()
        ),

        "observed_mean": mean_value,

        "observed_median": median_value,

        "observed_variance": observed_variance,

        "observed_sd": float(
            np.sqrt(
                observed_variance
            )
        ),

        "observed_iqr": float(
            q75 - q25
        ),

        "observed_90pct_span": float(
            q95 - q05
        ),

        "observed_range": float(
            np.max(values)
            - np.min(values)
        ),

        "observed_mad": float(
            np.median(
                np.abs(
                    values
                    - median_value
                )
            )
        ),

        # Same numeric value as population variance.
        "mean_baseline_mse": observed_variance,
    })

profile_metrics = pd.DataFrame(
    profile_rows
)

for metric in [
    "observed_variance",
    "observed_sd",
    "observed_iqr",
    "observed_90pct_span",
    "observed_range",
    "observed_mad",
]:

    profile_metrics[
        f"{metric}_percentile"
    ] = (
        100
        * profile_metrics[metric]
        .rank(
            method="average",
            pct=True,
        )
    )

profile_gene_set = set(
    profile_metrics["gene_symbol"]
)

print(
    f"Matched profile genes: "
    f"{len(profile_gene_set):,} / "
    f"{len(all_score_genes):,}"
)

# -----------------------------------------------------------------------------
# Detect per-protein z-scoring
# -----------------------------------------------------------------------------
fraction_variance_near_one = (
    profile_metrics[
        "observed_variance"
    ]
    .between(
        0.90,
        1.10,
    )
    .mean()
)

if fraction_variance_near_one >= 0.80:

    display(
        Markdown(
            "**Standardization warning:** at least 80% of proteins have "
            "variance between 0.90 and 1.10. The matrix may have been "
            "z-scored separately for each protein. If so, between-protein "
            "variance differences have been removed and this diagnostic "
            "cannot test the low-variance hypothesis reliably."
        )
    )

# -----------------------------------------------------------------------------
# Benjamini-Hochberg correction
# -----------------------------------------------------------------------------
def bh_adjust(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    output = np.full(
        len(p_values),
        np.nan,
    )

    valid = np.isfinite(
        p_values
    )

    if not valid.any():

        return output

    valid_values = p_values[
        valid
    ]

    order = np.argsort(
        valid_values
    )

    ranked = valid_values[
        order
    ]

    adjusted = (
        ranked
        * len(ranked)
        / np.arange(
            1,
            len(ranked) + 1,
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.clip(
        adjusted,
        0,
        1,
    )

    restored = np.empty_like(
        adjusted
    )

    restored[order] = adjusted

    output[
        np.where(valid)[0]
    ] = restored

    return output

# -----------------------------------------------------------------------------
# Build categories and tests for every model and threshold combination
# -----------------------------------------------------------------------------
CATEGORY_ORDER = [
    "Pearson only",
    "Shared",
    "MSE only",
    "Neither passes",
]

TEST_METRICS = [
    (
        "Lower observed variance",
        "observed_variance",
    ),
    (
        "Narrower observed IQR",
        "observed_iqr",
    ),
    (
        "Narrower 5th–95th percentile span",
        "observed_90pct_span",
    ),
    (
        "Lower robust MAD",
        "observed_mad",
    ),
]

membership_parts = []
category_summary_parts = []
test_rows = []
class_count_parts = []
class_profile_parts = []

for model in models_found:

    model_scores = (
        score_wide[
            score_wide["model"].eq(model)
        ]
        .dropna(
            subset=[
                "gene_symbol",
                "Pearson",
                "MSE",
            ]
        )
        .copy()
    )

    # One row per gene and model.
    model_scores = (
        model_scores
        .groupby(
            "gene_symbol",
            as_index=False,
        )
        .agg(
            Pearson=(
                "Pearson",
                "max",
            ),
            MSE=(
                "MSE",
                "min",
            ),
        )
    )

    common_profile_genes = (
        set(model_scores["gene_symbol"])
        & profile_gene_set
    )

    for pearson_threshold in PEARSON_THRESHOLDS_DIAG:

        for mse_threshold in MSE_THRESHOLDS_DIAG:

            pearson_good = (
                set(
                    model_scores.loc[
                        model_scores["Pearson"].ge(
                            pearson_threshold
                        ),
                        "gene_symbol",
                    ]
                )
                & common_profile_genes
            )

            mse_good = (
                set(
                    model_scores.loc[
                        model_scores["MSE"].le(
                            mse_threshold
                        ),
                        "gene_symbol",
                    ]
                )
                & common_profile_genes
            )

            membership_rows = []

            for gene in sorted(
                common_profile_genes
            ):

                pearson_pass = (
                    gene in pearson_good
                )

                mse_pass = (
                    gene in mse_good
                )

                if pearson_pass and mse_pass:

                    category = "Shared"

                elif pearson_pass:

                    category = "Pearson only"

                elif mse_pass:

                    category = "MSE only"

                else:

                    category = "Neither passes"

                membership_rows.append({
                    "model": model,
                    "pearson_threshold": pearson_threshold,
                    "mse_threshold": mse_threshold,
                    "gene_symbol": gene,
                    "category": category,
                })

            membership = pd.DataFrame(
                membership_rows
            )

            membership_parts.append(
                membership
            )

            merged = membership.merge(
                profile_metrics,
                on="gene_symbol",
                how="left",
            )

            # -------------------------------------------------------------
            # Category-level profile summary
            # -------------------------------------------------------------
            category_summary = (
                merged
                .groupby(
                    "category",
                    observed=True,
                )
                .agg(
                    n_genes=(
                        "gene_symbol",
                        "nunique",
                    ),

                    median_observed_variance=(
                        "observed_variance",
                        "median",
                    ),

                    median_variance_percentile=(
                        "observed_variance_percentile",
                        "median",
                    ),

                    median_observed_sd=(
                        "observed_sd",
                        "median",
                    ),

                    median_observed_iqr=(
                        "observed_iqr",
                        "median",
                    ),

                    median_observed_90pct_span=(
                        "observed_90pct_span",
                        "median",
                    ),

                    median_observed_mad=(
                        "observed_mad",
                        "median",
                    ),

                    median_mean_baseline_mse=(
                        "mean_baseline_mse",
                        "median",
                    ),
                )
                .reindex(
                    CATEGORY_ORDER
                )
                .dropna(
                    how="all"
                )
                .reset_index()
            )

            category_summary.insert(
                0,
                "mse_threshold",
                mse_threshold,
            )

            category_summary.insert(
                0,
                "pearson_threshold",
                pearson_threshold,
            )

            category_summary.insert(
                0,
                "model",
                model,
            )

            category_summary_parts.append(
                category_summary
            )

            # -------------------------------------------------------------
            # Directional tests
            #
            # H1: MSE-only values are smaller than the comparison group.
            # -------------------------------------------------------------
            for comparison_category in [
                "Pearson only",
                "Shared",
            ]:

                for hypothesis, metric in TEST_METRICS:

                    mse_only_values = (
                        merged.loc[
                            merged["category"].eq(
                                "MSE only"
                            ),
                            metric,
                        ]
                        .dropna()
                    )

                    comparison_values = (
                        merged.loc[
                            merged["category"].eq(
                                comparison_category
                            ),
                            metric,
                        ]
                        .dropna()
                    )

                    if (
                        len(mse_only_values)
                        >= MIN_N_FOR_TEST
                        and len(comparison_values)
                        >= MIN_N_FOR_TEST
                    ):

                        test_result = mannwhitneyu(
                            mse_only_values,
                            comparison_values,
                            alternative="less",
                        )

                        p_value = float(
                            test_result.pvalue
                        )

                    else:

                        p_value = np.nan

                    median_mse_only = (
                        float(
                            mse_only_values.median()
                        )
                        if len(mse_only_values)
                        else np.nan
                    )

                    median_comparison = (
                        float(
                            comparison_values.median()
                        )
                        if len(comparison_values)
                        else np.nan
                    )

                    reduction_pct = (
                        100
                        * (
                            median_comparison
                            - median_mse_only
                        )
                        / median_comparison
                        if (
                            np.isfinite(median_comparison)
                            and median_comparison != 0
                            and np.isfinite(median_mse_only)
                        )
                        else np.nan
                    )

                    test_rows.append({
                        "model": model,
                        "pearson_threshold": pearson_threshold,
                        "mse_threshold": mse_threshold,

                        "comparison": (
                            f"MSE only < "
                            f"{comparison_category}"
                        ),

                        "hypothesis": hypothesis,
                        "metric": metric,

                        "n_mse_only": len(
                            mse_only_values
                        ),

                        "n_comparison": len(
                            comparison_values
                        ),

                        "median_mse_only": median_mse_only,
                        "median_comparison": median_comparison,
                        "relative_reduction_pct": reduction_pct,

                        "direction_supported": (
                            median_mse_only
                            < median_comparison
                            if (
                                np.isfinite(median_mse_only)
                                and np.isfinite(median_comparison)
                            )
                            else False
                        ),

                        "one_sided_mann_whitney_p": p_value,
                    })

            # -------------------------------------------------------------
            # HPA-class counts
            # -------------------------------------------------------------
            class_merged = merged.merge(
                hpa_macro[
                    [
                        "gene_symbol",
                        "hpa_macro_class",
                    ]
                ]
                .drop_duplicates(),
                on="gene_symbol",
                how="inner",
            )

            class_counts = (
                class_merged
                .groupby(
                    [
                        "hpa_macro_class",
                        "category",
                    ],
                    observed=True,
                )["gene_symbol"]
                .nunique()
                .unstack(
                    fill_value=0
                )
                .reindex(
                    index=HPA_MACRO_CLASSES,
                    columns=CATEGORY_ORDER,
                    fill_value=0,
                )
            )

            class_counts["jointly_evaluated_n"] = (
                class_counts[
                    CATEGORY_ORDER
                ]
                .sum(
                    axis=1
                )
            )

            class_counts["successful_n"] = (
                class_counts[
                    [
                        "Pearson only",
                        "Shared",
                        "MSE only",
                    ]
                ]
                .sum(
                    axis=1
                )
            )

            class_counts["mse_only_pct_of_joint"] = np.where(
                class_counts["jointly_evaluated_n"] > 0,
                (
                    100
                    * class_counts["MSE only"]
                    / class_counts["jointly_evaluated_n"]
                ),
                np.nan,
            )

            class_counts["mse_only_pct_of_successes"] = np.where(
                class_counts["successful_n"] > 0,
                (
                    100
                    * class_counts["MSE only"]
                    / class_counts["successful_n"]
                ),
                np.nan,
            )

            class_counts = class_counts.reset_index()

            class_counts.insert(
                0,
                "mse_threshold",
                mse_threshold,
            )

            class_counts.insert(
                0,
                "pearson_threshold",
                pearson_threshold,
            )

            class_counts.insert(
                0,
                "model",
                model,
            )

            class_count_parts.append(
                class_counts
            )

            # -------------------------------------------------------------
            # HPA-class profile summary
            # -------------------------------------------------------------
            class_profiles = (
                class_merged
                .groupby(
                    [
                        "hpa_macro_class",
                        "category",
                    ],
                    observed=True,
                )
                .agg(
                    n_genes=(
                        "gene_symbol",
                        "nunique",
                    ),

                    median_observed_variance=(
                        "observed_variance",
                        "median",
                    ),

                    median_variance_percentile=(
                        "observed_variance_percentile",
                        "median",
                    ),

                    median_observed_iqr=(
                        "observed_iqr",
                        "median",
                    ),

                    median_observed_90pct_span=(
                        "observed_90pct_span",
                        "median",
                    ),

                    median_observed_mad=(
                        "observed_mad",
                        "median",
                    ),
                )
                .reset_index()
            )

            class_profiles.insert(
                0,
                "mse_threshold",
                mse_threshold,
            )

            class_profiles.insert(
                0,
                "pearson_threshold",
                pearson_threshold,
            )

            class_profiles.insert(
                0,
                "model",
                model,
            )

            class_profile_parts.append(
                class_profiles
            )

# -----------------------------------------------------------------------------
# Combine results
# -----------------------------------------------------------------------------
membership_all_models = pd.concat(
    membership_parts,
    ignore_index=True,
)

category_summary_all_models = pd.concat(
    category_summary_parts,
    ignore_index=True,
)

tests_all_models = pd.DataFrame(
    test_rows
)

class_counts_all_models = pd.concat(
    class_count_parts,
    ignore_index=True,
)

class_profiles_all_models = pd.concat(
    class_profile_parts,
    ignore_index=True,
)

# Correct across every model, threshold, comparison, and dispersion statistic.
tests_all_models["FDR_BH_global"] = bh_adjust(
    tests_all_models[
        "one_sided_mann_whitney_p"
    ]
)

# Also provide FDR correction within each model.
tests_all_models["FDR_BH_within_model"] = (
    tests_all_models
    .groupby(
        "model",
        group_keys=False,
    )["one_sided_mann_whitney_p"]
    .transform(
        bh_adjust
    )
)

tests_all_models["p_display"] = np.where(
    tests_all_models[
        "one_sided_mann_whitney_p"
    ].eq(0),
    "< numerical precision",
    tests_all_models[
        "one_sided_mann_whitney_p"
    ].map(
        lambda value: (
            f"{value:.3e}"
            if pd.notna(value)
            else ""
        )
    ),
)

# -----------------------------------------------------------------------------
# Compact cross-model variance table
# -----------------------------------------------------------------------------
variance_comparison = (
    tests_all_models[
        tests_all_models["metric"].eq(
            "observed_variance"
        )
        & tests_all_models["comparison"].eq(
            "MSE only < Pearson only"
        )
    ]
    .copy()
    .sort_values(
        [
            "model",
            "pearson_threshold",
            "mse_threshold",
        ]
    )
)

# -----------------------------------------------------------------------------
# Most red-dominated HPA classes per model and threshold pair
# -----------------------------------------------------------------------------
top_red_classes_all_models = (
    class_counts_all_models[
        class_counts_all_models[
            "successful_n"
        ].ge(
            MIN_N_FOR_TEST
        )
    ]
    .sort_values(
        [
            "model",
            "pearson_threshold",
            "mse_threshold",
            "mse_only_pct_of_successes",
            "MSE only",
        ],
        ascending=[
            True,
            True,
            True,
            False,
            False,
        ],
    )
    .groupby(
        [
            "model",
            "pearson_threshold",
            "mse_threshold",
        ],
        observed=True,
    )
    .head(
        TOP_RED_CLASSES_PER_CASE
    )
)

mse_only_class_profiles = (
    class_profiles_all_models[
        class_profiles_all_models[
            "category"
        ].eq(
            "MSE only"
        )
    ]
    .drop(
        columns="category"
    )
)

top_red_classes_all_models = (
    top_red_classes_all_models
    .merge(
        mse_only_class_profiles,
        on=[
            "model",
            "pearson_threshold",
            "mse_threshold",
            "hpa_macro_class",
        ],
        how="left",
    )
)

# -----------------------------------------------------------------------------
# Save all tables
# -----------------------------------------------------------------------------
profile_metrics.to_csv(
    OUTPUT_TABLE_DIR
    / "all_models_observed_profile_metrics.csv",
    index=False,
)

membership_all_models.to_csv(
    OUTPUT_TABLE_DIR
    / "all_models_pearson_mse_profile_membership.csv",
    index=False,
)

category_summary_all_models.to_csv(
    OUTPUT_TABLE_DIR
    / "all_models_profile_category_summary.csv",
    index=False,
)

tests_all_models.to_csv(
    OUTPUT_TABLE_DIR
    / "all_models_profile_directional_tests.csv",
    index=False,
)

variance_comparison.to_csv(
    OUTPUT_TABLE_DIR
    / "all_models_variance_comparison_compact.csv",
    index=False,
)

class_counts_all_models.to_csv(
    OUTPUT_TABLE_DIR
    / "all_models_hpa_mse_only_dominance.csv",
    index=False,
)

class_profiles_all_models.to_csv(
    OUTPUT_TABLE_DIR
    / "all_models_hpa_profile_summary.csv",
    index=False,
)

top_red_classes_all_models.to_csv(
    OUTPUT_TABLE_DIR
    / "all_models_top_mse_only_hpa_classes.csv",
    index=False,
)

# -----------------------------------------------------------------------------
# Display main cross-model result
# -----------------------------------------------------------------------------
display(
    Markdown(
        "## Cross-model test: do MSE-only proteins have lower variance?"
    )
)

display_columns = [
    "model",
    "pearson_threshold",
    "mse_threshold",
    "n_mse_only",
    "n_comparison",
    "median_mse_only",
    "median_comparison",
    "relative_reduction_pct",
    "direction_supported",
    "p_display",
    "FDR_BH_global",
]

display(
    variance_comparison[
        display_columns
    ]
    .style
    .hide(
        axis="index"
    )
    .format({
        "pearson_threshold": "{:.2f}",
        "mse_threshold": "{:.2f}",
        "median_mse_only": "{:.4g}",
        "median_comparison": "{:.4g}",
        "relative_reduction_pct": "{:.1f}%",
        "FDR_BH_global": "{:.3e}",
    })
)

display(
    Markdown(
        "## All directional tests"
    )
)

display(
    tests_all_models
    .sort_values(
        [
            "model",
            "pearson_threshold",
            "mse_threshold",
            "comparison",
            "metric",
        ]
    )
    .style
    .hide(
        axis="index"
    )
    .format({
        "pearson_threshold": "{:.2f}",
        "mse_threshold": "{:.2f}",
        "median_mse_only": "{:.4g}",
        "median_comparison": "{:.4g}",
        "relative_reduction_pct": "{:.1f}%",
        "one_sided_mann_whitney_p": "{:.3e}",
        "FDR_BH_global": "{:.3e}",
        "FDR_BH_within_model": "{:.3e}",
    })
)

display(
    Markdown(
        "## Most MSE-only-dominated HPA classes across all models"
    )
)

display(
    top_red_classes_all_models
    .style
    .hide(
        axis="index"
    )
    .format({
        "pearson_threshold": "{:.2f}",
        "mse_threshold": "{:.2f}",
        "mse_only_pct_of_joint": "{:.1f}%",
        "mse_only_pct_of_successes": "{:.1f}%",
        "median_observed_variance": "{:.4g}",
        "median_variance_percentile": "{:.1f}",
        "median_observed_iqr": "{:.4g}",
        "median_observed_90pct_span": "{:.4g}",
        "median_observed_mad": "{:.4g}",
    })
)

# -----------------------------------------------------------------------------
# One 2x2 variance-percentile grid per model
# -----------------------------------------------------------------------------
category_colors = globals().get(
    "PEARSON_MSE_COLORS",
    {
        "Pearson only": "#6F4E9C",
        "Shared": "#59A14F",
        "MSE only": "#E15759",
        "Neither passes": "#B07AA1",
    },
)

threshold_pairs = [
    (
        pearson_threshold,
        mse_threshold,
    )
    for pearson_threshold
    in PEARSON_THRESHOLDS_DIAG
    for mse_threshold
    in MSE_THRESHOLDS_DIAG
]

for model in models_found:

    fig, axes = plt.subplots(
        2,
        2,
        figsize=readable_figsize((
            12.5,
            8.0,
        )),
    )

    for ax, (
        pearson_threshold,
        mse_threshold,
    ) in zip(
        axes.ravel(),
        threshold_pairs,
    ):

        membership_subset = membership_all_models[
            membership_all_models[
                "model"
            ].eq(
                model
            )
            & np.isclose(
                membership_all_models[
                    "pearson_threshold"
                ],
                pearson_threshold,
            )
            & np.isclose(
                membership_all_models[
                    "mse_threshold"
                ],
                mse_threshold,
            )
        ]

        plot_data = membership_subset.merge(
            profile_metrics,
            on="gene_symbol",
            how="left",
        )

        labels = []
        values = []

        for category in [
            "Pearson only",
            "Shared",
            "MSE only",
        ]:

            category_values = (
                plot_data.loc[
                    plot_data["category"].eq(
                        category
                    ),
                    "observed_variance_percentile",
                ]
                .dropna()
                .to_numpy()
            )

            if len(category_values):

                labels.append(
                    category
                )

                values.append(
                    category_values
                )

        if values:

            boxplot = ax.boxplot(
                values,
                tick_labels=labels,
                showfliers=False,
                patch_artist=True,
            )

            for patch, label in zip(
                boxplot["boxes"],
                labels,
            ):

                patch.set_facecolor(
                    category_colors.get(
                        label,
                        "#BBBBBB",
                    )
                )

                patch.set_alpha(
                    0.80
                )

        ax.set_ylim(
            0,
            100,
        )

        None

        ax.set_ylabel(
            "Observed-variance percentile"
        )

        ax.set_xlabel(
            "Pearson/MSE category"
        )

        ax.tick_params(
            axis="x",
            rotation=18,
        )

        ax.grid(
            axis="y",
            alpha=0.16,
            linewidth=0.6,
        )

    # Notebook grid-label policy:
    # y-axis title only on the left panel;
    # x-axis title only on the right panel.
    for row_index in range(2):

        axes[
            row_index,
            1,
        ].set_ylabel("")

        axes[
            row_index,
            0,
        ].set_xlabel("")

    None

    fig.tight_layout(
        rect=(
            0,
            0.03,
            1,
            0.94,
        ),
        h_pad=1.68,
        w_pad=1.5,
    )

    safe_model = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        model,
    ).strip("_")

    figure_path = (
        OUTPUT_FIGURE_DIR
        / f"{safe_model}_observed_variance_percentile_grid.png"
    )

    save_readable_figure(fig, 
        figure_path,
        dpi=180,
        bbox_inches="tight",
    )

    show_readable_figures()

# -----------------------------------------------------------------------------
# Optional interpretation of the constant-mean baseline
# -----------------------------------------------------------------------------
if PROTEOMICS_IS_ON_MSE_SCALE:

    display(
        Markdown(
            "**Scale confirmed:** `mean_baseline_mse` can be compared "
            "directly with the MSE thresholds because the observed matrix "
            "uses the same preprocessing and scale."
        )
    )

else:

    display(
        Markdown(
            "**Scale not confirmed:** `mean_baseline_mse` equals the "
            "observed population variance, but it should not be compared "
            "directly with MSE cut-offs 0.20 or 0.30 until the matrix scale "
            "is confirmed to match the score calculation."
        )
    )

print("\nInterpretation guide:")

print(
    "1. For each model, inspect the rows with "
    "`comparison = MSE only < Pearson only`."
)

print(
    "2. The low-variance explanation is supported when "
    "`median_mse_only < median_comparison`, "
    "`direction_supported = True`, and FDR < 0.05."
)

print(
    "3. The MSE-only < Shared comparison tests whether low variance "
    "also distinguishes MSE-only proteins from proteins passing both metrics."
)

print(
    "4. Compare the relative-reduction percentages across Proxy, "
    "ElasticNet, and XGB rather than comparing p-values alone."
)

print(
    "5. Observed variance and mean-baseline MSE are numerically the same "
    "quantity here and must not be reported as independent evidence."
)

print(
    "6. This cell tests observed target variability. Testing whether model "
    "predictions collapse toward the mean still requires the sample-level "
    "prediction matrices for each model."
)

## Output guide

The most important exported tables are:

- `part1_full_hpa_vs_case_universe_long.csv`: full HPA counts and per-case intersections;
- `part1_sequential_full_hpa_percentage_decomposition.csv`: the five-component corrected cascade decomposition;
- `part1_pearson_mse_common_universe_overlap_audit.csv`: zero-overlap plausibility and common-universe checks;
- `part2_full_hpa_vs_standalone_universe.csv`: standalone universe intersections;
- `part2_standalone_model_coverage_full_hpa_percentage.csv`: standalone per-model coverage with the full HPA denominator;
- `part2_raw_score_pearson_mse_overlap_audit.csv`: direct Pearson/MSE overlap on identical protein/model rows.

All saved figures are vertical percentage bar plots. Count values remain available in the CSV tables for auditing, but no count-based figure is generated.

In [ ]:
print('Notebook V4 analysis complete.')
print('Tables:', TABLE_DIR)
print('Figures:', FIGURE_DIR)
print(f'Part 1 vertical percentage figures: {len(part1_figures)}')
if INDEPENDENT_AVAILABLE:
    print(f'Part 2 vertical percentage figures: {len(part2_figures)}')
else:
    print('Part 2 was skipped because no valid standalone score table was found.')

## Export all figures to PDF

Run this cell after the plotting cells. It creates `figures_pdf` next to the notebook and saves one PDF for every captured figure, including figures already closed.

In [ ]:
pdf_files = export_notebook_figures()
pdf_files